<div style="font-family:Arial, Helvetica, sans-serif; width:100%; max-width:1180px; margin:0 0 14px 0; box-sizing:border-box;">
  <div style="display:flex; align-items:center; gap:24px; padding:22px 26px; border-radius:28px; background:linear-gradient(135deg,#8B0000 0%,#4b0000 52%,#171717 100%); color:white; box-shadow:0 8px 28px rgba(0,0,0,.16); box-sizing:border-box;">
    <div style="background:white; border-radius:20px; padding:12px; display:flex; align-items:center; justify-content:center; min-width:104px;">
      <img src="graphics/Logo.png" alt="ClubProfile Logo" style="height:100px; width:auto; object-fit:contain;">
    </div>
    <div>
      <div style="font-size:52px; line-height:1; font-weight:900; letter-spacing:.2px;">ClubProfile</div>
      <div style="font-size:19px; opacity:.92; margin-top:10px;">Interaktives Scouting- und Vergleichstool für Spielerprofile im Kontext des FCN</div>
    </div>
  </div>
</div>

<div style="font-family:Arial, Helvetica, sans-serif; width:100%; max-width:1180px; box-sizing:border-box; line-height:1.5; margin:0 0 8px 0;">

## Bedienung

1. Referenzmodus auswählen und die Daten laden.
2. Über Suche, Positionsfilter und Trefferliste einen Spieler auswählen.
3. Im Tab **Rollenprofil** den rollenbasierten Steckbrief des ausgewählten Spielers ansehen.
4. Im Tab **Metrikvergleich / Spiderplot** den Spieler mit passenden Referenz- oder Vergleichsspielern visualisieren.
5. Im Tab **Teambuilder** eine Formation wählen, Rollen bei Bedarf anpassen und eine Elf aus **FCN-Spielern** oder **allen Spielern** generieren — wahlweise nach **Score** oder **Fit** optimiert.
6. Den Rollenprofil-Steckbrief kannst du über den Druck-Button als PDF speichern.

## Interpretation

**Rollenprofil:** Bewertet, wie gut ein Spieler zu verschiedenen Rollen passt. Die Bewertung kombiniert Metrik-Scores, Rollenabdeckung, Rollenspezifität, Datenqualität und einen Ligenfaktor.

**Spiderplot:** Zeigt Metrikwerte relativ zu einem FCN-Referenzspieler. Der Referenzspieler steht je Metrik bei **100 %**. Werte darüber liegen über dem Referenzspieler, Werte darunter darunter. Kleine Labels an der Referenzlinie zeigen die absoluten Rohwerte des Referenzspielers und beantworten damit die Frage: **Wie viel ist 100 % in dieser Metrik?**

**Teambuilder:** Ordnet elf Rollen einer Formation teamoptimal zu. Ein Spieler wird nur einmal vergeben; die Optimierung versucht, möglichst alle Rollen sinnvoll zu besetzen und danach je nach Auswahl den höchsten **Score** oder den besten **Fit** zu erreichen. Rote Karten markieren FCN-Spieler, blaue Karten markieren externe Spieler.

</div>


In [ ]:
# =========================
# ClubProfile 2.0 – Imports
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import math
import html
import textwrap
import base64
from io import BytesIO
from pathlib import Path
from collections import defaultdict
from matplotlib import patches

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML as IPyHTML
except ImportError as exc:
    raise ImportError(
        "ipywidgets ist nicht installiert. Installiere es z. B. mit: pip install ipywidgets"
    ) from exc


In [ ]:
# =========================
# ClubProfile 2.0 – Assets und Branding
# =========================

APP_TITLE = "ClubProfile"
APP_SUBTITLE = "Interaktives Scouting- und Vergleichstool für Spielerprofile im Kontext des FCN"
GRAPHICS_DIR = Path("graphics")
LOGO_PATH = GRAPHICS_DIR / "Logo.png"
BADGE_GOLD_PATH = GRAPHICS_DIR / "Badge_Gold.png"
BADGE_SILBER_PATH = GRAPHICS_DIR / "Badge_Silber.png"
BADGE_BRONZE_PATH = GRAPHICS_DIR / "Badge_Bronze.png"

CLUBPROFILE_RED = "#8B0000"
CLUBPROFILE_BLACK = "#171717"
CLUBPROFILE_GREY = "#5F6368"
CLUBPROFILE_BG = "#FAF7F7"


def image_to_data_uri(path):
    path = Path(path)
    if not path.exists():
        return ""
    suffix = path.suffix.lower().replace(".", "")
    mime = "image/png" if suffix == "png" else f"image/{suffix}"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{data}"


LOGO_URI = image_to_data_uri(LOGO_PATH)
BADGE_URIS = {
    "gold": image_to_data_uri(BADGE_GOLD_PATH),
    "silver": image_to_data_uri(BADGE_SILBER_PATH),
    "bronze": image_to_data_uri(BADGE_BRONZE_PATH),
}


def clubprofile_header_html():
    logo_html = (
        f"<img src='{html.escape(LOGO_URI, quote=True)}' alt='ClubProfile Logo' "
        "style='height:104px; width:auto; object-fit:contain;'>"
        if LOGO_URI else
        "<div style='height:72px;width:72px;border-radius:18px;background:#8B0000;color:white;display:flex;align-items:center;justify-content:center;font-weight:900;font-size:24px;'>CP</div>"
    )
    return f"""
    <div style="font-family:Arial, Helvetica, sans-serif; width:100%; max-width:1280px; margin:0 0 18px 0;">
      <div style="display:flex; align-items:center; gap:22px; padding:18px 20px; border-radius:24px; background:linear-gradient(135deg,#8B0000 0%,#4b0000 52%,#171717 100%); color:white; box-shadow:0 8px 28px rgba(0,0,0,.16);">
        <div style="background:white; border-radius:18px; padding:10px; display:flex; align-items:center; justify-content:center; min-width:92px;">{logo_html}</div>
        <div>
          <div style="font-size:52px; line-height:1; font-weight:900; letter-spacing:.2px;">ClubProfile</div>
          <div style="font-size:19px; opacity:.92; margin-top:8px;">{html.escape(APP_SUBTITLE)}</div>
        </div>
      </div>
    </div>
    """


In [ ]:
from pathlib import Path
import re
import math
import html
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Zentrale Dateipfade
# ---------------------------------------------------------------------
DATA_DIR = Path('.')
ORIGINAL_EXCEL = DATA_DIR / 'spieler_data.xlsx'
WORLD_CLASS_REFERENCE_EXCEL = DATA_DIR / 'weltklasse_referenz.xlsx'
SECOND_LEAGUE_REFERENCE_EXCEL = DATA_DIR / 'zweite_liga_referenz.xlsx'
TEAMBUILDER_TEMPLATES_EXCEL = DATA_DIR / 'teambuilder_formationen.xlsx'
ROLE_DEFINITIONS_EXCEL = DATA_DIR / 'rollen_definitionen.xlsx'
REFERENCE_MODE = 'best_2_bundesliga'  # 'world_class' oder 'best_2_bundesliga'
OUTPUT_EXCEL = DATA_DIR / f'best11_benchmark_rollenfits_v14_gui_{REFERENCE_MODE}.xlsx'

# Fallback für Sandbox/lokale Tests
_fallbacks = {
    'ORIGINAL_EXCEL': Path('/mnt/data/spieler_data.xlsx'),
    'WORLD_CLASS_REFERENCE_EXCEL': Path('/mnt/data/weltklasse_referenz.xlsx'),
    'SECOND_LEAGUE_REFERENCE_EXCEL': Path('/mnt/data/zweite_liga_referenz.xlsx'),
    'TEAMBUILDER_TEMPLATES_EXCEL': Path('/mnt/data/teambuilder_formationen.xlsx'),
    'ROLE_DEFINITIONS_EXCEL': Path('/mnt/data/rollen_definitionen.xlsx'),
}
for _name, _fallback in _fallbacks.items():
    if not globals()[_name].exists() and _fallback.exists():
        globals()[_name] = _fallback
if Path('/mnt/data').exists() and ORIGINAL_EXCEL.parent == Path('/mnt/data'):
    OUTPUT_EXCEL = Path('/mnt/data') / f'best11_benchmark_rollenfits_v10_gui_{REFERENCE_MODE}.xlsx'

# ---------------------------------------------------------------------
# Globale Score-Konfiguration
# ---------------------------------------------------------------------
APPLY_POSITION_FILTER = True
METRICS_TO_EXCLUDE = {'Cards', 'Fouls'}
APPLY_TECHNICAL_METRIC_CAP = True

MIN_METRIC_SCORE_USED = 0.0
MAX_METRIC_SCORE_USED = 200.0
APPLY_TECHNICAL_METRIC_CAP = True

# Metriken, die nicht als Benchmark-Quote berechnet werden. Für diese Metriken wird ein
# Perzentil-Score aus Spielerdatei + beiden Referenzdateien verwendet:
# Score = 2 * Perzentilrang. Median ≈ 100, Maximum = 200, Minimum nahe 0.
PERCENTILE_SCORE_METRICS = {'Acceleration with Ball'}

# Gates v10: einzelne schwache Core-Metriken deckeln nicht mehr den Rollen-Score,
# sondern nur den Role Fit. Harte Score-Caps greifen nur bei mehreren Core-Defiziten,
# niedriger Coverage oder optional Minuten-Caps.
CORE_METRIC_FIT_CAP_RULES = [
    {'threshold': 40, 'cap': 70, 'label': 'Fit-Cap: Core-Metrik < 40'},
    {'threshold': 50, 'cap': 80, 'label': 'Fit-Cap: Core-Metrik < 50'},
    {'threshold': 60, 'cap': 90, 'label': 'Fit-Cap: Core-Metrik < 60'},
]
# Alias für ältere Tabellen-/Methodikzellen.
CORE_METRIC_CAP_RULES = CORE_METRIC_FIT_CAP_RULES
MULTI_CORE_CAP_RULES = [
    {'n_core_below': 3, 'threshold': 60, 'cap': 65, 'label': 'Mind. 3 Core-Metriken < 60'},
    {'n_core_below': 2, 'threshold': 60, 'cap': 75, 'label': 'Mind. 2 Core-Metriken < 60'},
]
COVERAGE_CAP_RULES = [
    {'threshold': 0.60, 'cap': 70, 'label': 'Coverage < 60%'},
    {'threshold': 0.80, 'cap': 85, 'label': 'Coverage < 80%'},
]
APPLY_MINUTES_CAPS = False
MINUTES_CAP_RULES = [
    {'threshold': 300, 'cap': 60, 'label': 'Minuten < 300'},
    {'threshold': 600, 'cap': 80, 'label': 'Minuten < 600'},
    {'threshold': 900, 'cap': 90, 'label': 'Minuten < 900'},
]

# Rollen-spezifische Fit-Caps. Wichtig: Metriken wie xA, Flanken oder progressive Läufe
# werden nicht negativ bewertet. Wenn eine offensive Metrik für eine defensive Rolle
# nicht rollendefinierend ist, wird sie dort nicht verwendet statt invertiert.
ROLE_SPECIFIC_FIT_CAP_RULES = []
SCORE_GATE_TYPES = {'multi_core', 'coverage', 'minutes'}
FIT_GATE_TYPES = {'core_metric_fit', 'role_specific_fit'}

PROFILE_COMPLETENESS_WEIGHT = 1.00
ROLE_SPECIFICITY_WEIGHT = 0.00
SPECIFICITY_NEUTRAL = 50.0
SPECIFICITY_POINTS_PER_GAP_POINT = 2.0
MIN_FIT = 0.0
MAX_FIT = 100.0
FIT_MODIFIER_BASE = 0.70

# Role Match wird referenzunabhängig berechnet:
# 70% Fit gegen 2.-Liga-Benchmark + 30% Fit gegen Weltklasse-Benchmark.
REFERENCE_MODES = ['best_2_bundesliga', 'world_class']
ROLE_MATCH_REFERENCE_WEIGHTS = {'best_2_bundesliga': 0.70, 'world_class': 0.30}
CAP_WORLD_CLASS_LEAGUE_ADJUSTED_AT_100 = True

LEAGUE_FACTORS_BASE = {
    'Bundesliga': 1.13,
    '2. Bundesliga': 1.00,

    '3. Liga': 0.84,

    'Austrian Bundesliga': 0.96,
    'Czech Fortuna Liga': 0.94,      # aktuell eher: Czech Chance Liga

    'Cyprus 1. Division': 0.75,
    'Portuguese Segunda Liga': 0.76,
    'Serie C': 0.72,
    'Swiss Challenge League': 0.70,

    'Regionalliga Bayern': 0.62,
    'Regionalliga Nordost': 0.63,

    'U19 Bundesliga': 0.55,          # aktuell eher: U19 DFB-Nachwuchsliga
    'U17 Bundesliga': 0.43,          # aktuell eher: U17 DFB-Nachwuchsliga
}
# Gegen Weltklasse wird das Wettbewerbsniveau härter skaliert: 2. Bundesliga = 0.70.
# Die synthetische Weltklasse-Referenz selbst hätte Faktor 1.00.
LEAGUE_FACTORS_WORLD_CLASS = {k: round(min(v * 0.70, 1.00), 4) for k, v in LEAGUE_FACTORS_BASE.items()}
LEAGUE_FACTORS_WORLD_CLASS.update({
    'World Class Reference': 1.00,
    'Synthetic Benchmark': 1.00,
})
LEAGUE_FACTORS_BY_REFERENCE = {
    'best_2_bundesliga': LEAGUE_FACTORS_BASE,
    'world_class': LEAGUE_FACTORS_WORLD_CLASS,
}
DEFAULT_LEAGUE_FACTOR_BY_REFERENCE = {
    'best_2_bundesliga': 1.00,
    'world_class': 0.70,
}
# Rückwärtskompatibler Alias für Stellen, die nur die Default-Tabelle anzeigen.
LEAGUE_FACTORS = LEAGUE_FACTORS_BY_REFERENCE['best_2_bundesliga']
DEFAULT_LEAGUE_FACTOR = DEFAULT_LEAGUE_FACTOR_BY_REFERENCE['best_2_bundesliga']
DROP_METRICS_WITH_MISSING_BENCHMARK = True

META_COLS = ['Spieler', 'Alter', 'Position', 'Minuten', 'Team', 'Liga', 'Vergleichsgruppe', 'Kontext']
TRANSFER_COLS_ALL_ROLES = ['TM Vertrag bis', 'TM Marktwert', 'TM Profil-URL', 'Größe']
TRANSFER_COLS_TOP = ['TM Vertrag bis', 'TM Marktwert', 'TM Profil-URL', 'Größe']

ROLE_FAMILY_TO_PLAYER_FAMILIES = {
    'GK': {'GK'},
    'CB': {'CB'},
    'FB': {'FB'},
    'CM/CDM': {'DM/CM', 'CM'},
    'AM/Wing': {'AM', 'WING'},
    'Wing/ST': {'WING', 'ST'},
}

# ---------------------------------------------------------------------
# Mapping alter Rollen auf neue, deutsch benannte Makro-Rollen
# ---------------------------------------------------------------------
OLD_TO_NEW_ROLE_MAP = {
    # GK
    'GK': 'Moderner TW',
    'Shot-Stopping Distributor': 'Moderner TW',
    'Moderner TW': 'Moderner TW',

    # CB
    'Ball Playing CB': 'Mitspielender IV',
    'Wide CB': 'Mitspielender IV',
    'Mitspielender IV': 'Mitspielender IV',
    'Defensive CB': 'Defensiver IV',
    'Defensiver IV': 'Defensiver IV',

    # FB
    'Attacking FB': 'Offensiver AV',
    'Inverted FB': 'Offensiver AV',
    'Offensiver AV': 'Offensiver AV',
    'Defensive FB': 'Defensiver AV',
    'Defensiver AV': 'Defensiver AV',

    # CM/CDM
    'Number 6': 'Defensive 6',
    'Defensive Mid': 'Defensive 6',
    'Defensive 6': 'Defensive 6',
    'Deep-Lying Playmaker': 'Spielstarke 6',
    'Deep Lying Playmaker': 'Spielstarke 6',
    'Deep Playmaker': 'Spielstarke 6',
    'Possession Enabler': 'Spielstarke 6',
    'Spielstarke 6': 'Spielstarke 6',
    'Box-to-Box Midfielder': 'Box-to-Box',
    'Box-to-Box': 'Box-to-Box',
    'Box-to-box': 'Box-to-Box',
    'Progressive Midfielder': 'Progressiver 8er',
    'Progressive Mid': 'Progressiver 8er',
    'Progressiver 8er': 'Progressiver 8er',

    # AM/Wing
    'Classic CAM': 'Klassische 10',
    'Klassische 10': 'Klassische 10',
    'Advanced Playmaker': 'Kreativer Freigeist',
    'Playmaking Winger': 'Kreativer Freigeist',
    'Kreativer Freigeist': 'Kreativer Freigeist',
    'Traditional Winger': 'Klassischer Flügel',
    'Klassischer Flügel': 'Klassischer Flügel',
    'Inverted Winger': 'Invertierter Flügel',
    'Inside Forward': 'Invertierter Flügel',
    'Wide CAM': 'Invertierter Flügel',
    'Invertierter Flügel': 'Invertierter Flügel',

    # Wing/ST
    'Target Man': 'Zielspieler',
    'Zielspieler': 'Zielspieler',
    'Advanced Striker': 'Wandspieler',
    'Wandspieler': 'Wandspieler',
    'Playmaking Striker': 'Falsche 9',
    'Second Striker': 'Falsche 9',
    'Falsche 9': 'Falsche 9',
    'Link-Up Striker': 'Tiefenläufer',
    'Link Up Striker': 'Tiefenläufer',
    'Deep-Lying Striker': 'Tiefenläufer',
    'Deep Lying Striker': 'Tiefenläufer',
    'Tiefenläufer': 'Tiefenläufer',
}

DEFAULT_SIMPLIFIED_ROLE_CONFIG = {
    'Moderner TW': {
        'family': 'GK',
        'source_roles': ['Shot-Stopping Distributor', 'GK'],
        'definition': 'Torwart mit starkem Shot-Stopping, aktivem Herauskommen und sauberer Spieleröffnung über kurze sowie lange Pässe.',
        'metrics': [
            ('Prevented Goals', 0.30, 'hoch besser'),
            ('Save %', 0.15, 'hoch besser'),
            ('Goals Prevented %', 0.10, 'hoch besser'),
            ('Short Passes p90 (derived)', 0.15, 'hoch besser'),
            ('Long Pass Cmp %', 0.10, 'hoch besser'),
            ('% of Passes Being Short', 0.10, 'hoch besser'),
            ('Coming Off Line', 0.10, 'hoch besser'),
        ],
        'core': ['Prevented Goals', 'Save %', 'Short Passes p90 (derived)'],
    },

    'Mitspielender IV': {
        'family': 'CB',
        'source_roles': ['Ball Playing CB', 'Wide CB'],
        'definition': 'Innenverteidiger, der unter Druck sauber aufbaut, progressive und lange Pässe spielt und defensiv stabil bleibt.',
        'metrics': [
            ('Prog. Passes', 0.25, 'hoch besser'),
            ('Long Pass %', 0.20, 'hoch besser'),
            ('Prog. Carries', 0.10, 'hoch besser'),
            ('Defensive Duels Won %', 0.25, 'hoch besser'),
            ('Aerial Win %', 0.20, 'hoch besser'),
        ],
        'core': ['Prog. Passes', 'Long Pass %', 'Defensive Duels Won %'],
    },
    'Defensiver IV': {
        'family': 'CB',
        'source_roles': ['Defensive CB'],
        'definition': 'Absichernder Innenverteidiger mit Fokus auf Zweikampfstärke, Strafraumverteidigung, Luftduelle, Blocks und Basis-Aufbauqualität.',
        'metrics': [
            ('Defensive Duels Won %', 0.24, 'hoch besser'),
            ('Aerial Win %', 0.19, 'hoch besser'),
            ('Shot Blocks', 0.17, 'hoch besser'),
            ('Defensive Actions', 0.14, 'hoch besser'),
            ('Tackles + Interceptions pAdj (derived)', 0.11, 'hoch besser'),
            ('Interceptions (pAdj)', 0.05, 'hoch besser'),
            ('Short & Med Pass %', 0.05, 'hoch besser'),
            ('Long Pass %', 0.05, 'hoch besser'),
        ],
        'core': ['Defensive Duels Won %', 'Aerial Win %', 'Shot Blocks'],
    },

    'Offensiver AV': {
        'family': 'FB',
        'source_roles': ['Attacking FB', 'Inverted FB'],
        'definition': 'Außenverteidiger mit klarem Vorwärtsdrang: progressive Läufe, Dynamik, hohe Beteiligung im letzten Drittel sowie ausreichender defensiver 1v1-Stabilität.',
        'metrics': [
            ('Prog. Carries', 0.19, 'hoch besser'),
            ('Acceleration with Ball', 0.16, 'hoch besser'),
            ('Crosses', 0.13, 'hoch besser'),
            ('Expected Assists (xA)', 0.12, 'hoch besser'),
            ('Prog. Passes', 0.11, 'hoch besser'),
            ('Shot Assists', 0.08, 'hoch besser'),
            ('Dribble Success %', 0.08, 'hoch besser'),
            ('Cross Completion %', 0.08, 'hoch besser'),
            ('Defensive Duels Won %', 0.05, 'hoch besser'),
        ],
        'core': ['Prog. Carries', 'Acceleration with Ball', 'Expected Assists (xA)', 'Crosses'],
    },
    'Defensiver AV': {
        'family': 'FB',
        'source_roles': ['Defensive FB'],
        'definition': 'Absichernder Außenverteidiger mit starker 1v1-Verteidigung, Balleroberungen und Interceptions; Luftduelle, Blocks und einfache Offensivfähigkeit ergänzen das Profil niedrig gewichtet.',
        'metrics': [
            ('Defensive Duels Won %', 0.25, 'hoch besser'),
            ('Tackles + Interceptions pAdj (derived)', 0.22, 'hoch besser'),
            ('Interceptions (pAdj)', 0.15, 'hoch besser'),
            ('Defensive Actions', 0.12, 'hoch besser'),
            ('Aerial Win %', 0.08, 'hoch besser'),
            ('Shot Blocks', 0.06, 'hoch besser'),
            ('Short & Med Pass %', 0.05, 'hoch besser'),
            ('Crosses', 0.04, 'hoch besser'),
            ('Dribble Success %', 0.03, 'hoch besser'),
        ],
        'core': ['Defensive Duels Won %', 'Tackles + Interceptions pAdj (derived)', 'Interceptions (pAdj)', 'Defensive Actions'],
    },

    'Defensive 6': {
        'family': 'CM/CDM',
        'source_roles': ['Number 6', 'Defensive Mid'],
        'definition': 'Defensiver Sechser mit Fokus auf Raumkontrolle, Balleroberungen, Interceptions und stabilem Kurz-/Langpassspiel.',
        'metrics': [
            ('Tackles + Interceptions pAdj (derived)', 0.25, 'hoch besser'),
            ('Defensive Actions', 0.22, 'hoch besser'),
            ('Interceptions (pAdj)', 0.14, 'hoch besser'),
            ('Defensive Duels Won %', 0.12, 'hoch besser'),
            ('Short & Med Pass %', 0.12, 'hoch besser'),
            ('Aerial Win %', 0.06, 'hoch besser'),
            ('Long Pass %', 0.06, 'hoch besser'),
            ('Smart Pass %', 0.03, 'hoch besser'),
        ],
        'core': ['Tackles + Interceptions pAdj (derived)', 'Defensive Actions', 'Interceptions (pAdj)'],
    },
    'Spielstarke 6': {
        'family': 'CM/CDM',
        'source_roles': ['Deep-Lying Playmaker', 'Deep Playmaker', 'Possession Enabler'],
        'definition': 'Aufbau-Sechser mit viel Ballbesitz, progressivem Passspiel, hoher Passsicherheit, Pressingresistenz und ausreichender Gegenpressing-/Defensivpräsenz.',
        'metrics': [
            ('Prog. Passes', 0.21, 'hoch besser'),
            ('Short & Med Pass %', 0.17, 'hoch besser'),
            ('Long Pass %', 0.17, 'hoch besser'),
            ('Smart Pass %', 0.10, 'hoch besser'),
            ('Passes', 0.09, 'hoch besser'),
            ('Received Passes', 0.06, 'hoch besser'),
            ('Tackles + Interceptions pAdj (derived)', 0.09, 'hoch besser'),
            ('Defensive Actions', 0.07, 'hoch besser'),
            ('Dribble Success %', 0.04, 'hoch besser'),
        ],
        'core': ['Prog. Passes', 'Short & Med Pass %', 'Long Pass %'],
    },
    'Box-to-Box': {
        'family': 'CM/CDM',
        'source_roles': ['Box-to-Box Midfielder', 'Box-to-Box'],
        'definition': 'Laufstarker Mittelfeldspieler mit defensiver Arbeit, Ballprogression und Präsenz in höheren Zonen.',
        'metrics': [
            ('Tackles + Interceptions pAdj (derived)', 0.18, 'hoch besser'),
            ('Defensive Actions', 0.15, 'hoch besser'),
            ('Prog. Carries', 0.18, 'hoch besser'),
            ('Acceleration with Ball', 0.12, 'hoch besser'),
            ('Prog. Passes', 0.12, 'hoch besser'),
            ('Received Passes', 0.08, 'hoch besser'),
            ('Touches in Pen Box', 0.07, 'hoch besser'),
            ('Shot Assists', 0.05, 'hoch besser'),
            ('npxG', 0.05, 'hoch besser'),
        ],
        'core': ['Tackles + Interceptions pAdj (derived)', 'Prog. Carries', 'Defensive Actions'],
    },
    'Progressiver 8er': {
        'family': 'CM/CDM',
        'source_roles': ['Progressive Midfielder', 'Progressive Mid'],
        'definition': 'Balltragender Achter, der Linien vor allem über Carries, Dynamik und Dribbling überwindet und danach Anschlussaktionen findet.',
        'metrics': [
            ('Prog. Carries', 0.28, 'hoch besser'),
            ('Acceleration with Ball', 0.22, 'hoch besser'),
            ('Dribble Success %', 0.16, 'hoch besser'),
            ('Prog. Passes', 0.12, 'hoch besser'),
            ('Received Passes', 0.06, 'hoch besser'),
            ('Shot Assists', 0.06, 'hoch besser'),
            ('Expected Assists (xA)', 0.05, 'hoch besser'),
            ('Touches in Pen Box', 0.05, 'hoch besser'),
        ],
        'core': ['Prog. Carries', 'Acceleration with Ball', 'Dribble Success %'],
    },

    'Klassische 10': {
        'family': 'AM/Wing',
        'source_roles': ['Classic CAM'],
        'definition': 'Zentraler Offensivspieler zwischen den Linien mit Chance Creation, Boxnähe und eigenem Abschluss, weniger als Flankengeber.',
        'metrics': [
            ('Expected Assists (xA)', 0.18, 'hoch besser'),
            ('Shot Assists', 0.16, 'hoch besser'),
            ('Touches in Pen Box', 0.16, 'hoch besser'),
            ('npxG', 0.14, 'hoch besser'),
            ('Smart Passes', 0.12, 'hoch besser'),
            ('Non-Pen Goals', 0.10, 'hoch besser'),
            ('Assists + Second Assists (derived)', 0.08, 'hoch besser'),
            ('Received Passes', 0.04, 'hoch besser'),
        ],
        'core': ['Expected Assists (xA)', 'Shot Assists', 'Touches in Pen Box', 'npxG'],
    },
    'Kreativer Freigeist': {
        'family': 'AM/Wing',
        'source_roles': ['Advanced Playmaker', 'Playmaking Winger'],
        'definition': 'Freier Kreativspieler, der hochwertige Chancen über Smart Passes, xA-Qualität, Passprogression, Dribbling und Boxnähe vorbereitet.',
        'metrics': [
            ('Expected Assists (xA)', 0.14, 'hoch besser'),
            ('Shot Assists', 0.14, 'hoch besser'),
            ('Smart Passes', 0.22, 'hoch besser'),
            ('xA per Shot Assist', 0.14, 'hoch besser'),
            ('Prog. Passes', 0.09, 'hoch besser'),
            ('Prog. Carries', 0.08, 'hoch besser'),
            ('Assists + Second Assists (derived)', 0.06, 'hoch besser'),
            ('Dribble Success %', 0.08, 'hoch besser'),
            ('Touches in Pen Box', 0.05, 'hoch besser'),
        ],
        'core': ['Smart Passes', 'Expected Assists (xA)', 'xA per Shot Assist'],
    },
    'Klassischer Flügel': {
        'family': 'AM/Wing',
        'source_roles': ['Traditional Winger'],
        'definition': 'Breiter Flügelspieler, der über Tempo, Dribbling, Flanken und Vorlagen von außen wirkt.',
        'metrics': [
            ('Crosses', 0.25, 'hoch besser'),
            ('Cross Completion %', 0.20, 'hoch besser'),
            ('Acceleration with Ball', 0.15, 'hoch besser'),
            ('Dribble Success %', 0.12, 'hoch besser'),
            ('Prog. Carries', 0.12, 'hoch besser'),
            ('Shot Assists', 0.08, 'hoch besser'),
            ('Expected Assists (xA)', 0.05, 'hoch besser'),
            ('Touches in Pen Box', 0.03, 'hoch besser'),
        ],
        'core': ['Crosses', 'Cross Completion %', 'Acceleration with Ball'],
    },
    'Invertierter Flügel': {
        'family': 'AM/Wing',
        'source_roles': ['Inverted Winger', 'Inside Forward', 'Wide CAM'],
        'definition': 'Nach innen ziehender Flügel-/Halbraumspieler mit Ballprogression, Boxpräsenz, Abschlüssen und Scoring-Output.',
        'metrics': [
            ('Prog. Carries', 0.17, 'hoch besser'),
            ('Acceleration with Ball', 0.14, 'hoch besser'),
            ('Touches in Pen Box', 0.16, 'hoch besser'),
            ('npxG', 0.16, 'hoch besser'),
            ('Non-Pen Goals', 0.12, 'hoch besser'),
            ('Shots', 0.10, 'hoch besser'),
            ('Expected Assists (xA)', 0.08, 'hoch besser'),
            ('Shot Assists', 0.07, 'hoch besser'),
        ],
        'core': ['Prog. Carries', 'Touches in Pen Box', 'npxG'],
    },

    'Zielspieler': {
        'family': 'Wing/ST',
        'source_roles': ['Target Man'],
        'definition': 'Physischer Zielspieler mit Luftduellstärke, Strafraumpräsenz und Abschlussqualität.',
        'metrics': [
            ('Aerial Win %', 0.25, 'hoch besser'),
            ('Aerial Duels Won', 0.15, 'hoch besser'),
            ('Touches in Pen Box', 0.15, 'hoch besser'),
            ('npxG', 0.15, 'hoch besser'),
            ('Non-Pen Goals', 0.12, 'hoch besser'),
            ('npxG per Shot', 0.08, 'hoch besser'),
            ('Shots', 0.05, 'hoch besser'),
            ('Short & Med Pass %', 0.05, 'hoch besser'),
        ],
        'core': ['Aerial Win %', 'Touches in Pen Box', 'npxG'],
    },
    'Wandspieler': {
        'family': 'Wing/ST',
        'source_roles': ['Advanced Striker'],
        'definition': 'Mittelstürmer mit hohem Abschlussvolumen, Boxpräsenz, npxG und zusätzlicher Ablage-/Verbindungskomponente.',
        'metrics': [
            ('npxG', 0.22, 'hoch besser'),
            ('Non-Pen Goals', 0.20, 'hoch besser'),
            ('Shots', 0.15, 'hoch besser'),
            ('Touches in Pen Box', 0.13, 'hoch besser'),
            ('npxG per Shot', 0.10, 'hoch besser'),
            ('Aerial Win %', 0.08, 'hoch besser'),
            ('Short & Med Pass %', 0.06, 'hoch besser'),
            ('Received Passes', 0.06, 'hoch besser'),
        ],
        'core': ['npxG', 'Non-Pen Goals', 'Shots'],
    },
    'Falsche 9': {
        'family': 'Wing/ST',
        'source_roles': ['Playmaking Striker', 'Second Striker'],
        'definition': 'Zurückfallender Stürmer, der Chancen vorbereitet, Räume verbindet und trotzdem Torgefahr einbringt.',
        'metrics': [
            ('Expected Assists (xA)', 0.20, 'hoch besser'),
            ('Shot Assists', 0.18, 'hoch besser'),
            ('Smart Passes', 0.15, 'hoch besser'),
            ('Assists + Second Assists (derived)', 0.12, 'hoch besser'),
            ('Prog. Passes', 0.10, 'hoch besser'),
            ('Short & Med Pass %', 0.08, 'hoch besser'),
            ('Non-Pen Goals', 0.09, 'hoch besser'),
            ('Touches in Pen Box', 0.08, 'hoch besser'),
        ],
        'core': ['Expected Assists (xA)', 'Shot Assists', 'Smart Passes'],
    },
    'Tiefenläufer': {
        'family': 'Wing/ST',
        'source_roles': ['Link-Up Striker', 'Deep-Lying Striker'],
        'definition': 'Dynamischer Stürmer/Flügelstürmer, der Tiefe attackiert, über Tempo und Carries in die Box kommt und dort Abschlussvolumen erzeugt.',
        'metrics': [
            ('Non-Pen Goals', 0.15, 'hoch besser'),
            ('npxG', 0.15, 'hoch besser'),
            ('Touches in Pen Box', 0.16, 'hoch besser'),
            ('Acceleration with Ball', 0.17, 'hoch besser'),
            ('Prog. Carries', 0.14, 'hoch besser'),
            ('Assists + Second Assists (derived)', 0.06, 'hoch besser'),
            ('Expected Assists (xA)', 0.08, 'hoch besser'),
            ('Shot Assists', 0.04, 'hoch besser'),
            ('npxG per Shot', 0.05, 'hoch besser'),
        ],
        'core': ['Non-Pen Goals', 'npxG', 'Acceleration with Ball', 'Touches in Pen Box'],
    },
}

# ---------------------------------------------------------------------
# Externe Rollen-Konfiguration
# ---------------------------------------------------------------------
def _role_config_truthy(value, default=True):
    if pd.isna(value):
        return default
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"false", "falsch", "0", "nein", "no", "n", "inaktiv"}:
        return False
    if text in {"true", "wahr", "1", "ja", "yes", "y", "aktiv"}:
        return True
    return default


def _role_config_clean_text(value, default=""):
    if pd.isna(value):
        return default
    return str(value).strip()


def _load_simplified_role_config_from_excel(excel_path=None, fallback=None):
    """Lädt Rollen, Definitionen, Source Roles und Metrikgewichte aus Excel.

    Erwartete Sheets:
    - Rollen: Rolle, Positionsfamilie, Definition, Aktiv
    - Metriken: Rolle, Metrik_Order, Metrik, Gewicht, Richtung, Is_Core_Metric, Aktiv
    - Source_Roles (optional): Rolle, Source_Role, Source_Role_Order, Aktiv

    Die eigentliche Score-/Fit-Berechnung bleibt im Code; diese Datei steuert nur
    die Rollendefinitionen und Metrik-Zusammensetzung.
    """
    path = Path(excel_path or ROLE_DEFINITIONS_EXCEL)
    if not path.exists():
        if fallback is not None:
            return fallback.copy(), FileNotFoundError(
                f"Rollen-Konfiguration nicht gefunden: {path}. Nutze eingebettete Standardrollen."
            )
        raise FileNotFoundError(f"Rollen-Konfiguration nicht gefunden: {path}")

    roles_df = pd.read_excel(path, sheet_name="Rollen")
    metrics_df = pd.read_excel(path, sheet_name="Metriken")
    roles_df.columns = [str(c).strip() for c in roles_df.columns]
    metrics_df.columns = [str(c).strip() for c in metrics_df.columns]

    required_roles = {"Rolle", "Positionsfamilie", "Definition"}
    required_metrics = {"Rolle", "Metrik", "Gewicht", "Richtung"}
    missing_roles = required_roles - set(roles_df.columns)
    missing_metrics = required_metrics - set(metrics_df.columns)
    if missing_roles or missing_metrics:
        raise ValueError(
            "Rollen-Konfiguration unvollständig. "
            f"Fehlende Rollen-Spalten: {', '.join(sorted(missing_roles)) or '–'}; "
            f"fehlende Metriken-Spalten: {', '.join(sorted(missing_metrics)) or '–'}."
        )

    if "Aktiv" not in roles_df.columns:
        roles_df["Aktiv"] = True
    if "Aktiv" not in metrics_df.columns:
        metrics_df["Aktiv"] = True
    if "Metrik_Order" not in metrics_df.columns:
        metrics_df["Metrik_Order"] = metrics_df.groupby("Rolle").cumcount() + 1
    if "Is_Core_Metric" not in metrics_df.columns:
        metrics_df["Is_Core_Metric"] = False

    roles_df = roles_df.loc[roles_df["Aktiv"].apply(_role_config_truthy)].copy()
    metrics_df = metrics_df.loc[metrics_df["Aktiv"].apply(_role_config_truthy)].copy()
    metrics_df["__order"] = pd.to_numeric(metrics_df["Metrik_Order"], errors="coerce")
    metrics_df["__row_order"] = np.arange(len(metrics_df))
    metrics_df = metrics_df.sort_values(["Rolle", "__order", "__row_order"], na_position="last")

    source_roles = {}
    try:
        source_df = pd.read_excel(path, sheet_name="Source_Roles")
        source_df.columns = [str(c).strip() for c in source_df.columns]
        if {"Rolle", "Source_Role"}.issubset(source_df.columns):
            if "Aktiv" not in source_df.columns:
                source_df["Aktiv"] = True
            if "Source_Role_Order" not in source_df.columns:
                source_df["Source_Role_Order"] = source_df.groupby("Rolle").cumcount() + 1
            source_df = source_df.loc[source_df["Aktiv"].apply(_role_config_truthy)].copy()
            source_df["__order"] = pd.to_numeric(source_df["Source_Role_Order"], errors="coerce")
            source_df["__row_order"] = np.arange(len(source_df))
            source_df = source_df.sort_values(["Rolle", "__order", "__row_order"], na_position="last")
            for role, grp in source_df.groupby("Rolle", sort=False):
                vals = [_role_config_clean_text(v) for v in grp["Source_Role"].tolist()]
                source_roles[_role_config_clean_text(role)] = [v for v in vals if v]
    except Exception:
        source_roles = {}

    result = {}
    for _, role_row in roles_df.iterrows():
        role = _role_config_clean_text(role_row.get("Rolle"))
        if not role:
            continue
        metric_subset = metrics_df[metrics_df["Rolle"].astype(str).str.strip().eq(role)].copy()
        metrics = []
        core = []
        for _, metric_row in metric_subset.iterrows():
            metric = _role_config_clean_text(metric_row.get("Metrik"))
            if not metric:
                continue
            try:
                weight = float(metric_row.get("Gewicht", 0.0))
            except Exception:
                weight = 0.0
            direction = _role_config_clean_text(metric_row.get("Richtung"), "hoch besser")
            metrics.append((metric, weight, direction))
            if _role_config_truthy(metric_row.get("Is_Core_Metric"), default=False):
                core.append(metric)
        if not metrics:
            continue
        result[role] = {
            "family": _role_config_clean_text(role_row.get("Positionsfamilie")),
            "source_roles": source_roles.get(role, []),
            "definition": _role_config_clean_text(role_row.get("Definition")),
            "metrics": metrics,
            "core": core,
        }

    if not result:
        raise ValueError("Die Rollen-Konfigurationsdatei enthält keine aktiven Rollen mit aktiven Metriken.")
    return result, None


SIMPLIFIED_ROLE_CONFIG, ROLE_CONFIG_LOAD_ERROR = _load_simplified_role_config_from_excel(
    ROLE_DEFINITIONS_EXCEL,
    fallback=DEFAULT_SIMPLIFIED_ROLE_CONFIG,
)


# ClubProfile 2.0: zusätzlicher Fallback für lokale Tests in dieser Umgebung.
if not ORIGINAL_EXCEL.exists() and Path('/mnt/data/fcn_spielerdiagramme_werte_mit_transfermarkt.xlsx').exists():
    ORIGINAL_EXCEL = Path('/mnt/data/fcn_spielerdiagramme_werte_mit_transfermarkt.xlsx')


In [ ]:
# ---------------------------------------------------------------------
# Hilfsfunktionen: Rollen, Positionen, Metriken
# ---------------------------------------------------------------------
def normalize_role_name(name):
    text = str(name).strip().lower()
    text = re.sub(r'[\u2010-\u2015]', '-', text)
    text = text.replace('&', ' and ')
    text = re.sub(r'[^a-z0-9äöüß]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

ROLE_ALIAS_TO_NEW = {normalize_role_name(k): v for k, v in OLD_TO_NEW_ROLE_MAP.items()}

def map_old_role_to_new(role_name):
    return ROLE_ALIAS_TO_NEW.get(normalize_role_name(role_name), OLD_TO_NEW_ROLE_MAP.get(str(role_name).strip(), np.nan))

def _norm_pos(pos):
    return str(pos).strip().upper()

def player_position_families(position):
    pos = _norm_pos(position)
    if pos == 'GK':
        return {'GK'}
    if pos in {'CB', 'LCB', 'RCB'}:
        return {'CB'}
    if pos in {'FB', 'LB', 'RB', 'LWB', 'RWB'}:
        return {'FB'}
    if pos in {'DM', 'DMF', 'LDMF', 'RDMF'}:
        return {'DM/CM'}
    if pos in {'CM', 'CMF', 'LCMF', 'RCMF', 'LCMF3', 'RCMF3'}:
        return {'DM/CM', 'CM'}
    if pos in {'AM', 'AMF', 'CAM'}:
        return {'AM'}
    if pos in {'LW', 'RW', 'LWF', 'RWF', 'WING'}:
        return {'WING'}
    if pos in {'ST', 'CF'}:
        return {'ST'}
    return set()

def is_role_compatible(player_families, role_family):
    role_targets = ROLE_FAMILY_TO_PLAYER_FAMILIES.get(str(role_family).strip(), {str(role_family).strip()})
    return bool(player_families & role_targets)

def direction_is_higher_better(direction):
    return not str(direction).strip().lower().startswith('niedrig')

def safe_numeric(v):
    return pd.to_numeric(pd.Series([v]), errors='coerce').iloc[0]

def add_derived_metrics(players):
    df = players.copy()
    if {'Passes', '% of Passes Being Short'}.issubset(df.columns):
        df['Short Passes p90 (derived)'] = (
            pd.to_numeric(df['Passes'], errors='coerce')
            * pd.to_numeric(df['% of Passes Being Short'], errors='coerce')
            / 100
        )
    existing_ti = (
        pd.to_numeric(df['Tackles + Interceptions pAdj (derived)'], errors='coerce')
        if 'Tackles + Interceptions pAdj (derived)' in df.columns
        else pd.Series(np.nan, index=df.index, dtype='float64')
    )
    derived_ti = pd.Series(np.nan, index=df.index, dtype='float64')
    if 'Tackles & Int (pAdj)' in df.columns:
        derived_ti = pd.to_numeric(df['Tackles & Int (pAdj)'], errors='coerce')
    ti_parts = [c for c in ['Tackles (pAdj)', 'Interceptions (pAdj)'] if c in df.columns]
    if ti_parts:
        part_values = df[ti_parts].apply(pd.to_numeric, errors='coerce')
        part_sum = part_values.fillna(0).sum(axis=1).where(part_values.notna().any(axis=1))
        derived_ti = derived_ti.combine_first(part_sum)
    df['Tackles + Interceptions pAdj (derived)'] = existing_ti.combine_first(derived_ti)

    existing_assists = (
        pd.to_numeric(df['Assists + Second Assists (derived)'], errors='coerce')
        if 'Assists + Second Assists (derived)' in df.columns
        else pd.Series(np.nan, index=df.index, dtype='float64')
    )
    assist_parts = [c for c in ['Assists', 'Second Assists'] if c in df.columns]
    derived_assists = pd.Series(np.nan, index=df.index, dtype='float64')
    if assist_parts:
        vals = df[assist_parts].apply(pd.to_numeric, errors='coerce')
        derived_assists = vals.fillna(0).sum(axis=1).where(vals.notna().any(axis=1))
    if 'Assists & 2nd/3rd Assists' in df.columns:
        derived_assists = derived_assists.combine_first(pd.to_numeric(df['Assists & 2nd/3rd Assists'], errors='coerce'))
    df['Assists + Second Assists (derived)'] = existing_assists.combine_first(derived_assists)
    return df

def normalize_metric_weights(metrics):
    cleaned = []
    for metric, weight, direction in metrics:
        metric = str(metric).strip()
        if metric in METRICS_TO_EXCLUDE:
            continue
        cleaned.append((metric, float(weight), str(direction).strip()))
    total = sum(w for _, w, _ in cleaned)
    if total <= 0:
        return []
    return [(m, w / total, d) for m, w, d in cleaned]

def build_simplified_roles():
    rows = []
    for role_name, cfg in SIMPLIFIED_ROLE_CONFIG.items():
        metrics = normalize_metric_weights(cfg['metrics'])
        for order, (metric, weight, direction) in enumerate(metrics, start=1):
            rows.append({
                'Rolle': role_name,
                'Positionsfamilie': cfg['family'],
                'Source Roles': ', '.join(cfg['source_roles']),
                'Metrik': metric,
                'Metrik_Order': order,
                'Gewicht': weight,
                'Richtung': direction,
                'Higher_Is_Better': direction_is_higher_better(direction),
                'Is_Core_Metric': metric in cfg['core'],
                'Definition': cfg.get('definition', ''),
            })
    return pd.DataFrame(rows)

def role_metric_specs(role_name, roles_long):
    specs = []
    r = roles_long.loc[roles_long['Rolle'].eq(role_name)].sort_values('Metrik_Order')
    for _, row in r.iterrows():
        specs.append({
            'metric': row['Metrik'],
            'weight': float(row['Gewicht']),
            'direction': str(row['Richtung']).strip().lower(),
            'higher_is_better': bool(row['Higher_Is_Better']),
            'order': int(row['Metrik_Order']),
            'is_core': bool(row['Is_Core_Metric']),
        })
    return specs

In [ ]:
# ---------------------------------------------------------------------
# Input laden
# ---------------------------------------------------------------------
def build_legacy_role_mapping_table():
    """Erzeugt eine Dokumentation der alten Rollen-Aliase und ihrer neuen Makro-Rollen.

    Diese Tabelle ersetzt die frühere Abhängigkeit von best11_rollenprofil_abgleich.xlsx.
    Die eigentlichen Rollendefinitionen kommen aus SIMPLIFIED_ROLE_CONFIG.
    """
    rows = []
    for old_role, new_role in OLD_TO_NEW_ROLE_MAP.items():
        cfg = SIMPLIFIED_ROLE_CONFIG.get(new_role, {})
        rows.append({
            'Old Role Alias': old_role,
            'New Simplified Role': new_role,
            'Positionsfamilie': cfg.get('family', np.nan),
            'Definition': cfg.get('definition', np.nan),
            'Source Roles in New Role': ', '.join(cfg.get('source_roles', [])) if cfg else np.nan,
        })
    return pd.DataFrame(rows).drop_duplicates().sort_values(['New Simplified Role', 'Old Role Alias']).reset_index(drop=True)

def read_players_file(original_excel):
    players = pd.read_excel(original_excel, sheet_name='Werte_breit')

    # Transfermarkt-Daten robust ergänzen:
    # Wenn die Werte_breit-Tabelle die TM-Felder bereits enthält, bleiben sie erhalten.
    # Wenn einzelne Felder fehlen oder leer sind, werden sie aus dem Blatt "Transfermarkt"
    # bzw. ersatzweise aus "Übersicht" ergänzt. Das ist wichtig für Spieler, bei denen
    # TM-Daten nachträglich in die Ausgangsdatei aufgenommen wurden.
    transfer_sources = []
    for sheet in ['Transfermarkt', 'Übersicht']:
        try:
            tmp = pd.read_excel(original_excel, sheet_name=sheet)
            if 'Spieler' in tmp.columns:
                transfer_sources.append(tmp)
        except Exception:
            pass

    transfer_cols = [c for c in TRANSFER_COLS_ALL_ROLES if c != 'Größe'] + ['Größe']
    if transfer_sources:
        transfer_lookup = pd.concat(transfer_sources, ignore_index=True, sort=False)
        keep_cols = ['Spieler'] + [c for c in transfer_cols if c in transfer_lookup.columns]
        transfer_lookup = transfer_lookup[keep_cols].drop_duplicates('Spieler')

        # Fehlende Spalten anlegen, damit der spätere Merge/Fill stabil bleibt.
        for col in transfer_cols:
            if col not in players.columns:
                players[col] = pd.Series([pd.NA] * len(players), dtype='object')

        merge_cols = ['Spieler'] + [c for c in transfer_cols if c in transfer_lookup.columns]
        players = players.merge(
            transfer_lookup[merge_cols],
            on='Spieler',
            how='left',
            suffixes=('', '__tm_lookup')
        )

        for col in transfer_cols:
            lookup_col = f'{col}__tm_lookup'
            if lookup_col in players.columns:
                current = players[col]
                lookup = players[lookup_col]
                empty_current = current.isna() | current.astype(str).str.strip().isin(['', 'nan', 'None'])
                players.loc[empty_current, col] = lookup.loc[empty_current]
                players = players.drop(columns=[lookup_col])
    else:
        for col in transfer_cols:
            if col not in players.columns:
                players[col] = pd.Series([pd.NA] * len(players), dtype='object')

    players = add_derived_metrics(players)
    return players

def read_reference_file(reference_excel, reference_mode):
    overview = pd.read_excel(reference_excel, sheet_name='Übersicht')
    wide = pd.read_excel(reference_excel, sheet_name='Werte_breit')
    if 'Rolle' not in wide.columns:
        join_cols = [c for c in ['Spieler', 'Bilddatei'] if c in wide.columns and c in overview.columns]
        if join_cols:
            lookup = overview[join_cols + ['Rolle']].drop_duplicates(join_cols)
            wide = wide.merge(lookup, on=join_cols, how='left')
        elif len(wide) == len(overview):
            wide['Rolle'] = overview['Rolle'].values
        else:
            raise ValueError(f'Konnte Rollen in {reference_excel} nicht mappen.')
    wide = wide.copy()
    wide['Rolle_raw'] = wide['Rolle']
    wide['Rolle'] = wide['Rolle_raw'].apply(map_old_role_to_new)
    wide['Referenzmodus'] = reference_mode
    wide = add_derived_metrics(wide)
    return wide

def load_all_inputs(original_excel=None, world_reference_excel=None, second_reference_excel=None):
    original_excel = Path(original_excel or ORIGINAL_EXCEL)
    world_reference_excel = Path(world_reference_excel or WORLD_CLASS_REFERENCE_EXCEL)
    second_reference_excel = Path(second_reference_excel or SECOND_LEAGUE_REFERENCE_EXCEL)
    players = read_players_file(original_excel)
    legacy_roles = build_legacy_role_mapping_table()
    simplified_roles = build_simplified_roles()
    world = read_reference_file(world_reference_excel, 'world_class')
    second = read_reference_file(second_reference_excel, 'best_2_bundesliga')
    references = pd.concat([world, second], ignore_index=True)
    return players, legacy_roles, simplified_roles, references

def build_benchmark_values(roles_long, references, reference_mode):
    ref = references.loc[references['Referenzmodus'].eq(reference_mode)].copy()
    rows = []
    for role_name, role_metrics in roles_long.groupby('Rolle', sort=False):
        cfg = SIMPLIFIED_ROLE_CONFIG[role_name]
        rr = ref.loc[ref['Rolle'].eq(role_name)].copy()
        if rr.empty:
            print(f'Warnung: Keine Referenzspieler für neue Rolle: {role_name}')
        for _, spec in role_metrics.sort_values('Metrik_Order').iterrows():
            metric = spec['Metrik']
            higher = bool(spec['Higher_Is_Better'])
            values = pd.to_numeric(rr[metric], errors='coerce') if metric in rr.columns else pd.Series(dtype='float64')
            valid = values.dropna()
            if len(valid):
                selected_idx = valid.idxmax() if higher else valid.idxmin()
                benchmark_value = float(valid.loc[selected_idx])
                selected_row = rr.loc[selected_idx]
                contributing_player = selected_row.get('Spieler', np.nan)
                contributing_old_role = selected_row.get('Rolle_raw', np.nan)
                contributing_team = selected_row.get('Team', np.nan)
                contributing_liga = selected_row.get('Liga', np.nan)
                n_available = int(valid.shape[0])
            else:
                benchmark_value = np.nan
                contributing_player = np.nan
                contributing_old_role = np.nan
                contributing_team = np.nan
                contributing_liga = np.nan
                n_available = 0
            rows.append({
                'Referenzmodus': reference_mode,
                'Rolle': role_name,
                'Positionsfamilie': cfg['family'],
                'Source Roles': ', '.join(cfg['source_roles']),
                'Metrik': metric,
                'Metrik_Order': int(spec['Metrik_Order']),
                'Is_Core_Metric': bool(spec['Is_Core_Metric']),
                'Gewicht': float(spec['Gewicht']),
                'Richtung': spec['Richtung'],
                'Higher_Is_Better': higher,
                'Benchmarkwert': benchmark_value,
                'Benchmarkwert_fehlt': pd.isna(benchmark_value),
                'Benchmarkspieler_fuer_Metrik': contributing_player,
                'Benchmarkrolle_alt_fuer_Metrik': contributing_old_role,
                'Benchmarkteam_fuer_Metrik': contributing_team,
                'Benchmarkliga_fuer_Metrik': contributing_liga,
                'Anzahl_Referenzspieler_mit_Wert': n_available,
                'Referenzspieler_pool': ', '.join(rr['Spieler'].dropna().astype(str).unique()),
                'Referenzrollen_alt_pool': ', '.join(rr['Rolle_raw'].dropna().astype(str).unique()),
            })
    return pd.DataFrame(rows)

def activate_roles_for_benchmark(roles_long, references, reference_mode):
    """Kompatibilitätsfunktion: aktiviert Rollen für einen einzelnen Referenzmodus."""
    benchmark_values_initial = build_benchmark_values(roles_long, references, reference_mode)
    missing_initial = benchmark_values_initial.loc[benchmark_values_initial['Benchmarkwert_fehlt']].copy()
    if DROP_METRICS_WITH_MISSING_BENCHMARK and len(missing_initial):
        dropped = missing_initial[['Referenzmodus', 'Rolle', 'Metrik', 'Source Roles', 'Referenzspieler_pool']].copy()
        dropped_pairs = set(zip(dropped['Rolle'], dropped['Metrik']))
        active_roles = roles_long.loc[
            ~roles_long.apply(lambda r: (r['Rolle'], r['Metrik']) in dropped_pairs, axis=1)
        ].copy()
        active_roles['Gewicht'] = active_roles.groupby('Rolle')['Gewicht'].transform(lambda s: s / s.sum())
        benchmark_values = build_benchmark_values(active_roles, references, reference_mode)
    else:
        dropped = pd.DataFrame(columns=['Referenzmodus', 'Rolle', 'Metrik', 'Source Roles', 'Referenzspieler_pool'])
        active_roles = roles_long.copy()
        benchmark_values = benchmark_values_initial
    return active_roles, benchmark_values, dropped

def activate_roles_for_benchmarks(roles_long, references, reference_modes=REFERENCE_MODES):
    """Aktiviert Rollenmetriken konsistent für beide Benchmarkmodi.

    Für einen stabilen Role Match werden Metriken entfernt, wenn sie in mindestens
    einem der beiden Benchmarkmodi keinen Referenzwert haben. Danach werden die
    Gewichte je Rolle neu normalisiert und die Benchmarkwerte für beide Modi neu
    aufgebaut. Dadurch hängt die Top-Rollenzuordnung nicht vom Dropdown ab.
    """
    initial_by_mode = {mode: build_benchmark_values(roles_long, references, mode) for mode in reference_modes}
    missing = pd.concat(
        [bv.loc[bv['Benchmarkwert_fehlt']].copy() for bv in initial_by_mode.values()],
        ignore_index=True,
    ) if initial_by_mode else pd.DataFrame()
    if DROP_METRICS_WITH_MISSING_BENCHMARK and len(missing):
        dropped = missing[['Referenzmodus', 'Rolle', 'Metrik', 'Source Roles', 'Referenzspieler_pool']].drop_duplicates().copy()
        dropped_pairs = set(zip(dropped['Rolle'], dropped['Metrik']))
        active_roles = roles_long.loc[
            ~roles_long.apply(lambda r: (r['Rolle'], r['Metrik']) in dropped_pairs, axis=1)
        ].copy()
        active_roles['Gewicht'] = active_roles.groupby('Rolle')['Gewicht'].transform(lambda s: s / s.sum())
    else:
        dropped = pd.DataFrame(columns=['Referenzmodus', 'Rolle', 'Metrik', 'Source Roles', 'Referenzspieler_pool'])
        active_roles = roles_long.copy()
    benchmark_values_by_mode = {mode: build_benchmark_values(active_roles, references, mode) for mode in reference_modes}
    benchmark_values_all = pd.concat(benchmark_values_by_mode.values(), ignore_index=True)
    return active_roles, benchmark_values_by_mode, benchmark_values_all, dropped

In [ ]:
# ---------------------------------------------------------------------
# Scoring-Funktionen
# ---------------------------------------------------------------------
def build_percentile_distributions(players, references, metrics=PERCENTILE_SCORE_METRICS):
    """Erzeugt sortierte Wertelisten für Perzentil-Score-Metriken.

    Verwendet alle verfügbaren Daten aus Spielerdatei + beiden Referenzdateien.
    """
    frames = []
    if players is not None and len(players):
        frames.append(add_derived_metrics(players))
    if references is not None and len(references):
        frames.append(add_derived_metrics(references))
    if not frames:
        return {}
    pool = pd.concat(frames, ignore_index=True, sort=False)
    distributions = {}
    for metric in metrics:
        if metric not in pool.columns:
            distributions[metric] = np.array([], dtype=float)
            continue
        vals = pd.to_numeric(pool[metric], errors='coerce').dropna().astype(float).values
        distributions[metric] = np.sort(vals)
    return distributions

def percentile_metric_score(player_value, metric, percentile_distributions):
    p = safe_numeric(player_value)
    if pd.isna(p):
        return np.nan, 'Spielerwert fehlt'
    values = None if percentile_distributions is None else percentile_distributions.get(metric)
    if values is None or len(values) == 0:
        return np.nan, 'Perzentil-Verteilung fehlt'
    # Anteil der Werte <= Spielerwert. Höher ist besser: Maximum = 200, Median ≈ 100.
    pct = np.searchsorted(values, float(p), side='right') / len(values)
    return float(np.clip(2 * pct * 100, MIN_METRIC_SCORE_USED, MAX_METRIC_SCORE_USED)), 'Perzentil-Score über Spielerdatei + beide Referenzen'

def compute_metric_score(metric, player_value, benchmark_value, higher_is_better, percentile_distributions=None, max_score_for_zero_case=MAX_METRIC_SCORE_USED):
    if metric in PERCENTILE_SCORE_METRICS:
        return percentile_metric_score(player_value, metric, percentile_distributions)
    p = safe_numeric(player_value)
    b = safe_numeric(benchmark_value)
    if pd.isna(p):
        return np.nan, 'Spielerwert fehlt'
    if pd.isna(b):
        return np.nan, 'Benchmarkwert fehlt'
    if higher_is_better:
        if b == 0:
            if p == 0:
                return 100.0, 'Benchmarkwert = 0 und Spielerwert = 0'
            if p > 0:
                return float(max_score_for_zero_case), 'Benchmarkwert = 0; Score technisch gedeckelt'
            return 0.0, 'Benchmarkwert = 0; Spielerwert <= 0'
        return float(p / b * 100), 'ok'
    if p == 0:
        if b == 0:
            return 100.0, 'Benchmarkwert = 0 und Spielerwert = 0'
        return float(max_score_for_zero_case), 'Spielerwert = 0; Score technisch gedeckelt'
    return float(b / p * 100), 'ok' 

def clip_metric_score(score):
    if pd.isna(score):
        return np.nan
    if APPLY_TECHNICAL_METRIC_CAP:
        return float(np.clip(score, MIN_METRIC_SCORE_USED, MAX_METRIC_SCORE_USED))
    return float(score)

def technical_cap_note(score_raw, score_used):
    if pd.isna(score_raw) or pd.isna(score_used):
        return '', False
    if abs(float(score_raw) - float(score_used)) < 1e-9:
        return '', False
    if float(score_raw) > float(score_used):
        return f'technisch auf {score_used:.0f} gedeckelt (Rohwert {score_raw:.1f})', True
    return f'technisch auf {score_used:.0f} angehoben (Rohwert {score_raw:.1f})', True

def weighted_average(vals, weights):
    pairs = [(v, w) for v, w in zip(vals, weights) if pd.notna(v) and pd.notna(w) and w > 0]
    if not pairs:
        return np.nan
    tw = sum(w for _, w in pairs)
    return sum(v * w for v, w in pairs) / tw

def weighted_geometric_mean(vals, weights, eps=1e-9):
    pairs = [(v, w) for v, w in zip(vals, weights) if pd.notna(v) and pd.notna(w) and w > 0]
    if not pairs:
        return np.nan
    if any(v <= 0 for v, _ in pairs):
        return 0.0
    tw = sum(w for _, w in pairs)
    log_sum = sum(w * math.log(max(v, eps)) for v, w in pairs) / tw
    return float(math.exp(log_sum))

def reliability_from_minutes(minutes):
    m = safe_numeric(minutes)
    if pd.isna(m):
        return 0.50, 'unbekannt'
    if m < 300:
        return 0.35, 'sehr niedrig'
    if m < 600:
        return 0.55, 'niedrig'
    if m < 900:
        return 0.70, 'mittel'
    if m < 1500:
        return 0.85, 'gut'
    return 1.00, 'sehr gut'

def league_factor_for(league, reference_mode=REFERENCE_MODE):
    factors = LEAGUE_FACTORS_BY_REFERENCE.get(reference_mode, LEAGUE_FACTORS_BY_REFERENCE['best_2_bundesliga'])
    default = DEFAULT_LEAGUE_FACTOR_BY_REFERENCE.get(reference_mode, DEFAULT_LEAGUE_FACTOR)
    return float(factors.get(str(league).strip(), default))

def match_class(score):
    if pd.isna(score):
        return 'keine Daten'
    if score >= 100:
        return 'Benchmark+'
    if score >= 85:
        return 'sehr stark'
    if score >= 75:
        return 'stark'
    if score >= 65:
        return 'interessant'
    if score >= 50:
        return 'bedingt'
    return 'schwach'

def data_quality(coverage, rel_weight):
    if pd.isna(coverage):
        return 'niedrig'
    if coverage >= 0.80 and rel_weight >= 0.85:
        return 'hoch'
    if coverage >= 0.60 and rel_weight >= 0.70:
        return 'mittel'
    return 'niedrig'

def evaluate_gates(metric_score_map, core_metrics, coverage, minutes, role_name=None):
    triggered = []
    # Einzelne schwache Core-Metriken wirken nur auf den Role Fit, nicht als harter Score-Cap.
    for metric in core_metrics:
        score = metric_score_map.get(metric, np.nan)
        if pd.isna(score):
            triggered.append({'Gate': f'Fit-Cap: Core-Metrik fehlt: {metric}', 'Gate_Typ': 'core_metric_fit', 'Wert': np.nan, 'Schwelle': np.nan, 'Cap': 85.0})
            continue
        for rule in CORE_METRIC_FIT_CAP_RULES:
            if score < rule['threshold']:
                triggered.append({'Gate': f"{rule['label']}: {metric}", 'Gate_Typ': 'core_metric_fit', 'Wert': score, 'Schwelle': rule['threshold'], 'Cap': rule['cap']})
                break
    # Harte Rollenlevel-Caps erst bei mehreren Core-Defiziten.
    for rule in MULTI_CORE_CAP_RULES:
        n = sum(1 for m in core_metrics if pd.notna(metric_score_map.get(m, np.nan)) and metric_score_map.get(m, np.nan) < rule['threshold'])
        if n >= rule['n_core_below']:
            triggered.append({'Gate': rule['label'], 'Gate_Typ': 'multi_core', 'Wert': n, 'Schwelle': rule['threshold'], 'Cap': rule['cap']})
            break
    for rule in ROLE_SPECIFIC_FIT_CAP_RULES:
        if role_name == rule.get('role'):
            score = metric_score_map.get(rule.get('metric'), np.nan)
            if pd.notna(score) and score < rule['threshold']:
                triggered.append({'Gate': rule['label'], 'Gate_Typ': 'role_specific_fit', 'Wert': score, 'Schwelle': rule['threshold'], 'Cap': rule['cap']})
    for rule in COVERAGE_CAP_RULES:
        if pd.notna(coverage) and coverage < rule['threshold']:
            triggered.append({'Gate': rule['label'], 'Gate_Typ': 'coverage', 'Wert': coverage, 'Schwelle': rule['threshold'], 'Cap': rule['cap']})
            break
    if APPLY_MINUTES_CAPS:
        m = safe_numeric(minutes)
        for rule in MINUTES_CAP_RULES:
            if pd.notna(m) and m < rule['threshold']:
                triggered.append({'Gate': rule['label'], 'Gate_Typ': 'minutes', 'Wert': m, 'Schwelle': rule['threshold'], 'Cap': rule['cap']})
                break
    return triggered

def compute_single_reference(players, roles_long, benchmark_values, reference_mode=REFERENCE_MODE, percentile_distributions=None):
    df = add_derived_metrics(players)
    df['_player_families'] = df['Position'].apply(player_position_families)
    all_records, metric_recs, gate_recs = [], [], []
    bench_by_role = {r: g.copy() for r, g in benchmark_values.groupby('Rolle', sort=False)}
    roles_order = roles_long[['Rolle', 'Positionsfamilie']].drop_duplicates().reset_index(drop=True)
    for _, role_row in roles_order.iterrows():
        role_name = role_row['Rolle']
        role_family = role_row['Positionsfamilie']
        specs = role_metric_specs(role_name, roles_long)
        total_w = sum(s['weight'] for s in specs)
        core = [s['metric'] for s in specs if s['is_core']]
        if APPLY_POSITION_FILTER:
            candidates = df.loc[df['_player_families'].apply(lambda fam: is_role_compatible(fam, role_family))].copy()
        else:
            candidates = df.copy()
        eligible_n = len(candidates)
        bench_role = bench_by_role.get(role_name, pd.DataFrame())
        bench_by_metric = {row['Metrik']: row for _, row in bench_role.iterrows()}
        for _, player in candidates.iterrows():
            ms_raw, ms_used, weights = [], [], []
            fit_comp, fit_w = [], []
            present = 0.0
            score_map = {}
            used_metrics = []
            benchmark_players_used = []
            technical_metric_caps = []
            for spec in specs:
                metric = spec['metric']
                weight = spec['weight']
                higher = spec['higher_is_better']
                br = bench_by_metric.get(metric)
                bv = np.nan if br is None else br.get('Benchmarkwert', np.nan)
                benchmark_player_for_metric = np.nan if br is None else br.get('Benchmarkspieler_fuer_Metrik', np.nan)
                benchmark_old_role_for_metric = np.nan if br is None else br.get('Benchmarkrolle_alt_fuer_Metrik', np.nan)
                pv = player.get(metric, np.nan)
                score_raw, note = compute_metric_score(metric, pv, bv, higher, percentile_distributions=percentile_distributions)
                score_used = clip_metric_score(score_raw)
                cap_note, cap_applied = technical_cap_note(score_raw, score_used)
                if cap_applied:
                    note = f'{note}; {cap_note}' if note else cap_note
                    technical_metric_caps.append(f'{metric}: {score_raw:.1f}→{score_used:.0f}')
                fit_component = np.nan if pd.isna(score_used) else min(score_used, 100.0)
                if pd.notna(score_used):
                    present += weight
                    ms_raw.append(score_raw)
                    ms_used.append(score_used)
                    weights.append(weight)
                    fit_comp.append(fit_component)
                    fit_w.append(weight)
                    score_map[metric] = score_used
                    used_metrics.append(metric)
                    if pd.notna(benchmark_player_for_metric):
                        benchmark_players_used.append(str(benchmark_player_for_metric))
                metric_recs.append({
                    'Spieler': player.get('Spieler', np.nan),
                    'Rolle': role_name,
                    'Positionsfamilie': role_family,
                    'Metrik': metric,
                    'Gewicht': weight,
                    'Is_Core_Metric': spec['is_core'],
                    'Spielerwert': safe_numeric(pv),
                    'Benchmarkwert': safe_numeric(bv),
                    'Metric Score Raw': score_raw,
                    'Metric Score Used for Aggregation': score_used,
                    'Fit Component': fit_component,
                    'Score Note': note,
                    'Technical Cap Applied': cap_applied,
                    'Benchmarkspieler_fuer_Metrik': benchmark_player_for_metric,
                    'Benchmarkrolle_alt_fuer_Metrik': benchmark_old_role_for_metric,
                })
            coverage = np.nan if total_w == 0 else present / total_w
            raw_unclipped = weighted_average(ms_raw, weights)
            raw_score = weighted_average(ms_used, weights)
            profile_completeness = weighted_geometric_mean(fit_comp, fit_w)
            gates = evaluate_gates(score_map, core, coverage, player.get('Minuten', np.nan), role_name=role_name)
            cap = min([g['Cap'] for g in gates if g.get('Gate_Typ') in SCORE_GATE_TYPES], default=np.nan)
            fit_cap = min([g['Cap'] for g in gates if g.get('Gate_Typ') in FIT_GATE_TYPES], default=np.nan)
            capped_score = raw_score if pd.notna(raw_score) else np.nan
            if pd.notna(capped_score) and pd.notna(cap):
                capped_score = min(capped_score, cap)
            league_factor = league_factor_for(player.get('Liga', ''), reference_mode)
            league_adjusted_score = np.nan if pd.isna(capped_score) else capped_score * league_factor
            if reference_mode == 'world_class' and CAP_WORLD_CLASS_LEAGUE_ADJUSTED_AT_100 and pd.notna(league_adjusted_score):
                league_adjusted_score = min(league_adjusted_score, 100.0)
            rel_weight, rel_label = reliability_from_minutes(player.get('Minuten', np.nan))
            base = {col: player[col] if col in player.index else np.nan for col in META_COLS}
            base.update({
                'Rolle': role_name,
                'Positionsfamilie': role_family,
                'Referenzmodus': reference_mode,
                'Benchmarkspieler_Pool': ', '.join(sorted(set(benchmark_players_used))),
                'Raw Role Score Unclipped': raw_unclipped,
                'Raw Role Score': raw_score,
                'Applied Cap': cap,
                'Applied Fit Cap': fit_cap,
                'Capped Role Score': capped_score,
                'Ligafaktor': league_factor,
                'League Adjusted Role Score': league_adjusted_score,
                'Profile Completeness': profile_completeness,
                'Role Fit': np.nan,
                'Role Specificity': np.nan,
                'Best Other Role by Fit Base': np.nan,
                'Fit Gap to Best Other': np.nan,
                'Fit Modifier': np.nan,
                'Final Role Score': np.nan,
                'Reliability Weight': rel_weight,
                'Reliability': rel_label,
                'Reliability Adjusted Final Score': np.nan,
                'Match-Klasse': '',
                'Datenqualität': data_quality(coverage, rel_weight),
                'Coverage': coverage,
                'Eligible_N': eligible_n,
                'Metrics_Used': ', '.join(used_metrics),
                'Core Metrics': ', '.join(core),
                'Triggered Gates': '; '.join(g['Gate'] for g in gates),
                'Technical Metric Caps': '; '.join(technical_metric_caps),
            })
            for col in TRANSFER_COLS_ALL_ROLES:
                base[col] = player[col] if col in player.index else np.nan
            all_records.append(base)
            for g in gates:
                gate_recs.append({'Spieler': player.get('Spieler', np.nan), 'Rolle': role_name, **g, 'Raw Role Score': raw_score, 'Capped Role Score': capped_score})
    all_roles = pd.DataFrame(all_records)
    if not all_roles.empty:
        all_roles['Profile Completeness'] = pd.to_numeric(all_roles['Profile Completeness'], errors='coerce')
        for player_name, idxs in all_roles.groupby('Spieler').groups.items():
            grp = all_roles.loc[idxs]
            for idx, row in grp.iterrows():
                own = row['Profile Completeness']
                other = grp.loc[grp.index != idx, 'Profile Completeness'].dropna()
                best_other = other.max() if len(other) else np.nan
                gap = np.nan if pd.isna(own) or pd.isna(best_other) else own - best_other
                if pd.isna(gap):
                    specificity = np.nan if pd.isna(own) else 100.0
                else:
                    specificity = float(np.clip(SPECIFICITY_NEUTRAL + SPECIFICITY_POINTS_PER_GAP_POINT * gap, 0, 100))
                role_fit = np.nan if pd.isna(own) else own
                fit_cap = row.get('Applied Fit Cap', np.nan)
                if pd.notna(role_fit) and pd.notna(fit_cap):
                    role_fit = min(role_fit, fit_cap)
                fit_modifier = final_score = rel_adjusted = np.nan
                if pd.notna(role_fit) and pd.notna(row['League Adjusted Role Score']):
                    fit_modifier = FIT_MODIFIER_BASE + (1 - FIT_MODIFIER_BASE) * (role_fit / 100)
                    final_score = row['League Adjusted Role Score'] * fit_modifier
                    rel_adjusted = final_score * row['Reliability Weight'] + 60 * (1 - row['Reliability Weight'])
                all_roles.loc[idx, 'Best Other Role by Fit Base'] = best_other
                all_roles.loc[idx, 'Fit Gap to Best Other'] = gap
                all_roles.loc[idx, 'Role Specificity'] = specificity
                all_roles.loc[idx, 'Role Fit'] = role_fit
                all_roles.loc[idx, 'Fit Modifier'] = fit_modifier
                all_roles.loc[idx, 'Final Role Score'] = final_score
                all_roles.loc[idx, 'Reliability Adjusted Final Score'] = rel_adjusted
                all_roles.loc[idx, 'Match-Klasse'] = match_class(final_score)
        num_cols = ['Raw Role Score Unclipped', 'Raw Role Score', 'Applied Cap', 'Applied Fit Cap', 'Capped Role Score', 'League Adjusted Role Score', 'Profile Completeness', 'Best Other Role by Fit Base', 'Fit Gap to Best Other', 'Role Specificity', 'Role Fit', 'Fit Modifier', 'Final Role Score', 'Reliability Adjusted Final Score', 'Coverage', 'Ligafaktor']
        for c in num_cols:
            if c in all_roles.columns:
                all_roles[c] = all_roles[c].round(4 if c in {'Fit Modifier', 'Ligafaktor'} else 2)
        all_roles = all_roles.sort_values(['Spieler', 'Final Role Score', 'Rolle'], ascending=[True, False, True], na_position='last').reset_index(drop=True)
    return all_roles, pd.DataFrame(metric_recs), pd.DataFrame(gate_recs)

def _weighted_role_fit_from_modes(row, weights=ROLE_MATCH_REFERENCE_WEIGHTS):
    vals, ws = [], []
    for mode, weight in weights.items():
        col = f'Role Fit ({mode})'
        if col in row.index and pd.notna(row[col]):
            vals.append(float(row[col]))
            ws.append(float(weight))
    return weighted_average(vals, ws)

def _combine_reference_fits(mode_results, selected_reference_mode):
    """Übernimmt die Scores des ausgewählten Modus, aber ersetzt den Role Fit durch
    die gewichtete, referenzunabhängige Profile Completeness aus beiden Benchmarkmodi.
    """
    selected = mode_results[selected_reference_mode]['all_roles'].copy()
    key = ['Spieler', 'Rolle']
    fit_cols_all = []
    for mode, result in mode_results.items():
        fit_cols = key + ['Profile Completeness', 'Role Specificity', 'Role Fit']
        fit_df = result['all_roles'][fit_cols].copy()
        fit_df = fit_df.rename(columns={
            'Profile Completeness': f'Profile Completeness ({mode})',
            'Role Specificity': f'Role Specificity ({mode})',
            'Role Fit': f'Role Fit ({mode})',
        })
        fit_cols_all.append(fit_df)
    merged = selected.copy()
    # Die ausgewählten Einzel-Fit-Spalten entfernen wir vor dem Merge, um klare Spaltennamen zu behalten.
    for c in ['Profile Completeness', 'Role Specificity', 'Role Fit']:
        if c in merged.columns:
            merged = merged.drop(columns=[c])
    for fit_df in fit_cols_all:
        merged = merged.merge(fit_df, on=key, how='left')
    merged['Role Fit'] = merged.apply(_weighted_role_fit_from_modes, axis=1)
    merged['Unified Role Fit'] = merged['Role Fit']
    merged['Role Match Weights'] = ' + '.join(f"{w:.0%} {m}" for m, w in ROLE_MATCH_REFERENCE_WEIGHTS.items())
    # Lesbare Profil-/Spezifitätsfelder als gewichtete Mittelwerte aus beiden Modi.
    # Role Fit entspricht dabei bewusst der Profile Completeness; Role Specificity bleibt nur Diagnose.
    for base_col in ['Profile Completeness', 'Role Specificity']:
        cols = [f'{base_col} ({m})' for m in ROLE_MATCH_REFERENCE_WEIGHTS]
        merged[base_col] = merged.apply(lambda r: weighted_average([r.get(c, np.nan) for c in cols], list(ROLE_MATCH_REFERENCE_WEIGHTS.values())), axis=1)
    # Final Score mit ausgewähltem Score-Niveau, aber einheitlichem Role Fit neu berechnen.
    merged['Fit Modifier'] = np.nan
    merged['Final Role Score'] = np.nan
    merged['Reliability Adjusted Final Score'] = np.nan
    valid = merged['Role Fit'].notna() & merged['League Adjusted Role Score'].notna()
    merged.loc[valid, 'Fit Modifier'] = FIT_MODIFIER_BASE + (1 - FIT_MODIFIER_BASE) * (merged.loc[valid, 'Role Fit'] / 100)
    merged.loc[valid, 'Final Role Score'] = merged.loc[valid, 'League Adjusted Role Score'] * merged.loc[valid, 'Fit Modifier']
    if selected_reference_mode == 'world_class' and CAP_WORLD_CLASS_LEAGUE_ADJUSTED_AT_100:
        merged['Final Role Score'] = merged['Final Role Score'].clip(upper=100)
    merged['Reliability Adjusted Final Score'] = merged['Final Role Score'] * merged['Reliability Weight'] + 60 * (1 - merged['Reliability Weight'])
    merged['Match-Klasse'] = merged['Final Role Score'].apply(match_class)
    num_cols = ['Profile Completeness', 'Role Specificity', 'Role Fit', 'Unified Role Fit', 'Fit Modifier', 'Final Role Score', 'Reliability Adjusted Final Score']
    for c in num_cols:
        if c in merged.columns:
            merged[c] = pd.to_numeric(merged[c], errors='coerce').round(4 if c == 'Fit Modifier' else 2)
    merged = merged.sort_values(['Spieler', 'Role Fit', 'Final Role Score', 'Rolle'], ascending=[True, False, False, True], na_position='last').reset_index(drop=True)
    return merged

def compute(players, roles_long, benchmark_values_by_mode, reference_mode=REFERENCE_MODE, percentile_distributions=None):
    """Berechnet Scores im gewählten Referenzmodus, aber einen stabilen Role Match
    aus beiden Referenzen. Die Referenz-Auswahl normiert damit nur noch den Score.
    """
    # Backward-compatible: falls nur eine Benchmark-Tabelle übergeben wird.
    if isinstance(benchmark_values_by_mode, pd.DataFrame):
        all_roles, metric_details, gate_checks = compute_single_reference(players, roles_long, benchmark_values_by_mode, reference_mode=reference_mode, percentile_distributions=percentile_distributions)
        metric_details['Scoring_Reference_Mode'] = reference_mode
        gate_checks['Scoring_Reference_Mode'] = reference_mode
        return all_roles, metric_details, gate_checks
    mode_results = {}
    metric_frames, gate_frames = [], []
    for mode, benchmark_values in benchmark_values_by_mode.items():
        ar, md, gc = compute_single_reference(players, roles_long, benchmark_values, reference_mode=mode, percentile_distributions=percentile_distributions)
        md = md.copy(); md['Scoring_Reference_Mode'] = mode
        gc = gc.copy(); gc['Scoring_Reference_Mode'] = mode
        mode_results[mode] = {'all_roles': ar, 'metric_details': md, 'gate_checks': gc}
        metric_frames.append(md)
        gate_frames.append(gc)
    selected = _combine_reference_fits(mode_results, reference_mode)
    metric_details = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
    gate_checks = pd.concat(gate_frames, ignore_index=True) if gate_frames else pd.DataFrame()
    return selected, metric_details, gate_checks


In [ ]:
def reference_label_by_role(benchmark_values):
    labels = {}
    for role, grp in benchmark_values.groupby('Rolle'):
        counts = grp['Benchmarkspieler_fuer_Metrik'].dropna().astype(str).value_counts()
        if len(counts):
            labels[role] = counts.index[0]
        else:
            pool = ', '.join(grp['Referenzspieler_pool'].dropna().astype(str).unique())
            labels[role] = pool.split(', ')[0] if pool else np.nan
    return labels

def build_top_matches(all_roles, players, benchmark_values, rank_by='Role Fit'):
    ref_labels = reference_label_by_role(benchmark_values)
    rows = []
    grouped = all_roles.sort_values(['Spieler', rank_by, 'Final Role Score', 'Rolle'], ascending=[True, False, False, True], na_position='last').groupby('Spieler', sort=False)
    for player_name, grp in grouped:
        grp = grp.sort_values([rank_by, 'Final Role Score', 'Rolle'], ascending=[False, False, True], na_position='last').reset_index(drop=True)
        top = grp.iloc[0]
        player_row = players.loc[players['Spieler'].eq(player_name)].iloc[0]
        row = {col: player_row[col] if col in player_row.index else np.nan for col in META_COLS}
        row.update({
            'Ranking Metric': 'Unified Role Fit',
            'Top Rolle': top['Rolle'],
            'Top Referenzspieler': ref_labels.get(top['Rolle'], np.nan),
            'Top Final Role Score': top['Final Role Score'],
            'Top League Adjusted Role Score': top['League Adjusted Role Score'],
            'Top Capped Role Score': top['Capped Role Score'],
            'Top Raw Role Score': top['Raw Role Score'],
            'Top Profile Completeness': top['Profile Completeness'],
            'Top Role Specificity': top['Role Specificity'],
            'Top Role Fit': top['Role Fit'],
            'Top Unified Role Fit': top.get('Unified Role Fit', top['Role Fit']),
            'Top Role Fit 2BL': top.get('Role Fit (best_2_bundesliga)', np.nan),
            'Top Role Fit Weltklasse': top.get('Role Fit (world_class)', np.nan),
            'Top Fit Modifier': top['Fit Modifier'],
            'Top Reliability Adjusted Final Score': top['Reliability Adjusted Final Score'],
            'Match-Klasse': top['Match-Klasse'],
            'Datenqualität': top['Datenqualität'],
            'Reliability': top['Reliability'],
            'Coverage': top['Coverage'],
            'Kandidaten im Rollenset': top['Eligible_N'],
            'Ligafaktor': top['Ligafaktor'],
            'Referenzmodus': top['Referenzmodus'],
            'Benchmarkspieler_Pool Top-Rolle': top['Benchmarkspieler_Pool'],
            'Zweitrolle': grp.iloc[1]['Rolle'] if len(grp) > 1 else np.nan,
            'Zweitrolle Final Role Score': grp.iloc[1]['Final Role Score'] if len(grp) > 1 else np.nan,
            'Drittrolle': grp.iloc[2]['Rolle'] if len(grp) > 2 else np.nan,
            'Drittrolle Final Role Score': grp.iloc[2]['Final Role Score'] if len(grp) > 2 else np.nan,
            'Triggered Gates Top-Rolle': top['Triggered Gates'],
            'Technical Metric Caps Top-Rolle': top.get('Technical Metric Caps', np.nan),
            'Metrics_Used Top-Rolle': top['Metrics_Used'],
        })
        for col in TRANSFER_COLS_TOP:
            row[col] = player_row[col] if col in player_row.index else np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values('Top Final Role Score', ascending=False).reset_index(drop=True)

def build_synthetic_benchmark_candidates(benchmark_values, reference_mode):
    rows = []
    for role_name, grp in benchmark_values.groupby('Rolle', sort=False):
        cfg = SIMPLIFIED_ROLE_CONFIG[role_name]
        row = {
            'Spieler': f'SYNTHETIC_BENCHMARK__{role_name}',
            'Alter': np.nan,
            'Position': {'GK': 'GK', 'CB': 'CB', 'FB': 'RB', 'CM/CDM': 'CM', 'AM/Wing': 'RW', 'Wing/ST': 'ST'}.get(cfg['family'], 'CM'),
            'Minuten': 3000,
            'Team': 'Synthetic Benchmark',
            'Liga': '2. Bundesliga' if reference_mode == 'best_2_bundesliga' else 'World Class Reference',
            'Vergleichsgruppe': 'Synthetic',
            'Kontext': f'{reference_mode} synthetic max-by-metric benchmark',
        }
        for _, r in grp.iterrows():
            row[r['Metrik']] = r['Benchmarkwert']
        rows.append(row)
    return pd.DataFrame(rows)

def build_methodik_table(reference_mode):
    rows = [
        {'Parameter': 'REFERENCE_MODE', 'Wert': reference_mode, 'Beschreibung': 'Gewählter Benchmark-Modus.'},
        {'Parameter': 'Simplified Roles v14', 'Wert': str(len(SIMPLIFIED_ROLE_CONFIG)), 'Beschreibung': 'Deutsch benannte Makro-Rollen mit überarbeiteten Gewichten.'},
        {'Parameter': 'Role Match', 'Wert': '70% best_2_bundesliga + 30% world_class', 'Beschreibung': 'Top-Rolle wird anhand einer referenzunabhängigen, gewichteten Profile Completeness gewählt.'},
        {'Parameter': 'Reference selection', 'Wert': 'normiert nur Score, nicht Top-Rolle', 'Beschreibung': 'Das Dropdown bestimmt Raw/Capped/League-adjusted/Final Score, aber nicht den Rollenmatch.'},
        {'Parameter': 'Umbenennungen', 'Wert': 'Moderner TW, Mitspielender IV, Defensiver IV, Offensiver AV, Defensiver AV', 'Beschreibung': 'Englische Rollenbezeichnungen wurden in deutsche Rollennamen überführt.'},
        {'Parameter': 'Nachschärfung', 'Wert': 'Rollen v10/v14, weichere Gates, Acceleration-Percentile', 'Beschreibung': 'Rollen-Gewichte wurden überarbeitet; einzelne Core-Defizite deckeln nur den Role Fit; Acceleration with Ball wird über einen Perzentilscore aus Spieler- und Referenzdaten bewertet.'},
        {'Parameter': 'Excluded Metrics', 'Wert': ', '.join(sorted(METRICS_TO_EXCLUDE)), 'Beschreibung': 'Diese Metriken werden aus allen Rollen entfernt.'},
        {'Parameter': 'Benchmark aggregation', 'Wert': 'max je Metrik über gemappte Referenzspieler', 'Beschreibung': 'Alle Rollenmetriken sind höher-ist-besser; Acceleration with Ball nutzt statt Benchmark-Quote einen Perzentilscore.'},
        {'Parameter': 'Raw Role Score', 'Wert': 'gewichteter Durchschnitt der Metric Score Used', 'Beschreibung': 'Direkter Benchmarkvergleich vor Gates, Liga und Role-Fit-Modifier.'},
        {'Parameter': 'Capped Role Score', 'Wert': 'Raw Role Score nach harten Gates/Caps', 'Beschreibung': 'Harte Score-Caps nur bei mehreren Core-Defiziten, Coverage-Problemen oder optional Minuten-Caps.'},
        {'Parameter': 'League Adjusted Role Score', 'Wert': 'Capped Role Score * referenzmodus-spezifischer Ligafaktor', 'Beschreibung': 'Gegen Weltklasse gilt 2. Bundesliga = 0.70; Weltklasse-Referenz = 1.00.'},
        {'Parameter': 'Role Fit', 'Wert': 'Profile Completeness', 'Beschreibung': 'Profil-Fit auf Basis der rollentypischen Metrikvollständigkeit; Role Specificity bleibt nur eine Diagnosegröße und fließt nicht mehr in Fit oder Score ein.'},
        {'Parameter': 'Final Role Score', 'Wert': 'League Adjusted Role Score * Fit Modifier', 'Beschreibung': 'Standard-Ranking-Score; in der GUI als Score angezeigt.'},
        {'Parameter': 'Acceleration with Ball', 'Wert': '2 * Perzentilrang über Spieler + Referenzen', 'Beschreibung': 'Median ≈ 100, Maximum = 200, Minimum nahe 0; kein Benchmark-Quotient.'},
    ]
    return pd.DataFrame(rows)

def build_export_tables(players, old_roles, roles_long, references, benchmark_values_by_mode, benchmark_values_all, dropped_metrics, all_roles, metric_details, gate_checks, reference_mode, percentile_distributions=None):
    selected_benchmark_values = benchmark_values_by_mode[reference_mode]
    top_matches = build_top_matches(all_roles, players, selected_benchmark_values)
    summary = top_matches.head(15).copy()
    role_distribution = top_matches['Top Rolle'].value_counts().rename_axis('Rolle').reset_index(name='Anzahl als Top Match')
    role_coverage = all_roles.groupby('Rolle', as_index=False).agg(
        Spieler_Rollen_Kombinationen=('Spieler', 'count'),
        Avg_Coverage=('Coverage', 'mean'),
        Avg_Final_Role_Score=('Final Role Score', 'mean'),
        Max_Final_Role_Score=('Final Role Score', 'max'),
        Avg_Role_Fit=('Role Fit', 'mean'),
        Max_Raw_Role_Score=('Raw Role Score', 'max'),
    ).sort_values('Rolle')
    synthetic_candidates = build_synthetic_benchmark_candidates(selected_benchmark_values, reference_mode)
    synthetic_roles, _, _ = compute_single_reference(synthetic_candidates, roles_long, selected_benchmark_values, reference_mode=reference_mode, percentile_distributions=percentile_distributions)
    synthetic_self_check = synthetic_roles.loc[
        synthetic_roles.apply(lambda r: r['Spieler'] == f"SYNTHETIC_BENCHMARK__{r['Rolle']}", axis=1)
    ].copy()
    synthetic_self_check = synthetic_self_check[[
        'Referenzmodus', 'Spieler', 'Rolle', 'Raw Role Score', 'Capped Role Score', 'League Adjusted Role Score',
        'Profile Completeness', 'Role Specificity', 'Role Fit', 'Final Role Score', 'Coverage', 'Triggered Gates'
    ]].sort_values('Rolle')
    reference_candidates = references.loc[references['Referenzmodus'].eq(reference_mode) & references['Rolle'].notna()].copy()
    reference_player_roles, _, _ = compute_single_reference(reference_candidates, roles_long, selected_benchmark_values, reference_mode=reference_mode, percentile_distributions=percentile_distributions)
    reference_player_check = reference_player_roles.merge(
        reference_candidates[['Spieler', 'Rolle', 'Rolle_raw']].rename(columns={'Rolle': 'Soll-Rolle', 'Rolle_raw': 'Alte Referenzrolle'}),
        on='Spieler', how='left'
    )
    reference_player_check = reference_player_check.loc[reference_player_check['Rolle'].eq(reference_player_check['Soll-Rolle'])].copy()
    keep_cols = ['Referenzmodus', 'Spieler', 'Alte Referenzrolle', 'Rolle', 'Raw Role Score', 'Capped Role Score', 'League Adjusted Role Score', 'Profile Completeness', 'Role Specificity', 'Role Fit', 'Final Role Score', 'Coverage', 'Triggered Gates']
    reference_player_check = reference_player_check[[c for c in keep_cols if c in reference_player_check.columns]].sort_values(['Rolle', 'Spieler'])
    league_factor_table = pd.DataFrame([{'Referenzmodus': mode, 'Liga': k, 'Ligafaktor': v} for mode, factors in LEAGUE_FACTORS_BY_REFERENCE.items() for k, v in factors.items()]).sort_values(['Referenzmodus', 'Liga'])
    gate_config = pd.concat([
        pd.DataFrame(CORE_METRIC_FIT_CAP_RULES).assign(Regeltyp='single_core_fit'),
        pd.DataFrame(MULTI_CORE_CAP_RULES).assign(Regeltyp='multi_core'),
        pd.DataFrame(COVERAGE_CAP_RULES).assign(Regeltyp='coverage'),
        pd.DataFrame(ROLE_SPECIFIC_FIT_CAP_RULES).assign(Regeltyp='role_specific_fit'),
    ], ignore_index=True, sort=False)
    source_role_mapping = pd.DataFrame([{'Old Role Alias': k, 'New Simplified Role': v} for k, v in OLD_TO_NEW_ROLE_MAP.items()]).sort_values(['New Simplified Role', 'Old Role Alias'])
    methodik = build_methodik_table(reference_mode)
    return {
        'Summary': summary,
        'Top_Matches': top_matches,
        'Alle_Rollen': all_roles,
        'Metrik_Details': metric_details,
        'Gate_Checks': gate_checks,
        'Synthetic_Self_Check': synthetic_self_check,
        'Reference_Player_Check': reference_player_check,
        'Role_Distribution': role_distribution,
        'Role_Coverage': role_coverage,
        'Rollendefinitionen_v10': roles_long,
        'Legacy_Role_Mapping': old_roles,
        'Source_Role_Mapping': source_role_mapping,
        'Benchmarkwerte': benchmark_values_all,
        'Dropped_Metrics': dropped_metrics,
        'League_Factors': league_factor_table,
        'Gate_Config': gate_config,
        'Methodik': methodik,
    }

def export_tables_to_excel(tables, output_excel):
    output_excel = Path(output_excel)
    with pd.ExcelWriter(output_excel, engine='xlsxwriter') as writer:
        for sheet_name, df in tables.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        workbook = writer.book
        header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D9EAF7', 'border': 1})
        for sheet_name, df in tables.items():
            ws = writer.sheets[sheet_name[:31]]
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, max(len(df), 1), max(len(df.columns) - 1, 0))
            for col_idx, col in enumerate(df.columns):
                ws.write(0, col_idx, col, header_fmt)
                if len(df):
                    series = df[col].astype(str)
                    width = min(max(len(str(col)) + 2, series.str.len().quantile(0.90) + 2), 45)
                else:
                    width = min(max(len(str(col)) + 2, 12), 45)
                ws.set_column(col_idx, col_idx, width)
    return output_excel

def run_pipeline(reference_mode=REFERENCE_MODE, original_excel=None, world_reference_excel=None, second_reference_excel=None, output_excel=None, export_excel=True):
    original_excel = Path(original_excel or ORIGINAL_EXCEL)
    world_reference_excel = Path(world_reference_excel or WORLD_CLASS_REFERENCE_EXCEL)
    second_reference_excel = Path(second_reference_excel or SECOND_LEAGUE_REFERENCE_EXCEL)
    output_excel = Path(output_excel or (DATA_DIR / f'best11_benchmark_rollenfits_v14_gui_{reference_mode}.xlsx'))
    if Path('/mnt/data').exists() and original_excel.parent == Path('/mnt/data'):
        output_excel = Path('/mnt/data') / f'best11_benchmark_rollenfits_v14_gui_{reference_mode}.xlsx'
    players, legacy_roles, roles_base, references = load_all_inputs(original_excel, world_reference_excel, second_reference_excel)
    roles_long, benchmark_values_by_mode, benchmark_values_all, dropped_metrics = activate_roles_for_benchmarks(roles_base, references, REFERENCE_MODES)
    percentile_distributions = build_percentile_distributions(players, references)
    all_roles, metric_details, gate_checks = compute(players, roles_long, benchmark_values_by_mode, reference_mode=reference_mode, percentile_distributions=percentile_distributions)
    tables = build_export_tables(players, legacy_roles, roles_long, references, benchmark_values_by_mode, benchmark_values_all, dropped_metrics, all_roles, metric_details, gate_checks, reference_mode, percentile_distributions=percentile_distributions)
    if export_excel:
        export_tables_to_excel(tables, output_excel)
    return {
        'reference_mode': reference_mode,
        'players': players,
        'legacy_roles': legacy_roles,
        'roles_long': roles_long,
        'references': references,
        'benchmark_values': benchmark_values_all,
        'benchmark_values_by_mode': benchmark_values_by_mode,
        'dropped_metrics': dropped_metrics,
        'percentile_distributions': percentile_distributions,
        'all_roles': all_roles,
        'metric_details': metric_details,
        'gate_checks': gate_checks,
        'tables': tables,
        'top_matches': tables['Top_Matches'],
        'role_distribution': tables['Role_Distribution'],
        'output_excel': output_excel,
    }

In [ ]:
# ---------------------------------------------------------------------
# GUI-Hilfen für Google Colab / Jupyter
# ---------------------------------------------------------------------
def _fmt(value, digits=1, na='–'):
    if pd.isna(value):
        return na
    if isinstance(value, (int, float, np.number)):
        return f'{float(value):.{digits}f}'
    return str(value)

def _escape(value, na='–'):
    if pd.isna(value):
        return na
    return html.escape(str(value))

def _first_nonempty(*values, default='–'):
    for value in values:
        if pd.notna(value) and str(value).strip():
            return value
    return default

def role_definition_text(role, roles_long):
    if role not in SIMPLIFIED_ROLE_CONFIG:
        return ''
    cfg = SIMPLIFIED_ROLE_CONFIG[role]
    return str(cfg.get('definition', '')).strip()

def display_metric_label(metric):
    return re.sub(r'\s*\(derived\)\s*$', '', str(metric)).strip()

def display_position_label(position):
    """Gibt Positionskürzel in nutzerfreundlichen deutschen Kürzeln aus."""
    raw = str(position or '').strip()
    if not raw or raw.lower() == 'nan':
        return '–'
    key = raw.upper()
    mapping = {
        'GK': 'TW', 'GOALKEEPER': 'TW', 'TW': 'TW',
        'CB': 'IV', 'LCB': 'IV', 'RCB': 'IV',
        'LB': 'LV', 'LWB': 'LV',
        'RB': 'RV', 'RWB': 'RV',
        'FB': 'AV',
        'DM': 'DM', 'DMF': 'DM', 'LDMF': 'DM', 'RDMF': 'DM',
        'CM': 'ZM', 'CMF': 'ZM', 'LCMF': 'ZM', 'RCMF': 'ZM', 'LCMF3': 'ZM', 'RCMF3': 'ZM',
        'AM': 'OM', 'AMF': 'OM', 'CAM': 'OM',
        'LW': 'Flügel', 'RW': 'Flügel', 'LWF': 'Flügel', 'RWF': 'Flügel', 'LM': 'Flügel', 'RM': 'Flügel',
        'ST': 'ST', 'CF': 'ST', 'FW': 'ST',
    }
    return html.escape(mapping.get(key, raw))

def metric_score_class(score):
    v = safe_numeric(score)
    if pd.isna(v):
        return 'score-na'
    if v >= 120:
        return 'score-darkgreen'
    if v >= 90:
        return 'score-lightgreen'
    if v >= 75:
        return 'score-yellow'
    if v >= 60:
        return 'score-orange'
    return 'score-red'

# Optional: Hier können später Bilddateien für Score-Badges hinterlegt werden.
# Solange die Pfade leer sind, bleibt das Kartendesign unverändert.
BADGE_IMAGE_PATHS = {
    'gold': None,    # z.B. 'gold_badge.png'
    'silver': None,  # z.B. 'silver_badge.png'
    'bronze': None,  # z.B. 'bronze_badge.png'
}

# Standardmäßig deaktiviert. Später kann das mit True + gültigen BADGE_IMAGE_PATHS wieder aktiviert werden.
ENABLE_SCORE_BADGES = False

def score_medal_badge_html(score):
    if not ENABLE_SCORE_BADGES:
        return ''
    v = safe_numeric(score)
    if pd.isna(v):
        return ''
    if v > 90:
        medal = 'gold'
        alt = 'Gold-Badge'
    elif v > 75:
        medal = 'silver'
        alt = 'Silber-Badge'
    elif v > 60:
        medal = 'bronze'
        alt = 'Bronze-Badge'
    else:
        return ''
    path = BADGE_IMAGE_PATHS.get(medal)
    if not path:
        return ''
    return f"<img class='medal-badge medal-{medal}' src='{html.escape(str(path), quote=True)}' alt='{alt}' title='{alt}'>"



def _hover_marker_html(tooltip, label='Mehr Informationen'):
    """Kleines Lupen-Symbol für Felder mit erklärendem Hover-Text."""
    if tooltip is None or (isinstance(tooltip, float) and pd.isna(tooltip)) or not str(tooltip).strip():
        return ''
    tooltip_attr = html.escape(str(tooltip).strip(), quote=True)
    label_attr = html.escape(str(label), quote=True)
    return f"<span class='hover-marker' title='{tooltip_attr}' aria-label='{label_attr}'>🔍</span>"

def _role_row_for_player(state, player_name, role):
    if pd.isna(role) or not str(role).strip():
        return None
    all_roles = state['all_roles']
    rows = all_roles.loc[
        all_roles['Spieler'].astype(str).eq(str(player_name))
        & all_roles['Rolle'].astype(str).eq(str(role))
    ]
    if rows.empty:
        return None
    return rows.iloc[0]

def _benchmark_players_for_role(state, player_name, role):
    """Für die plakative Steckbriefanzeige immer den Weltklasse-Referenzspieler nutzen."""
    bv = state.get('benchmark_values', pd.DataFrame())
    if isinstance(bv, pd.DataFrame) and not bv.empty:
        filt = bv[
            bv['Referenzmodus'].astype(str).eq('world_class')
            & bv['Rolle'].astype(str).eq(str(role))
        ]
        refs = [str(x).strip() for x in filt.get('Benchmarkspieler_fuer_Metrik', pd.Series(dtype=object)).dropna() if str(x).strip()]
        if refs:
            counts = pd.Series(refs).value_counts()
            return str(counts.index[0])
        pools = [str(x).strip() for x in filt.get('Referenzspieler_pool', pd.Series(dtype=object)).dropna() if str(x).strip()]
        if pools:
            return pools[0].split(', ')[0]
    return '–' 

def _role_metrics_html(state, player_name, role, max_rows=12):
    md = state.get('metric_details', pd.DataFrame())
    if md.empty or pd.isna(role):
        return "<div class='muted'>Keine Attributdetails verfügbar.</div>"
    ref_mode = state.get('reference_mode', REFERENCE_MODE)
    filt = md[
        md['Spieler'].astype(str).eq(str(player_name))
        & md['Rolle'].astype(str).eq(str(role))
    ].copy()
    if 'Scoring_Reference_Mode' in filt.columns:
        filt = filt[filt['Scoring_Reference_Mode'].astype(str).eq(str(ref_mode))]
    if filt.empty:
        return "<div class='muted'>Keine Attributdetails für diese Rolle im aktuellen Referenzmodus verfügbar.</div>"
    if 'Gewicht' in filt.columns:
        filt = filt.sort_values('Gewicht', ascending=False)
    rows = []
    for _, r in filt.head(max_rows).iterrows():
        metric = _escape(display_metric_label(r.get('Metrik', '')))
        weight = _fmt(float(r.get('Gewicht', 0)) * 100 if pd.notna(r.get('Gewicht', np.nan)) else np.nan, 0)
        player_val = _fmt(r.get('Spielerwert', np.nan), 2)
        bench_val = _fmt(r.get('Benchmarkwert', np.nan), 2)
        score_num = r.get('Metric Score Used for Aggregation', r.get('Metric Score Raw', np.nan))
        score_val = _fmt(score_num, 0)
        score_cls = metric_score_class(score_num)
        rows.append(
            f"<tr>"
            f"<td>{metric}</td>"
            f"<td>{weight}%</td>"
            f"<td>{player_val}</td>"
            f"<td>{bench_val}</td>"
            f"<td class='{score_cls}'>{score_val}</td>"
            f"</tr>"
        )
    hidden_note = ''
    if len(filt) > max_rows:
        hidden_note = f"<div class='muted small'>+ {len(filt) - max_rows} weitere Attribute nicht angezeigt.</div>"
    return f"""
      <table class='metric-table'>
        <thead><tr><th>Attribut (pro 90)</th><th>Gewicht</th><th>Wert</th><th>Benchmark</th><th>Metrik-Score</th></tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
      {hidden_note}
    """


def _slugify_filename(value):
    text = str(value).strip().lower()
    text = re.sub(r'[^a-z0-9äöüß]+', '_', text, flags=re.IGNORECASE)
    text = re.sub(r'_+', '_', text).strip('_')
    return text or 'spieler'

def _standalone_profile_html(profile_inner_html, player_name='spielerprofil'):
    """Macht aus dem Steckbrief-HTML eine einfache standalone HTML-Datei."""
    return f"""<!doctype html>
<html lang="de">
<head>
<meta charset="utf-8">
<title>{html.escape(str(player_name))} – Rollenprofil</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
</head>
<body style="margin:24px;background:#f3f4f6;">
{profile_inner_html}
</body>
</html>"""

def _role_notes_html(role_row=None, top_row=None, top_role=False):
    notes = []
    if top_row is not None and top_role:
        dq = top_row.get('Datenqualität', np.nan)
        gates = top_row.get('Triggered Gates Top-Rolle', '')
        metric_caps = top_row.get('Technical Metric Caps Top-Rolle', '')
    elif role_row is not None:
        dq = role_row.get('Datenqualität', np.nan)
        gates = role_row.get('Triggered Gates', '')
        metric_caps = role_row.get('Technical Metric Caps', '')
    else:
        dq = np.nan
        gates = ''
        metric_caps = ''

    if pd.notna(dq) and str(dq).strip():
        notes.append(f"Datenqualität: {_escape(dq)}")
    else:
        notes.append('Datenqualität: keine auffällige Einschränkung')

    cap_parts = []
    if pd.notna(gates) and str(gates).strip():
        cap_parts.append(_escape(gates))
    if pd.notna(metric_caps) and str(metric_caps).strip():
        cap_parts.append('Metrik-Score gedeckelt: ' + _escape(metric_caps))
    if cap_parts:
        notes.append('Caps/Gates: ' + ' | '.join(cap_parts))
    else:
        notes.append('Caps/Gates: keine')

    return '<br>'.join(notes)

def _role_card_html(state, player_name, role, rank_label, top_row=None, is_top=False):
    if pd.isna(role) or not str(role).strip():
        return f"""
        <div class='role-card empty-card'>
          <div class='rank'>{html.escape(rank_label)}</div>
          <h3>Keine weitere Rolle</h3>
          <p class='muted'>Für diesen Spieler wurde keine weitere kompatible Rolle berechnet.</p>
        </div>
        """

    role = str(role)
    role_row = _role_row_for_player(state, player_name, role)
    if is_top and top_row is not None:
        score = top_row.get('Top Final Role Score', np.nan)
        role_fit = top_row.get('Top Role Fit', np.nan)
    elif role_row is not None:
        score = role_row.get('Final Role Score', np.nan)
        role_fit = role_row.get('Role Fit', role_row.get('Unified Role Fit', np.nan))
    else:
        score = np.nan
        role_fit = np.nan

    ref_players = _benchmark_players_for_role(state, player_name, role)
    definition = role_definition_text(role, state['roles_long'])
    role_definition_html = (
        f"<div class='role-definition'>{_escape(definition)}</div>"
        if definition else ""
    )
    attribute_tooltip = 'Die Attribute sind Wyscout-Daten, die mithilfe von Best11Scouting.streamlit.app ausgelesen wurden.'
    attribute_info_icon = _hover_marker_html(attribute_tooltip, 'Attribut-Herkunft anzeigen')
    metric_html = _role_metrics_html(state, player_name, role)
    notes_html = _role_notes_html(role_row=role_row, top_row=top_row, top_role=is_top)
    ref_mode_label = '2. Bundesliga' if state.get('reference_mode') == 'best_2_bundesliga' else 'Weltklasse'
    medal_badge = score_medal_badge_html(score)

    return f"""
    <div class='role-card'>
      <div class='rank'>{html.escape(rank_label)}</div>
      <div class='score-badge'>{_fmt(score, 1)}</div>
      {medal_badge}
      <h3 class='role-name'>{html.escape(role)}</h3>
      {role_definition_html}
      <div class='role-subtitle'>Score gegen Referenz: {html.escape(ref_mode_label)}</div>

      <div class='role-line'><span>Rollen-Match</span><b>{_fmt(role_fit, 1)}</b></div>
      <div class='role-line'><span>Idealspieler</span><b>{html.escape(ref_players)}</b></div>

      <div class='section-title'>Attribute {attribute_info_icon}</div>
      {metric_html}

      <div class='section-title'>Notizen</div>
      <div class='notes-box'>{notes_html}</div>
    </div>
    """

def player_profile_html(player_name, state):
    tm = state['top_matches']
    row_df = tm.loc[tm['Spieler'].astype(str).eq(str(player_name))]
    if row_df.empty:
        return f"<div>Kein Spieler gefunden: {html.escape(str(player_name))}</div>"

    row = row_df.iloc[0]
    player_name = str(row.get('Spieler', player_name))
    top_role = row.get('Top Rolle', np.nan)
    second_role = row.get('Zweitrolle', np.nan)
    liga = row.get('Liga', np.nan)
    league_factor = row.get('Ligafaktor', np.nan)
    liga_display = _escape(liga)
    league_factor_display = _fmt(league_factor, 2) if pd.notna(league_factor) else '–'
    league_factor_tooltip_text = 'Geschätzter Ligenkoeffizient: Die Werte orientieren sich näherungsweise an Opta-Koeffizienten und durchschnittlichen Marktwerten.'
    league_factor_tooltip = html.escape(league_factor_tooltip_text, quote=True)
    league_factor_info_icon = _hover_marker_html(league_factor_tooltip_text, 'Ligenkoeffizient erklären')

    url = row.get('TM Profil-URL', np.nan)
    url_html = f"<a href='{html.escape(str(url))}' target='_blank'>Transfermarkt-Profil</a>" if pd.notna(url) and str(url).startswith('http') else '–'
    top_card = _role_card_html(state, player_name, top_role, 'Top-Rolle', top_row=row, is_top=True)
    second_card = _role_card_html(state, player_name, second_role, 'Zweitbeste Rolle', top_row=row, is_top=False)
    profile_id = 'profile_' + _slugify_filename(player_name)
    player_title = html.escape(player_name, quote=True)
    # Druck/Export wird in der App über einen echten ipywidgets-Button gesteuert.


    profile_logo_html = (
        f"<img src='{html.escape(LOGO_URI, quote=True)}' alt='ClubProfile Logo' style='height:54px;width:auto;object-fit:contain;'>"
        if 'LOGO_URI' in globals() and LOGO_URI else
        "<div style='height:44px;width:44px;border-radius:12px;background:#8B0000;color:white;display:flex;align-items:center;justify-content:center;font-weight:900;'>CP</div>"
    )

    html_block = f'''
    <style>
      .profile-wrap {{
        font-family: Arial, sans-serif;
        width: 100%; max-width: 1180px; box-sizing: border-box; overflow-x: hidden;
        color: #1f2933;
      }}
      .export-actions {{
        display:flex;
        justify-content:flex-end;
        gap:8px;
        margin: 0 0 12px 0;
      }}
      .export-actions button {{
        border:1px solid #d1d5db;
        border-radius:10px;
        padding:8px 11px;
        background:#ffffff;
        cursor:pointer;
        font-weight:700;
        color:#374151;
      }}
      .export-actions button:hover {{background:#f3f4f6;}}
      .info-panel {{
        border: 1px solid #e1e5ea;
        border-radius: 18px;
        padding: 18px 20px;
        margin-bottom: 18px;
        background: linear-gradient(180deg, #ffffff 0%, #f8fafc 100%);
        box-shadow: 0 3px 14px rgba(0,0,0,.06);
      }}
      .info-header {{display:flex; justify-content:space-between; gap:18px; align-items:flex-start; margin-bottom: 12px;}}
      .info-header h2 {{margin:0; font-size: 38px; line-height:1.02; letter-spacing:.1px; font-family:'Arial Black','Trebuchet MS',Arial,sans-serif; font-weight:950; color:#8B0000;}}
      .info-chips {{display:flex; gap:10px; flex-wrap:wrap; justify-content:flex-end; align-items:stretch;}}
      .profile-logo-chip {{display:flex;align-items:center;justify-content:center;min-width:68px;padding:6px 8px;border:1px solid #d9e1ea;border-radius:16px;background:#fff;box-shadow:0 2px 8px rgba(0,0,0,.04);}}
      .info-chip {{border: 1px solid #d9e1ea; border-radius: 16px; padding: 9px 13px; background: #fff; color:#405064; min-width: 104px; box-shadow: 0 2px 8px rgba(0,0,0,.04);}}
      .info-chip .chip-label {{display:block; font-size: 10px; text-transform:uppercase; letter-spacing:.07em; color:#6b7785; margin-bottom:3px; font-weight:800;}}
      .info-chip .chip-value {{display:block; font-size: 17px; line-height:1.05; font-weight:900; color:#111827; text-align:right;}}
      .info-grid {{display:grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 10px;}}
      .info-item {{background:#fff; border: 1px solid #edf0f4; border-radius: 12px; padding: 9px 11px;}}
      .info-item.wide {{grid-column: span 2;}}
      .info-item.full {{grid-column: 1 / -1;}}
      .info-label {{font-size: 11px; text-transform: uppercase; letter-spacing: .06em; color:#6b7785; margin-bottom: 4px;}}
      .info-value {{font-weight: 700; font-size: 14px; word-break: break-word;}}
      .cards-grid {{display:grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 18px;}}
      .role-card {{
        position: relative;
        border: 1px solid #d9dee7;
        border-radius: 22px;
        padding: 18px 18px 16px 18px;
        min-height: 445px;
        background: radial-gradient(circle at top right, #f2f6fb 0%, #ffffff 38%, #fbfcfe 100%);
        box-shadow: 0 8px 24px rgba(0,0,0,.08);
        overflow: hidden;
      }}
      .role-card:before {{
        content: '';
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        height: 7px;
        background: linear-gradient(90deg, #111827, #6b7280);
      }}
      .empty-card {{background:#fafafa;}}
      .rank {{font-size: 12px; color:#6b7280; text-transform:uppercase; letter-spacing:.08em; font-weight:700; margin-top:4px;}}
      .score-badge {{
        position:absolute;
        top: 18px;
        right: 22px;
        width: 118px;
        text-align:right;
        font-size: 32px;
        font-weight: 900;
        line-height: 1;
        color: #111827;
        z-index: 3;
      }}
      .score-badge:after {{content: ' Score'; display:block; font-size: 11px; font-weight:700; color:#6b7280; text-align:right; margin-top:4px;}}
      .medal-badge {{position:absolute; top: 68px; right: 24px; width: 76px; height: 76px; object-fit: contain; z-index: 2; filter: drop-shadow(0 5px 10px rgba(0,0,0,.16));}}
      .role-name {{font-size: 22px; margin: 20px 128px 4px 0; line-height:1.15;}}
      .role-definition {{font-size: 12.5px; line-height:1.35; color:#4b5563; margin: 0 128px 12px 0;}}
      .role-subtitle {{font-size: 12px; color:#6b7280; margin: 0 128px 12px 0;}}
      .role-line {{display:flex; justify-content:space-between; gap:12px; padding: 8px 0; border-bottom: 1px solid #eef1f5; font-size: 14px;}}
      .role-line span {{color:#5f6b7a;}}
      .role-line b {{text-align:right;}}
      .section-title {{margin-top: 14px; margin-bottom: 8px; font-size: 12px; color:#374151; text-transform:uppercase; letter-spacing:.08em; font-weight:900;}}
      .hover-marker {{
        display:inline-flex;
        align-items:center;
        justify-content:center;
        width: 16px;
        height: 16px;
        margin-left: 4px;
        border-radius: 50%;
        background:#eef2f7;
        color:#334155;
        font-size: 10px;
        line-height: 1;
        vertical-align: middle;
        cursor: help;
        text-transform:none;
        letter-spacing:0;
      }}
      .hover-marker:hover {{background:#dbeafe; color:#1d4ed8;}}
      .metric-table {{width:100%; border-collapse: collapse; font-size: 12px; background:#fff; border-radius: 12px; overflow:hidden;}}
      .metric-table th {{text-align:left; background:#f3f5f8; color:#4b5563; padding: 7px 8px; font-weight:800;}}
      .metric-table td {{padding: 7px 8px; border-top: 1px solid #eef1f5; vertical-align: top;}}
      .metric-table td:nth-child(2), .metric-table td:nth-child(3), .metric-table td:nth-child(4), .metric-table td:nth-child(5),
      .metric-table th:nth-child(2), .metric-table th:nth-child(3), .metric-table th:nth-child(4), .metric-table th:nth-child(5) {{text-align:right;}}
      .metric-table td.score-darkgreen {{background:#14532d; color:#fff; font-weight:900; border-radius: 0;}}
      .metric-table td.score-lightgreen {{background:#bbf7d0; color:#14532d; font-weight:800;}}
      .metric-table td.score-yellow {{background:#fef08a; color:#713f12; font-weight:800;}}
      .metric-table td.score-orange {{background:#fed7aa; color:#7c2d12; font-weight:800;}}
      .metric-table td.score-red {{background:#fecaca; color:#7f1d1d; font-weight:800;}}
      .metric-table td.score-na {{background:#f3f4f6; color:#6b7280;}}
      .notes-box {{background:#f8fafc; border:1px solid #e7ebf0; border-radius: 12px; padding: 8px 10px; font-size: 12.5px; line-height:1.4;}}
      .muted {{color:#6b7280;}}
      .small {{font-size: 12px; margin-top: 6px;}}
      @page {{ size: A4 portrait; margin: 8mm; }}
      @media print {{
        body {{background:#fff !important;}}
        body * {{visibility: hidden !important;}}
        #{profile_id}, #{profile_id} * {{visibility: visible !important;}}
        #{profile_id} {{position:absolute; left:0; top:0; width:100%; max-width: 194mm; margin: 0 auto; transform-origin: top center;}}
        .export-actions {{display:none !important;}}
        .profile-wrap {{max-width: none !important; width:100% !important;}}
        .role-card, .info-panel {{break-inside: avoid; page-break-inside: avoid;}}
        a {{color:#111827; text-decoration:none;}}
      }}
      @media (max-width: 900px) {{
        .info-grid {{grid-template-columns: repeat(2, minmax(0, 1fr));}}
        .info-item.wide {{grid-column: span 2;}}
        .cards-grid {{grid-template-columns: 1fr;}}
      }}
    </style>

    <div class='profile-wrap'>
      <div id='{profile_id}'>
        <div class='info-panel'>
          <div class='info-header'>
            <h2>{html.escape(player_name)}</h2>
            <div class='info-chips'>
              <div class='info-chip'><span class='chip-label'>Position</span><span class='chip-value'>{display_position_label(row.get('Position', '–'))}</span></div>
              <div class='info-chip'><span class='chip-label'>Spielzeit</span><span class='chip-value'>{_fmt(row.get('Minuten', np.nan), 0)} Min.</span></div>
              <div class='profile-logo-chip'>{profile_logo_html}</div>
            </div>
          </div>
          <div class='info-grid'>
            <div class='info-item'><div class='info-label'>Team</div><div class='info-value'>{_escape(row.get('Team', '–'))}</div></div>
            <div class='info-item'><div class='info-label'>Liga</div><div class='info-value'>{liga_display}</div></div>
            <div class='info-item' title='{league_factor_tooltip}'><div class='info-label'>Ligenkoeffizient {league_factor_info_icon}</div><div class='info-value'>{league_factor_display}</div></div>
            <div class='info-item'><div class='info-label'>Größe</div><div class='info-value'>{_escape(row.get('Größe', np.nan))}</div></div>
            <div class='info-item'><div class='info-label'>Alter</div><div class='info-value'>{_fmt(row.get('Alter', np.nan), 0)}</div></div>
            <div class='info-item'><div class='info-label'>TM Marktwert</div><div class='info-value'>{_escape(row.get('TM Marktwert', np.nan))}</div></div>
            <div class='info-item'><div class='info-label'>TM Vertrag bis</div><div class='info-value'>{_escape(row.get('TM Vertrag bis', np.nan))}</div></div>
            <div class='info-item'><div class='info-label'>TM Profil</div><div class='info-value'>{url_html}</div></div>
          </div>
        </div>
        <div class='cards-grid'>
          {top_card}
          {second_card}
        </div>
      </div>
    </div>
    '''
    return html_block

# Globale Variable für die GUI. Dadurch kann man im Notfall auch ohne Widgets manuell anzeigen:
# display_player_profile('Spielername')
GUI_STATE = {'state': None}

def display_player_profile(player_name, state=None):
    """Fallback-Funktion: zeigt einen Steckbrief direkt aus einer Code-Zelle an."""
    from IPython.display import display, HTML
    state = state or GUI_STATE.get('state')
    if state is None:
        print('Noch keine Scores berechnet. Bitte zuerst die Pipeline/GUI ausführen.')
        return
    names = state['top_matches']['Spieler'].dropna().astype(str).tolist()
    if player_name not in names:
        query = str(player_name).lower()
        matches = [n for n in names if query in n.lower()]
        if not matches:
            print(f'Kein Spieler gefunden: {player_name}')
            return
        player_name = matches[0]
        print(f'Nutze ersten Treffer: {player_name}')
    display(HTML(player_profile_html(player_name, state)))


def _normalize_search_text(value):
    """Robuste Normalisierung für Suche nach Namen, Positionen und Rollen."""
    if pd.isna(value):
        return ''
    text = str(value).strip().lower()
    replacements = {
        'ä': 'ae', 'ö': 'oe', 'ü': 'ue', 'ß': 'ss',
        'á': 'a', 'à': 'a', 'â': 'a', 'é': 'e', 'è': 'e', 'ê': 'e',
        'í': 'i', 'ì': 'i', 'î': 'i', 'ó': 'o', 'ò': 'o', 'ô': 'o',
        'ú': 'u', 'ù': 'u', 'û': 'u',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def _position_search_aliases(position):
    """Erweitert Positionskürzel um deutsche/englische Suchbegriffe.

    So findet z.B. die Suche nach "IV" oder "Innenverteidiger" auch Spieler,
    deren Position in der Datei nur als CB/LCB/RCB erfasst ist.
    """
    raw = str(position or '')
    p = _normalize_search_text(raw)
    aliases = []

    def add(*words):
        aliases.extend(words)

    # Torwart
    if re.search(r'\b(gk|tw|goalkeeper|torwart)\b', p):
        add('tw', 'torwart', 'goalkeeper', 'gk')

    # Innenverteidiger
    if re.search(r'\b(cb|lcb|rcb|iv)\b', p) or 'innenverteidiger' in p or 'center back' in p or 'centre back' in p:
        add('iv', 'innenverteidiger', 'cb', 'center back', 'centre back', 'zentraler verteidiger', 'mitspielender iv', 'defensiver iv')

    # Außenverteidiger / Wingback
    if re.search(r'\b(lb|rb|lwb|rwb|fb|av)\b', p) or 'aussenverteidiger' in p or 'ausenverteidiger' in p or 'fullback' in p or 'wingback' in p:
        add('av', 'aussenverteidiger', 'außenverteidiger', 'fullback', 'wingback', 'linker verteidiger', 'rechter verteidiger', 'offensiver av', 'defensiver av')

    # Defensive / zentrale Mittelfeldspieler
    if re.search(r'\b(dm|cdm|dmf|6|sechser)\b', p) or 'defensive midfield' in p:
        add('dm', 'cdm', 'sechser', '6er', 'defensive 6', 'spielstarke 6', 'defensives mittelfeld')
    if re.search(r'\b(cm|cmf|8|achter)\b', p) or 'central midfield' in p or 'zentrales mittelfeld' in p:
        add('cm', 'achter', '8er', 'box to box', 'box-to-box', 'progressiver 8er', 'zentrales mittelfeld')

    # Offensives Mittelfeld / Flügel
    if re.search(r'\b(am|cam|amf|10|zehner)\b', p) or 'attacking midfield' in p or 'offensives mittelfeld' in p:
        add('am', 'cam', 'zehner', '10er', 'offensives mittelfeld', 'klassische 10', 'kreativer freigeist')
    if re.search(r'\b(lw|rw|lm|rm|wing|winger|fluegel|fluegelspieler)\b', p) or 'fluegel' in p or 'flügel' in raw.lower():
        add('wing', 'winger', 'fluegel', 'flügel', 'fluegelspieler', 'flügelspieler', 'klassischer fluegel', 'invertierter fluegel')

    # Stürmer
    if re.search(r'\b(st|cf|fw|striker|forward|9)\b', p) or 'stuermer' in p or 'stürmer' in raw.lower():
        add('st', 'cf', 'stuermer', 'stürmer', 'striker', 'forward', 'neuner', 'zielspieler', 'wandspieler', 'falsche 9', 'tiefenlaeufer', 'tiefenläufer')

    return _normalize_search_text(' '.join(dict.fromkeys(aliases)))


def create_colab_gui():
    try:
        import ipywidgets as widgets
        from IPython.display import display, HTML, clear_output
    except Exception as e:
        print('ipywidgets/IPython konnten nicht geladen werden:', e)
        return

    def save_upload(upload_widget, target_path):
        if not upload_widget.value:
            return None
        val = upload_widget.value
        # ipywidgets 7: dict; ipywidgets 8/Colab: tuple/list of dicts
        if isinstance(val, dict):
            item = next(iter(val.values()))
            content = item.get('content')
            name = item.get('metadata', {}).get('name') or item.get('name') or target_path.name
        else:
            item = val[0]
            content = item.get('content')
            name = item.get('name') or target_path.name
        target_path.write_bytes(bytes(content))
        return name

    def resolve_input_paths():
        required_paths = [Path('spieler_data.xlsx'), Path('weltklasse_referenz.xlsx'), Path('zweite_liga_referenz.xlsx')]
        fallback_paths = [Path(ORIGINAL_EXCEL), Path(WORLD_CLASS_REFERENCE_EXCEL), Path(SECOND_LEAGUE_REFERENCE_EXCEL)]
        missing = []
        resolved = []
        for std_path, fb_path in zip(required_paths, fallback_paths):
            if std_path.exists():
                resolved.append(std_path)
            elif fb_path.exists():
                resolved.append(fb_path)
            else:
                missing.append(std_path.name)
        return resolved, missing

    original_upload = widgets.FileUpload(accept='.xlsx', multiple=False, description='Spielerdatei')
    world_upload = widgets.FileUpload(accept='.xlsx', multiple=False, description='Weltklasse')
    second_upload = widgets.FileUpload(accept='.xlsx', multiple=False, description='2. Liga')
    world_class_checkbox = widgets.Checkbox(
        value=False,
        description='Weltklasse-Referenz verwenden',
        indent=False,
        layout=widgets.Layout(width='280px')
    )
    world_class_warning = widgets.HTML(
        "<span style='color:#b45309;font-weight:700;'>Warnung: Experimentell!</span> Standard ist 2. Bundesliga."
    )
    run_button = widgets.Button(description='Scores berechnen', button_style='success', icon='calculator')
    status_out = widgets.Output()
    search_container = widgets.VBox([
        widgets.HTML('<i>Nach der Berechnung erscheint hier die Spielersuche.</i>')
    ])

    def build_search_ui(state):
        top_df = state['top_matches'].copy()
        if top_df.empty or 'Spieler' not in top_df.columns:
            search_container.children = [widgets.HTML('<b>Keine Spieler in den Ergebnissen gefunden.</b>')]
            return

        top_df = top_df.dropna(subset=['Spieler']).copy()
        top_df['Spieler'] = top_df['Spieler'].astype(str)
        if 'Position' not in top_df.columns:
            top_df['Position'] = ''
        top_df = top_df.sort_values('Spieler').drop_duplicates('Spieler')

        player_records = []
        for _, rec in top_df.iterrows():
            name = str(rec.get('Spieler', '')).strip()
            position = str(rec.get('Position', '') if pd.notna(rec.get('Position', np.nan)) else '').strip()
            label = f'{name} · {position}' if position else name
            search_blob = _normalize_search_text(' '.join([
                name,
                position,
                _position_search_aliases(position),
                str(rec.get('Team', '')),
                str(rec.get('Top Rolle', '')),
            ]))
            player_records.append({'name': name, 'position': position, 'label': label, 'search': search_blob})

        if not player_records:
            search_container.children = [widgets.HTML('<b>Keine Spieler in den Ergebnissen gefunden.</b>')]
            return

        filter_text = widgets.Text(
            value='',
            placeholder='Name oder Position eingeben, z.B. IV, Innenverteidiger, AV, Sechser ...',
            description='Suche:',
            layout=widgets.Layout(width='620px')
        )
        player_dropdown = widgets.Dropdown(
            options=[(r['label'], r['name']) for r in player_records[:100]],
            description='Treffer:',
            layout=widgets.Layout(width='620px')
        )
        hint = widgets.HTML('<b>Spieler-Suche</b><br>Tippe einen Namen oder eine Position ein, z.B. <code>IV</code>, <code>Innenverteidiger</code>, <code>AV</code>, <code>Sechser</code> oder <code>Flügel</code>. Der Steckbrief aktualisiert sich automatisch.')
        count_label = widgets.HTML('')
        profile_out = widgets.Output()

        def current_matches():
            q = _normalize_search_text(filter_text.value)
            if q:
                return [r for r in player_records if q in r['search']]
            return player_records

        def show_player(change=None):
            selected = player_dropdown.value
            with profile_out:
                clear_output(wait=True)
                if not selected:
                    display(HTML('<div>Bitte einen Spieler auswählen.</div>'))
                    return
                display(HTML(player_profile_html(selected, state)))

        def refresh_dropdown(change=None):
            matches = current_matches()
            shown = matches[:100]
            if shown:
                old_value = player_dropdown.value
                player_dropdown.options = [(r['label'], r['name']) for r in shown]
                shown_names = [r['name'] for r in shown]
                player_dropdown.value = old_value if old_value in shown_names else shown[0]['name']
                count_label.value = f'<span>{len(matches)} Treffer, zeige maximal 100.</span>'
            else:
                player_dropdown.options = [('Keine Treffer', '')]
                player_dropdown.value = ''
                count_label.value = '<span>Keine Treffer.</span>'
            show_player()

        filter_text.observe(refresh_dropdown, names='value')
        player_dropdown.observe(show_player, names='value')

        refresh_dropdown()
        search_container.children = [
            hint,
            widgets.HBox([filter_text]),
            widgets.HBox([player_dropdown, count_label]),
            profile_out,
        ]
        show_player()

    def on_run_clicked(_):
        with status_out:
            clear_output(wait=True)
            try:
                if original_upload.value:
                    save_upload(original_upload, Path('spieler_data.xlsx'))
                if world_upload.value:
                    save_upload(world_upload, Path('weltklasse_referenz.xlsx'))
                if second_upload.value:
                    save_upload(second_upload, Path('zweite_liga_referenz.xlsx'))

                resolved, missing = resolve_input_paths()
                if missing:
                    display(HTML('<b>Fehlende Dateien:</b> ' + html.escape(', '.join(missing)) + '<br>Bitte lade die fehlenden Dateien hoch oder lege sie im Colab-Arbeitsverzeichnis ab.'))
                    return
                original_path, world_path, second_path = resolved

                state = run_pipeline(
                    reference_mode=('world_class' if world_class_checkbox.value else 'best_2_bundesliga'),
                    original_excel=original_path,
                    world_reference_excel=world_path,
                    second_reference_excel=second_path,
                    export_excel=True,
                )
                GUI_STATE['state'] = state
            except Exception:
                display(HTML('<b>Fehler während Berechnung oder GUI-Aufbau.</b>'))
                raise
        if GUI_STATE.get('state') is not None:
            build_search_ui(GUI_STATE['state'])

    run_button.on_click(on_run_clicked)
    display(HTML('<h2>Benchmark-Rollenfit v16</h2><p>Schritt 1: Lade die beiden Benchmark-Tabellen hoch. Falls die Spielerdatei noch nicht im Colab-Verzeichnis liegt, lade zusätzlich <code>spieler_data.xlsx</code> hoch. Standardmäßig wird gegen <b>2. Bundesliga</b> normiert. Die Weltklasse-Referenz kann per Checkbox aktiviert werden und ist experimentell. <code>best11_rollenprofil_abgleich.xlsx</code> wird nicht mehr benötigt.</p>'))
    display(HTML('<b>Spielerdatei optional, falls noch nicht vorhanden:</b>'))
    display(original_upload)
    display(HTML('<b>Benchmark-Dateien:</b>'))
    display(widgets.HBox([world_upload, second_upload]))
    display(widgets.HBox([world_class_checkbox, world_class_warning]))
    display(run_button)
    display(status_out)
    display(search_container)

# create_colab_gui()  # In ClubProfile 2.0 nicht automatisch starten.


# ClubProfile 2.0: Badge-Grafiken aus dem graphics/-Ordner aktivieren.
# score_medal_badge_html liest dieses globale Dict zur Laufzeit.
BADGE_IMAGE_PATHS = {
    'gold': BADGE_URIS.get('gold') or None,
    'silver': BADGE_URIS.get('silver') or None,
    'bronze': BADGE_URIS.get('bronze') or None,
}


In [ ]:
# =========================
# ClubProfile 2.0 – Spiderplot-Modul ausführen in eigener Namespace
# =========================

SPIDER_NS = {
    "pd": pd,
    "np": np,
    "plt": plt,
    "re": re,
    "html": html,
    "textwrap": textwrap,
    "Path": Path,
    "defaultdict": defaultdict,
    "patches": patches,
    "widgets": widgets,
    "display": display,
    "clear_output": clear_output,
    "IPyHTML": IPyHTML,
    "LOGO_PATH": LOGO_PATH,
}

SPIDER_SETTINGS_SOURCE = '# =========================\n# Einstellungen\n# =========================\n\n# Wenn True, werden nicht explizit definierte Positionen zusätzlich einzeln als Gruppe angeboten.\nINCLUDE_OTHER_POSITIONS = True\n\n# Metriken, die nicht dargestellt werden sollen.\nMETRICS_TO_EXCLUDE = {"Fouls", "Fouls Drawn", "Cards"}\n\n# Jugend-/Zweitteam-Spieler von Nürnberg ausschließen?\nEXCLUDE_NUERNBERG_YOUTH = False\n\nYOUTH_TEAM_PATTERNS = [\n    r"\\bii\\b",         # Nürnberg II\n    r"\\bu[-\\s]?17\\b",  # Nürnberg U17, U-17, U 17\n    r"\\bu[-\\s]?19\\b",\n    r"\\bu[-\\s]?21\\b",\n]\n\n# Deckelung für die visuelle Darstellung.\nCAP_PERCENT = 300\nCAPPED_LABEL_BASE_OFFSET = 14\nCAPPED_LABEL_LEVEL_GAP = 18\n\nOUTPUT_DIR = Path("spider_plots")\nOUTPUT_DIR.mkdir(exist_ok=True)\n\n# Spaltennamen.\nPLAYER_COL = "Spieler"\nPOSITION_COL = "Position"\nMETRIC_COL = "Metric"\nVALUE_COL = "Wert"\nLEAGUE_COL = "Liga"\nTEAM_COL_CANDIDATES = ["Team", "Verein", "Club", "Mannschaft"]\n\n# Positionsgruppen wie im ursprünglichen Notebook.\nPOSITION_GROUPS = {\n    "GK": ["GK"],\n    "FB_LB_RB": ["LB", "RB", "LWB", "RWB", "FB"],\n    "CB_LCB_RCB": ["LCB", "RCB", "CB"],\n    "DM_CM": ["DMF", "LDMF", "RDMF", "LCMF3", "RCMF3", "CMF", "LCMF", "RCMF"],\n    "AM_WING": ["AMF", "LWF", "RWF"],\n    "WING_CF": ["LWF", "RWF", "CF"],\n}\n\n# Logische Reihenfolge der Metriken.\nMETRIC_ORDER = [\n    # Abschluss / Torgefahr\n    "Shots",\n    "Goals/Shot on Target %",\n    "Non-Pen Goals",\n    "npxG",\n    "npxG per Shot",\n    "Touches in Pen Box",\n\n    # Kreativität / Chance Creation\n    "Assists",\n    "Second Assists",\n    "Assists & 2nd/3rd Assists",\n    "Shot Assists",\n    "Expected Assists (xA)",\n    "xA per Shot Assist",\n    "Smart Passes",\n    "Smart Pass %",\n    "Crosses",\n    "Cross Completion %",\n\n    # Passspiel / Ballzirkulation / Progression\n    "Received Passes",\n    "Passes",\n    "Short & Med Pass %",\n    "% of Passes Being Short",\n    "% of Passes Being Lateral",\n    "Long Pass %",\n    "Long Pass Cmp %",\n    "Prog. Passes",\n    "Prog. Carries",\n\n    # Dribbling / Balltransport\n    "Acceleration with Ball",\n    "Dribble Success %",\n\n    # Defensivarbeit\n    "Defensive Actions",\n    "Defensive Duels Won %",\n    "Tackles (pAdj)",\n    "Interceptions (pAdj)",\n    "Tackles & Int (pAdj)",\n    "Shot Blocks",\n    "Aerial Duels Won",\n    "Aerial Win %",\n\n    # Torwart-spezifisch\n    "Save %",\n    "Shots Against",\n    "Goals Conceded",\n    "Prevented Goals",\n    "Goals Prevented %",\n    "Coming Off Line",\n]\n\nMETRIC_ORDER_MAP = {metric: i for i, metric in enumerate(METRIC_ORDER)}\n'
SPIDER_HELPERS_SOURCE = '# =========================\n# Hilfsfunktionen: Daten, Gruppen, relative Werte, Transfermarkt, Layout\n# =========================\n\nFCN_RED = "#8B0000"\nFCN_RED_LIGHT = "#C62828"\nFCN_BLACK = "#1F1F1F"\nFCN_GREY = "#5F6368"\nFCN_BG = "#FAF7F7"\nPANEL_BG = "#FBFBFC"\nPANEL_BORDER = "#D9D9DE"\nNON_FCN_COLORS = ["#F39C12", "#1B9E77", "#4C78A8", "#7F7F7F"]\nFCN_PLAYER_COLORS = [FCN_RED, FCN_BLACK, FCN_RED_LIGHT]\n\n\ndef sanitize_filename(text):\n    text = re.sub(r"[^\\w\\s-]", "", str(text), flags=re.UNICODE)\n    text = re.sub(r"[-\\s]+", "_", text)\n    return text.strip("_")[:120]\n\n\ndef is_nuernberg_text(text):\n    text = str(text).lower()\n    patterns = [\n        "nürnberg",\n        "nuernberg",\n        "nurnberg",\n        "1. fc nürnberg",\n        "1. fc nuernberg",\n        "1. fc nurnberg",\n        "fcn",\n        "1. fcn",\n    ]\n    return any(p in text for p in patterns)\n\n\ndef is_nuernberg_youth_team(text):\n    text = str(text).lower()\n    if not is_nuernberg_text(text):\n        return False\n    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in YOUTH_TEAM_PATTERNS)\n\n\ndef sort_metrics_logically(metrics):\n    return sorted(metrics, key=lambda m: (METRIC_ORDER_MAP.get(m, 10_000), m))\n\n\ndef first_non_empty(values):\n    for value in values:\n        if pd.isna(value):\n            continue\n        if isinstance(value, str) and value.strip() == "":\n            continue\n        return value\n    return np.nan\n\n\ndef format_display_value(value):\n    if pd.isna(value):\n        return "k. A."\n    if isinstance(value, pd.Timestamp):\n        return value.strftime("%d.%m.%Y")\n    text = str(value).strip()\n    if text == "" or text.lower() == "nan":\n        return "k. A."\n    return text\n\n\ndef compact_url(url):\n    url = format_display_value(url)\n    if url == "k. A.":\n        return url\n    url = re.sub(r"^https?://", "", url)\n    return url.replace("www.", "")\n\n\ndef detect_first_existing_column(df_like, candidates):\n    return next((c for c in candidates if c in df_like.columns), None)\n\n\ndef wrap_metric_label(label, width=16):\n    label = str(label)\n    if len(label) <= width:\n        return label\n\n    words = label.split()\n    if len(words) == 1:\n        return textwrap.fill(label, width=width)\n\n    wrapped = textwrap.fill(label, width=width, break_long_words=False, break_on_hyphens=False)\n    return wrapped\n\n\ndef wrap_card_line(text, width=34):\n    text = str(text)\n    return textwrap.fill(text, width=width, break_long_words=False, break_on_hyphens=False)\n\n\ndef load_and_prepare_data(file_path, sheet_name):\n    raw_df = pd.read_excel(file_path, sheet_name=sheet_name)\n\n    required_cols = [PLAYER_COL, POSITION_COL, METRIC_COL, VALUE_COL, LEAGUE_COL]\n    missing_cols = [c for c in required_cols if c not in raw_df.columns]\n    if missing_cols:\n        raise ValueError(f"Diese Spalten fehlen im Sheet: {missing_cols}")\n\n    detected_team_col = next((c for c in TEAM_COL_CANDIDATES if c in raw_df.columns), None)\n\n    keep_cols = required_cols.copy()\n    if detected_team_col is not None:\n        keep_cols.append(detected_team_col)\n\n    clean_df = raw_df[keep_cols].copy()\n    clean_df = clean_df.dropna(subset=[PLAYER_COL, POSITION_COL, METRIC_COL, VALUE_COL])\n    clean_df[PLAYER_COL] = clean_df[PLAYER_COL].astype(str).str.strip()\n    clean_df[POSITION_COL] = clean_df[POSITION_COL].astype(str).str.strip()\n    clean_df[METRIC_COL] = clean_df[METRIC_COL].astype(str).str.strip()\n    clean_df[VALUE_COL] = pd.to_numeric(clean_df[VALUE_COL], errors="coerce")\n    clean_df = clean_df.dropna(subset=[VALUE_COL])\n\n    clean_df = clean_df[~clean_df[METRIC_COL].isin(METRICS_TO_EXCLUDE)].copy()\n\n    if EXCLUDE_NUERNBERG_YOUTH:\n        if detected_team_col is None:\n            print("Warnung: Kein Team-Feld gefunden, Jugend-/Zweitteam-Filter kann nicht angewendet werden.")\n        else:\n            before_players = clean_df[PLAYER_COL].nunique()\n            clean_df = clean_df[~clean_df[detected_team_col].apply(is_nuernberg_youth_team)].copy()\n            after_players = clean_df[PLAYER_COL].nunique()\n            print(f"Jugend-/Zweitteam-Filter aktiv: {before_players - after_players} Spieler entfernt.")\n\n    group_cols = [PLAYER_COL, POSITION_COL, METRIC_COL, LEAGUE_COL]\n    if detected_team_col is not None:\n        group_cols.append(detected_team_col)\n\n    clean_df = clean_df.groupby(group_cols, as_index=False)[VALUE_COL].mean()\n\n    return clean_df, detected_team_col\n\n\ndef load_transfermarkt_data(file_path, sheet_name):\n    xls = pd.ExcelFile(file_path)\n    if sheet_name not in xls.sheet_names:\n        print(f"Hinweis: Transfermarkt-Sheet \'{sheet_name}\' wurde nicht gefunden.")\n        empty = pd.DataFrame(\n            columns=[PLAYER_COL, "tm_team", "tm_market_value", "tm_contract_until", "tm_height", "tm_profile_url"]\n        ).set_index(PLAYER_COL)\n        return empty, {}\n\n    raw_tm_df = pd.read_excel(file_path, sheet_name=sheet_name)\n    if PLAYER_COL not in raw_tm_df.columns:\n        print(f"Hinweis: Im Transfermarkt-Sheet fehlt die Spalte \'{PLAYER_COL}\'.")\n        empty = pd.DataFrame(\n            columns=[PLAYER_COL, "tm_team", "tm_market_value", "tm_contract_until", "tm_height", "tm_profile_url"]\n        ).set_index(PLAYER_COL)\n        return empty, {}\n\n    raw_tm_df = raw_tm_df.copy()\n    raw_tm_df[PLAYER_COL] = raw_tm_df[PLAYER_COL].astype(str).str.strip()\n    raw_tm_df = raw_tm_df[raw_tm_df[PLAYER_COL] != ""]\n\n    column_candidates = {\n        "tm_team": ["TM aktueller Verein", "Team", "Team in Ausgangstabelle"],\n        "tm_market_value": ["TM Marktwert"],\n        "tm_contract_until": ["TM Vertrag bis"],\n        "tm_height": ["Größe", "Groesse", "TM Größe", "TM Groesse"],\n        "tm_profile_url": ["TM Profil-URL", "Profil-URL", "Transfermarkt-Profil-URL"],\n    }\n\n    detected_columns = {\n        key: detect_first_existing_column(raw_tm_df, candidates)\n        for key, candidates in column_candidates.items()\n    }\n\n    rows = []\n    for player_name, player_rows in raw_tm_df.groupby(PLAYER_COL, sort=True):\n        row = {PLAYER_COL: player_name}\n        for target_col, source_col in detected_columns.items():\n            row[target_col] = first_non_empty(player_rows[source_col]) if source_col else np.nan\n        rows.append(row)\n\n    if not rows:\n        empty = pd.DataFrame(\n            columns=[PLAYER_COL, "tm_team", "tm_market_value", "tm_contract_until", "tm_height", "tm_profile_url"]\n        ).set_index(PLAYER_COL)\n        return empty, detected_columns\n\n    tm_df_clean = pd.DataFrame(rows).set_index(PLAYER_COL)\n    return tm_df_clean, detected_columns\n\n\ndef build_plot_groups():\n    groups = {}\n    used_positions = set()\n\n    for group_name, positions in POSITION_GROUPS.items():\n        group_df = df[df[POSITION_COL].isin(positions)].copy()\n        if not group_df.empty:\n            groups[group_name] = group_df\n            used_positions.update(positions)\n\n    if INCLUDE_OTHER_POSITIONS:\n        remaining_positions = sorted(\n            p for p in df[POSITION_COL].dropna().unique()\n            if p not in used_positions\n        )\n        for pos in remaining_positions:\n            group_df = df[df[POSITION_COL] == pos].copy()\n            if not group_df.empty:\n                groups[pos] = group_df\n\n    return groups\n\n\ndef get_group_df_by_name(group_name):\n    if group_name in POSITION_GROUPS:\n        positions = POSITION_GROUPS[group_name]\n        return df[df[POSITION_COL].isin(positions)].copy()\n    return df[df[POSITION_COL] == group_name].copy()\n\n\ndef get_all_available_group_names():\n    return list(plot_groups.keys())\n\n\ndef get_candidate_groups_for_player(player_name):\n    player_df = df[df[PLAYER_COL] == player_name].copy()\n    if player_df.empty:\n        return []\n\n    candidate_groups = []\n    for group_name in get_all_available_group_names():\n        group_df = get_group_df_by_name(group_name)\n        if player_name in set(group_df[PLAYER_COL].unique()):\n            candidate_groups.append(group_name)\n\n    return candidate_groups\n\n\ndef prepare_relative_values(group_df, reference_player):\n    values = group_df.pivot_table(\n        index=PLAYER_COL,\n        columns=METRIC_COL,\n        values=VALUE_COL,\n        aggfunc="mean",\n    )\n    values = values.dropna(axis=1, how="all")\n\n    if reference_player not in values.index:\n        raise ValueError(f"Referenzspieler \'{reference_player}\' ist nicht in dieser Gruppe enthalten.")\n\n    ref_values = values.loc[reference_player]\n    usable_metrics = ref_values[(ref_values.notna()) & (ref_values != 0)].index.tolist()\n    usable_metrics = sort_metrics_logically(usable_metrics)\n\n    values = values[usable_metrics]\n    ref_values = ref_values[usable_metrics]\n    relative_values = values.divide(ref_values, axis=1) * 100\n\n    return relative_values, values, ref_values\n\n\ndef get_nuernberg_players():\n    if team_col is None:\n        print("Warnung: Kein Team-Feld gefunden. Referenzliste fällt auf alle Spieler zurück.")\n        candidate_players = sorted(df[PLAYER_COL].dropna().unique())\n    else:\n        candidate_players = []\n        for player, player_df in df.groupby(PLAYER_COL):\n            teams = player_df[team_col].dropna().astype(str).unique().tolist()\n            if any(is_nuernberg_text(team) for team in teams):\n                candidate_players.append(player)\n        candidate_players = sorted(candidate_players)\n\n    return [p for p in candidate_players if get_candidate_groups_for_player(p)]\n\n\ndef get_player_team_from_main_data(player_name):\n    if team_col is None:\n        return np.nan\n    player_rows = df[df[PLAYER_COL] == player_name]\n    if player_rows.empty:\n        return np.nan\n    team_values = player_rows[team_col].dropna().astype(str).unique().tolist()\n    return team_values[0] if team_values else np.nan\n\n\ndef get_player_league_from_main_data(player_name):\n    player_rows = df[df[PLAYER_COL] == player_name]\n    if player_rows.empty:\n        return np.nan\n    league_values = player_rows[LEAGUE_COL].dropna().astype(str).unique().tolist()\n    return league_values[0] if league_values else np.nan\n\n\ndef build_legend_label(player_name):\n    league = format_display_value(get_player_league_from_main_data(player_name))\n    if league == "k. A.":\n        return player_name\n    return f"{player_name} | {league}"\n\n\ndef is_fcn_player(player_name):\n    candidate_texts = []\n\n    team_from_main = get_player_team_from_main_data(player_name)\n    if pd.notna(team_from_main):\n        candidate_texts.append(team_from_main)\n\n    if \'tm_info_df\' in globals() and not tm_info_df.empty and player_name in tm_info_df.index:\n        tm_team = tm_info_df.loc[player_name, \'tm_team\']\n        if pd.notna(tm_team):\n            candidate_texts.append(tm_team)\n\n    return any(is_nuernberg_text(text) for text in candidate_texts)\n\n\ndef get_transfermarkt_profile(player_name):\n    fallback_team = get_player_team_from_main_data(player_name)\n\n    profile = {\n        \'Team\': format_display_value(fallback_team),\n        \'TM Marktwert\': \'k. A.\',\n        \'TM Vertrag bis\': \'k. A.\',\n        \'Größe\': \'k. A.\',\n        \'Profil-URL\': \'k. A.\',\n    }\n\n    if \'tm_info_df\' not in globals() or tm_info_df.empty:\n        return profile\n\n    if player_name not in tm_info_df.index:\n        return profile\n\n    player_row = tm_info_df.loc[player_name]\n    if isinstance(player_row, pd.DataFrame):\n        player_row = player_row.iloc[0]\n\n    team_value = player_row.get(\'tm_team\', np.nan)\n    if pd.notna(team_value):\n        profile[\'Team\'] = format_display_value(team_value)\n\n    profile[\'TM Marktwert\'] = format_display_value(player_row.get(\'tm_market_value\', np.nan))\n    profile[\'TM Vertrag bis\'] = format_display_value(player_row.get(\'tm_contract_until\', np.nan))\n    profile[\'Größe\'] = format_display_value(player_row.get(\'tm_height\', np.nan))\n    profile[\'Profil-URL\'] = format_display_value(player_row.get(\'tm_profile_url\', np.nan))\n\n    return profile\n\n\ndef build_clickable_links_html(non_fcn_players):\n    if not non_fcn_players:\n        return ""\n\n    blocks = []\n    for player in non_fcn_players:\n        profile = get_transfermarkt_profile(player)\n        url = profile.get(\'Profil-URL\', \'k. A.\')\n        if url == \'k. A.\':\n            blocks.append(\n                f"<li><strong>{html.escape(player)}</strong>: kein Transfermarkt-Link verfügbar</li>"\n            )\n        else:\n            safe_url = html.escape(url, quote=True)\n            safe_name = html.escape(player)\n            blocks.append(\n                f\'<li><strong>{safe_name}</strong>: <a href="{safe_url}" target="_blank" rel="noopener noreferrer">Transfermarkt-Profil öffnen ↗</a></li>\'\n            )\n\n    return f\'\'\'\n    <div style="\n        margin-top:10px;\n        background:{FCN_BG};\n        border:1px solid {PANEL_BORDER};\n        border-left:6px solid {FCN_RED};\n        border-radius:12px;\n        padding:12px 16px;\n        font-family:Arial, Helvetica, sans-serif;\n        width:1180px;\n    ">\n        <div style="font-size:16px;font-weight:700;color:{FCN_RED};margin-bottom:8px;">Klickbare Transfermarkt-Links</div>\n        <ul style="margin:0;padding-left:18px;line-height:1.7;">\n            {\'\'.join(blocks)}\n        </ul>\n    </div>\n    \'\'\'\n\n\ndef style_for_player(player_name, fcn_counter, non_fcn_counter):\n    if is_fcn_player(player_name):\n        color = FCN_PLAYER_COLORS[min(fcn_counter, len(FCN_PLAYER_COLORS) - 1)]\n        fcn_counter += 1\n    else:\n        color = NON_FCN_COLORS[non_fcn_counter % len(NON_FCN_COLORS)]\n        non_fcn_counter += 1\n    return color, fcn_counter, non_fcn_counter\n\n\ndef clip_for_plot(series, cap=CAP_PERCENT):\n    arr = series.to_numpy(dtype=float)\n    return np.where(np.isnan(arr), np.nan, np.minimum(arr, cap))\n\n\ndef round_up_to_step(x, step=25):\n    return int(np.ceil(x / step) * step)\n\n\ndef determine_dynamic_radial_limit(relative_values, cap=CAP_PERCENT, step=25):\n    arr = relative_values.to_numpy(dtype=float)\n    arr = arr[np.isfinite(arr)]\n\n    if arr.size == 0:\n        return 100\n\n    displayed_max = np.min([np.nanmax(arr), cap])\n    if displayed_max <= 0:\n        return 100\n\n    if displayed_max % step == 0:\n        axis_limit = displayed_max + step\n    else:\n        axis_limit = round_up_to_step(displayed_max, step)\n\n    axis_limit = min(axis_limit, cap)\n    axis_limit = max(axis_limit, 100)\n\n    return int(axis_limit)\n\n\ndef build_radial_ticks(axis_limit, step=25):\n    return list(range(step, int(axis_limit) + 1, step))\n\n\ndef format_reference_raw_value(metric, value):\n    """Formatiert den absoluten Rohwert des Referenzspielers für kleine Labels am 100%-Ring."""\n    if pd.isna(value):\n        return ""\n\n    value = float(value)\n    suffix = "%" if "%" in str(metric) else ""\n\n    if suffix:\n        if abs(value) >= 10:\n            text = f"{value:.0f}" if abs(value - round(value)) < 0.05 else f"{value:.1f}"\n        else:\n            text = f"{value:.1f}"\n    else:\n        if abs(value) >= 100:\n            text = f"{value:.0f}"\n        elif abs(value) >= 10:\n            text = f"{value:.1f}"\n        elif abs(value) >= 1:\n            text = f"{value:.2f}".rstrip("0").rstrip(".")\n        else:\n            text = f"{value:.2f}" if abs(value) >= 0.1 else f"{value:.3f}"\n            text = text.rstrip("0").rstrip(".")\n\n    return f"{text}{suffix}"\n\n\ndef place_reference_value_annotations(ax, angles, metrics, ref_values, base_radius=100, axis_limit=100):\n    """Beschriftet die Datenpunkte des Referenzspielers mit dessen absoluten Rohwerten."""\n    if ref_values is None or len(metrics) == 0:\n        return\n\n    max_label_radius = max(axis_limit, base_radius) + 20\n\n    for idx, (angle, metric) in enumerate(zip(angles[:-1], metrics)):\n        if metric not in ref_values.index:\n            continue\n\n        label = format_reference_raw_value(metric, ref_values.loc[metric])\n        if not label:\n            continue\n\n        # Kleine Radial-Staffelung verhindert, dass benachbarte Labels direkt aufeinander liegen.\n        radial_offset = 8 if idx % 2 == 0 else -8\n        label_radius = base_radius + radial_offset\n        label_radius = min(max(label_radius, 18), max_label_radius)\n\n        cos_a = np.cos(angle)\n        if cos_a > 0.35:\n            ha = "left"\n        elif cos_a < -0.35:\n            ha = "right"\n        else:\n            ha = "center"\n\n        ax.annotate(\n            label,\n            xy=(angle, base_radius),\n            xytext=(angle, label_radius),\n            textcoords="data",\n            ha=ha,\n            va="center",\n            fontsize=7.4,\n            fontweight="bold",\n            color=FCN_RED,\n            clip_on=False,\n            bbox=dict(\n                boxstyle="round,pad=0.22",\n                facecolor="white",\n                edgecolor=FCN_RED,\n                linewidth=0.65,\n                alpha=0.88,\n            ),\n            zorder=25,\n        )\n\n\ndef place_capped_annotations(ax, capped_annotations, cap=CAP_PERCENT, axis_limit=None):\n    if not capped_annotations:\n        return\n\n    if axis_limit is None:\n        axis_limit = cap\n\n    grouped = defaultdict(list)\n    for item in capped_annotations:\n        grouped[item["metric_idx"]].append(item)\n\n    angle_jitter = np.deg2rad(2.0)\n\n    for metric_idx, items in grouped.items():\n        items = sorted(items, key=lambda x: x["true_value"])\n\n        for level, item in enumerate(items):\n            base_angle = item["angle"]\n            true_value = item["true_value"]\n            color = item["color"]\n\n            if level == 0:\n                jitter_factor = 0\n            elif level % 2 == 1:\n                jitter_factor = (level + 1) // 2\n            else:\n                jitter_factor = -(level // 2)\n\n            label_angle = base_angle + jitter_factor * angle_jitter\n            label_radius = max(axis_limit, cap) + CAPPED_LABEL_BASE_OFFSET + level * CAPPED_LABEL_LEVEL_GAP\n\n            cos_a = np.cos(label_angle)\n            if cos_a > 0.25:\n                ha = "left"\n            elif cos_a < -0.25:\n                ha = "right"\n            else:\n                ha = "center"\n\n            ax.annotate(\n                f"{true_value:.0f}%",\n                xy=(base_angle, cap),\n                xytext=(label_angle, label_radius),\n                textcoords="data",\n                ha=ha,\n                va="center",\n                fontsize=8,\n                fontweight="bold",\n                color=color,\n                clip_on=False,\n                arrowprops=dict(\n                    arrowstyle="-",\n                    color=color,\n                    lw=0.8,\n                    alpha=0.75,\n                    shrinkA=0,\n                    shrinkB=0,\n                ),\n                zorder=20,\n            )\n\n\ndef draw_player_card(ax, x, y_top, width, height, player_name):\n    profile = get_transfermarkt_profile(player_name)\n\n    card = patches.FancyBboxPatch(\n        (x, y_top - height),\n        width,\n        height,\n        boxstyle="round,pad=0.012,rounding_size=0.02",\n        linewidth=1.0,\n        edgecolor=PANEL_BORDER,\n        facecolor="white",\n        transform=ax.transAxes,\n    )\n    ax.add_patch(card)\n\n    accent = patches.FancyBboxPatch(\n        (x, y_top - 0.035),\n        width,\n        0.02,\n        boxstyle="round,pad=0,rounding_size=0.02",\n        linewidth=0,\n        facecolor=FCN_RED,\n        transform=ax.transAxes,\n    )\n    ax.add_patch(accent)\n\n    ax.text(\n        x + 0.03,\n        y_top - 0.06,\n        player_name,\n        transform=ax.transAxes,\n        ha="left",\n        va="top",\n        fontsize=11.5,\n        fontweight="bold",\n        color=FCN_BLACK,\n    )\n\n    lines = [\n        f"Team: {profile[\'Team\']}",\n        f"TM Marktwert: {profile[\'TM Marktwert\']}",\n        f"TM Vertrag bis: {profile[\'TM Vertrag bis\']}",\n        f"Größe: {profile[\'Größe\']}",\n    ]\n    wrapped_lines = []\n    for line in lines:\n        wrapped_lines.extend(wrap_card_line(line, width=32).split("\\n"))\n\n    ax.text(\n        x + 0.03,\n        y_top - 0.12,\n        "\\n".join(wrapped_lines),\n        transform=ax.transAxes,\n        ha="left",\n        va="top",\n        fontsize=9.4,\n        color=FCN_BLACK,\n        linespacing=1.45,\n    )\n\n\ndef draw_info_panel(info_ax, legend_items, non_fcn_players):\n    info_ax.axis("off")\n    info_ax.set_xlim(0, 1)\n    info_ax.set_ylim(0, 1)\n    info_ax.set_facecolor(PANEL_BG)\n\n    outer = patches.FancyBboxPatch(\n        (0.02, 0.02),\n        0.96,\n        0.96,\n        boxstyle="round,pad=0.012,rounding_size=0.02",\n        linewidth=1.1,\n        edgecolor=PANEL_BORDER,\n        facecolor=PANEL_BG,\n        transform=info_ax.transAxes,\n    )\n    info_ax.add_patch(outer)\n\n    info_ax.text(0.07, 0.965, "Infobereich", transform=info_ax.transAxes,\n                 ha="left", va="top", fontsize=15, fontweight="bold", color=FCN_RED)\n\n    info_ax.text(0.07, 0.91, "Legende", transform=info_ax.transAxes,\n                 ha="left", va="top", fontsize=12.0, fontweight="bold", color=FCN_BLACK)\n\n    y = 0.865\n    for item in legend_items:\n        legend_label = wrap_card_line(item[\'label\'], width=22)\n        line_count = legend_label.count("\\n") + 1\n        info_ax.plot([0.08, 0.18], [y, y], transform=info_ax.transAxes,\n                     color=item[\'color\'], linewidth=2.5, linestyle=item[\'linestyle\'], solid_capstyle=\'round\')\n        info_ax.text(0.21, y, legend_label, transform=info_ax.transAxes,\n                     ha="left", va="center", fontsize=8.8, color=FCN_BLACK, linespacing=1.14)\n        y -= 0.030 * line_count + 0.013\n\n    info_ax.text(\n        0.07,\n        y - 0.008,\n        "Labels an der FCN-Linie =\\nabsolute Referenzwerte",\n        transform=info_ax.transAxes,\n        ha="left",\n        va="top",\n        fontsize=8.2,\n        color=FCN_GREY,\n        linespacing=1.14,\n    )\n\n    steckbrief_heading_y = y - 0.070\n    info_ax.text(0.07, steckbrief_heading_y, "Steckbrief(e)", transform=info_ax.transAxes,\n                 ha="left", va="top", fontsize=12.0, fontweight="bold", color=FCN_BLACK)\n\n    card_start_y = steckbrief_heading_y - 0.055\n    if non_fcn_players:\n        players_to_show = non_fcn_players[:3]\n        n_cards = len(players_to_show)\n        gap = 0.020\n        bottom_padding = 0.035\n        available_height = card_start_y - bottom_padding - (n_cards - 1) * gap\n        card_height = available_height / n_cards\n        card_height = min(card_height, 0.31)\n        current_y = card_start_y\n        for player in players_to_show:\n            draw_player_card(info_ax, x=0.06, y_top=current_y, width=0.88, height=card_height, player_name=player)\n            current_y -= card_height + gap\n    else:\n        note_text = wrap_card_line(\n            "Alle dargestellten Spieler spielen beim FCN – daher sind keine externen Steckbriefe nötig.",\n            width=31,\n        )\n        note_lines = note_text.count("\\n") + 1\n        note_height = min(0.22, max(0.13, 0.055 + note_lines * 0.035))\n        note = patches.FancyBboxPatch(\n            (0.06, card_start_y - note_height), 0.88, note_height,\n            boxstyle="round,pad=0.012,rounding_size=0.02",\n            linewidth=1.0, edgecolor=PANEL_BORDER, facecolor="white", transform=info_ax.transAxes\n        )\n        info_ax.add_patch(note)\n        info_ax.text(\n            0.09, card_start_y - 0.045,\n            note_text,\n            transform=info_ax.transAxes, ha="left", va="top", fontsize=9.2, color=FCN_BLACK,\n            linespacing=1.25,\n        )\n'

SPIDER_PLOT_SOURCE = '# =========================\n# Spiderplot-Funktion für GUI: Referenz + 1 oder 2 Vergleichsspieler\n# =========================\n\ndef make_spider_plot_comparison(reference_player, comparison_players, explicit_group, save_plot=False):\n    comparison_players = [p for p in comparison_players if p is not None and p != ""]\n\n    if not reference_player:\n        raise ValueError("Bitte einen Referenzspieler auswählen.")\n    if len(comparison_players) < 1:\n        raise ValueError("Bitte mindestens einen Vergleichsspieler auswählen.")\n    if reference_player in comparison_players:\n        raise ValueError("Referenzspieler und Vergleichsspieler müssen unterschiedlich sein.")\n    if len(set(comparison_players)) != len(comparison_players):\n        raise ValueError("Vergleichsspieler dürfen nicht doppelt ausgewählt werden.")\n    if explicit_group is None:\n        raise ValueError("Keine Gruppe ausgewählt. Bitte Vergleichsspieler 1 wählen.")\n\n    group_df_full = get_group_df_by_name(explicit_group).copy()\n    group_players = set(group_df_full[PLAYER_COL].unique())\n\n    missing = [p for p in [reference_player] + comparison_players if p not in group_players]\n    if missing:\n        raise ValueError(f"Diese Spieler sind nicht in der Gruppe \'{explicit_group}\': {missing}")\n\n    player_order = [reference_player] + comparison_players\n    group_df = group_df_full[group_df_full[PLAYER_COL].isin(player_order)].copy()\n\n    relative_values, raw_values, ref_values = prepare_relative_values(\n        group_df=group_df,\n        reference_player=reference_player,\n    )\n\n    relative_values = relative_values.reindex(player_order)\n    metrics = relative_values.columns.tolist()\n\n    if len(metrics) < 3:\n        raise ValueError(\n            f"Für den Plot gibt es weniger als 3 nutzbare Metriken in der Gruppe \'{explicit_group}\'."\n        )\n\n    non_fcn_players = [player for player in player_order if not is_fcn_player(player)]\n    legend_items = []\n    capped_annotations = []\n\n    fig = plt.figure(figsize=(10.0, 6.7), constrained_layout=True)\n    gs = fig.add_gridspec(1, 2, width_ratios=[3.25, 1.38], wspace=0.04)\n    ax = fig.add_subplot(gs[0, 0], polar=True)\n    info_ax = fig.add_subplot(gs[0, 1])\n\n    fig.patch.set_facecolor("white")\n    ax.set_facecolor(FCN_BG)\n\n    try:\n        if "LOGO_PATH" in globals() and LOGO_PATH and Path(LOGO_PATH).exists():\n            logo_img = plt.imread(str(LOGO_PATH))\n            logo_inset = ax.inset_axes([-0.155, 0.855, 0.14, 0.14], transform=ax.transAxes)\n            logo_inset.imshow(logo_img)\n            logo_inset.axis("off")\n            logo_inset.set_zorder(30)\n    except Exception:\n        pass\n\n    n_metrics = len(metrics)\n    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()\n    angles += angles[:1]\n\n    fcn_counter = 0\n    non_fcn_counter = 0\n\n    for idx, player in enumerate(player_order):\n        true_vals = relative_values.loc[player]\n        clipped_vals = clip_for_plot(true_vals, cap=CAP_PERCENT)\n        vals_closed = clipped_vals.tolist() + [clipped_vals[0]]\n\n        if idx == 0:\n            linewidth = 2.8\n            linestyle = "-"\n            alpha_fill = 0.08\n        elif idx == 1:\n            linewidth = 2.3\n            linestyle = "--"\n            alpha_fill = 0.05\n        else:\n            linewidth = 2.2\n            linestyle = ":"\n            alpha_fill = 0.04\n\n        color, fcn_counter, non_fcn_counter = style_for_player(player, fcn_counter, non_fcn_counter)\n\n        ax.plot(angles, vals_closed, linewidth=linewidth, linestyle=linestyle, color=color)\n\n        legend_items.append({\n            "player": player,\n            "label": build_legend_label(player),\n            "color": color,\n            "linestyle": linestyle,\n        })\n\n        if not np.isnan(clipped_vals).any():\n            ax.fill(angles, vals_closed, alpha=alpha_fill, color=color)\n\n        for metric_idx, angle in enumerate(angles[:-1]):\n            true_value = true_vals.iloc[metric_idx]\n            if pd.notna(true_value) and true_value > CAP_PERCENT:\n                capped_annotations.append({\n                    "metric_idx": metric_idx,\n                    "metric": metrics[metric_idx],\n                    "angle": angle,\n                    "true_value": true_value,\n                    "player": player,\n                    "color": color,\n                })\n\n    axis_limit = determine_dynamic_radial_limit(relative_values=relative_values, cap=CAP_PERCENT, step=25)\n\n    if capped_annotations:\n        counts_by_metric = defaultdict(int)\n        for item in capped_annotations:\n            counts_by_metric[item["metric_idx"]] += 1\n        max_stack = max(counts_by_metric.values())\n        ylim_top = max(axis_limit, CAP_PERCENT) + CAPPED_LABEL_BASE_OFFSET + (max_stack - 1) * CAPPED_LABEL_LEVEL_GAP + 35\n    else:\n        ylim_top = axis_limit\n\n    ax.set_ylim(0, ylim_top)\n    ax.grid(alpha=0.45)\n\n    yticks = build_radial_ticks(axis_limit, step=25)\n    ax.set_yticks(yticks)\n    ax.set_yticklabels([f"{y}%" for y in yticks], fontsize=8.4, color=FCN_BLACK)\n    ax.set_rlabel_position(142)\n\n    place_reference_value_annotations(\n        ax=ax,\n        angles=angles,\n        metrics=metrics,\n        ref_values=ref_values,\n        base_radius=100,\n        axis_limit=axis_limit,\n    )\n\n    place_capped_annotations(\n        ax=ax,\n        capped_annotations=capped_annotations,\n        cap=CAP_PERCENT,\n        axis_limit=axis_limit,\n    )\n\n    wrapped_metrics = [wrap_metric_label(metric, width=16) for metric in metrics]\n    ax.set_xticks(angles[:-1])\n    ax.set_xticklabels(wrapped_metrics, fontsize=8.6, color=FCN_BLACK)\n    ax.tick_params(axis="x", pad=7)\n\n    comparison_text = " vs. ".join(player_order)\n    title = (\n        f"{comparison_text}\\n"\n        "Werte jeweils pro90, relativ bezogen auf Referenzspieler (= 100%)"\n    )\n    ax.set_title(title, fontsize=11.2, pad=9, color=FCN_BLACK, fontweight="normal")\n\n    draw_info_panel(info_ax, legend_items=legend_items, non_fcn_players=non_fcn_players)\n\n    if save_plot:\n        filename = sanitize_filename(f"gui_tm_fcn_{explicit_group}_{\'_vs_\'.join(player_order)}")\n        png_path = OUTPUT_DIR / f"{filename}.png"\n        pdf_path = OUTPUT_DIR / f"{filename}.pdf"\n        fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())\n        fig.savefig(pdf_path, bbox_inches="tight", facecolor=fig.get_facecolor())\n        print(f"Gespeichert: {png_path}")\n        print(f"Gespeichert: {pdf_path}")\n\n    return {\n        "group": explicit_group,\n        "figure": fig,\n        "non_fcn_players": non_fcn_players,\n    }\n'

exec(SPIDER_SETTINGS_SOURCE, SPIDER_NS)
exec(SPIDER_HELPERS_SOURCE, SPIDER_NS)
exec(SPIDER_PLOT_SOURCE, SPIDER_NS)

# Die Spiderplot-Daten liegen in derselben Spielerdatei wie die Rollenprofile.
SPIDER_NS["FILE_PATH"] = ORIGINAL_EXCEL
SPIDER_NS["SHEET_NAME"] = "Werte_lang"
SPIDER_NS["TM_SHEET_NAME"] = "Transfermarkt"
SPIDER_NS["OUTPUT_DIR"] = Path("spider_plots")
SPIDER_NS["OUTPUT_DIR"].mkdir(exist_ok=True)

# Daten laden. Falls das Sheet fehlt, wird später im Spiderplot-Tab ein Hinweis angezeigt.
try:
    SPIDER_NS["df"], SPIDER_NS["team_col"] = SPIDER_NS["load_and_prepare_data"](ORIGINAL_EXCEL, "Werte_lang")
    SPIDER_NS["tm_info_df"], SPIDER_NS["tm_detected_cols"] = SPIDER_NS["load_transfermarkt_data"](ORIGINAL_EXCEL, "Transfermarkt")
    SPIDER_NS["plot_groups"] = SPIDER_NS["build_plot_groups"]()
    SPIDER_NS["nuernberg_players"] = SPIDER_NS["get_nuernberg_players"]()
    SPIDER_READY = True
    SPIDER_LOAD_ERROR = None
except Exception as exc:
    SPIDER_READY = False
    SPIDER_LOAD_ERROR = exc

In [ ]:
# =========================
# ClubProfile 2.0 – finaler Spiderplot-Layout-Override
# =========================

SPIDER_FINAL_DESIGN_PATCH = 'def draw_player_card(ax, x, y_top, width, height, player_name):\n    profile = get_transfermarkt_profile(player_name)\n\n    card = patches.FancyBboxPatch(\n        (x, y_top - height),\n        width,\n        height,\n        boxstyle="round,pad=0.012,rounding_size=0.02",\n        linewidth=1.0,\n        edgecolor=PANEL_BORDER,\n        facecolor="white",\n        transform=ax.transAxes,\n    )\n    ax.add_patch(card)\n\n    # Kopfbereich kompakter: Name direkt nahe an die obere Kartenkante.\n    name_y = y_top - 0.026\n    ax.text(\n        x + 0.03,\n        name_y,\n        player_name,\n        transform=ax.transAxes,\n        ha="left",\n        va="top",\n        fontsize=11.2,\n        fontweight="bold",\n        color=FCN_BLACK,\n    )\n\n    # Dünner roter Trennstrich direkt unter dem Namen.\n    line_y = name_y - 0.026\n    ax.plot(\n        [x + 0.03, x + width - 0.03],\n        [line_y, line_y],\n        transform=ax.transAxes,\n        color=FCN_RED,\n        linewidth=1.2,\n        solid_capstyle="round",\n    )\n\n    lines = [\n        f"Team: {profile[\'Team\']}",\n        f"TM Marktwert: {profile[\'TM Marktwert\']}",\n        f"TM Vertrag bis: {profile[\'TM Vertrag bis\']}",\n        f"Größe: {profile[\'Größe\']}",\n    ]\n    wrapped_lines = []\n    for line in lines:\n        wrapped_lines.extend(wrap_card_line(line, width=32).split("\\n"))\n\n    ax.text(\n        x + 0.03,\n        line_y - 0.020,\n        "\\n".join(wrapped_lines),\n        transform=ax.transAxes,\n        ha="left",\n        va="top",\n        fontsize=9.1,\n        color=FCN_BLACK,\n        linespacing=1.30,\n    )\n\n\ndef draw_info_panel(info_ax, legend_items, non_fcn_players):\n    info_ax.axis("off")\n    info_ax.set_xlim(0, 1)\n    info_ax.set_ylim(0, 1)\n    info_ax.set_facecolor(PANEL_BG)\n\n    outer = patches.FancyBboxPatch(\n        (0.02, 0.02),\n        0.96,\n        0.96,\n        boxstyle="round,pad=0.012,rounding_size=0.02",\n        linewidth=1.1,\n        edgecolor=PANEL_BORDER,\n        facecolor=PANEL_BG,\n        transform=info_ax.transAxes,\n    )\n    info_ax.add_patch(outer)\n\n    info_ax.text(\n        0.07, 0.95, "Infobereich",\n        transform=info_ax.transAxes,\n        ha="left", va="top",\n        fontsize=15.5, fontweight="bold", color=FCN_RED,\n    )\n\n    info_ax.text(\n        0.07, 0.895, "Legende",\n        transform=info_ax.transAxes,\n        ha="left", va="top",\n        fontsize=12.0, fontweight="bold", color=FCN_BLACK,\n    )\n\n    # Genug Abstand nach der Überschrift, aber kompakte Abstände zwischen den Einträgen.\n    y = 0.835\n    for item in legend_items:\n        legend_label = wrap_card_line(item["label"], width=22)\n        line_count = legend_label.count("\\n") + 1\n        marker_y = y - 0.012\n        info_ax.plot(\n            [0.08, 0.18], [marker_y, marker_y],\n            transform=info_ax.transAxes,\n            color=item["color"], linewidth=2.6,\n            linestyle=item["linestyle"], solid_capstyle="round",\n        )\n        info_ax.text(\n            0.21, y, legend_label,\n            transform=info_ax.transAxes,\n            ha="left", va="top",\n            fontsize=8.9, color=FCN_BLACK,\n            linespacing=1.16, wrap=True,\n        )\n        y -= 0.030 * line_count + 0.018\n\n    # Hinweis steht im Titel; dadurch bleibt im Infobereich mehr Platz für Steckbriefe.\n    steckbrief_heading_y = y - 0.030\n    info_ax.text(\n        0.07, steckbrief_heading_y, "Steckbrief(e)",\n        transform=info_ax.transAxes,\n        ha="left", va="top",\n        fontsize=12.0, fontweight="bold", color=FCN_BLACK,\n    )\n\n    card_start_y = steckbrief_heading_y - 0.055\n    if non_fcn_players:\n        players_to_show = non_fcn_players[:3]\n        n_cards = len(players_to_show)\n        gap = 0.026\n        bottom_padding = 0.04\n        available_height = card_start_y - bottom_padding - (n_cards - 1) * gap\n        card_height = max(0.15, available_height / n_cards)\n        card_height = min(card_height, 0.285)\n        current_y = card_start_y\n        for player in players_to_show:\n            draw_player_card(\n                info_ax,\n                x=0.06,\n                y_top=current_y,\n                width=0.88,\n                height=card_height,\n                player_name=player,\n            )\n            current_y -= card_height + gap\n    else:\n        note_text = wrap_card_line(\n            "Alle dargestellten Spieler spielen beim FCN – daher sind keine externen Steckbriefe nötig.",\n            width=31,\n        )\n        note = patches.FancyBboxPatch(\n            (0.06, card_start_y - 0.17), 0.88, 0.135,\n            boxstyle="round,pad=0.012,rounding_size=0.02",\n            linewidth=1.0,\n            edgecolor=PANEL_BORDER,\n            facecolor="white",\n            transform=info_ax.transAxes,\n        )\n        info_ax.add_patch(note)\n        info_ax.text(\n            0.09, card_start_y - 0.055,\n            note_text,\n            transform=info_ax.transAxes,\n            ha="left", va="top",\n            fontsize=9.2, color=FCN_BLACK,\n            linespacing=1.25, wrap=True,\n        )\n\n\ndef make_spider_plot_comparison(reference_player, comparison_players, explicit_group, save_plot=False):\n    comparison_players = [p for p in comparison_players if p is not None and p != ""]\n\n    if not reference_player:\n        raise ValueError("Bitte einen Referenzspieler auswählen.")\n    if len(comparison_players) < 1:\n        raise ValueError("Bitte mindestens einen Vergleichsspieler auswählen.")\n    if reference_player in comparison_players:\n        raise ValueError("Referenzspieler und Vergleichsspieler müssen unterschiedlich sein.")\n    if len(set(comparison_players)) != len(comparison_players):\n        raise ValueError("Vergleichsspieler dürfen nicht doppelt ausgewählt werden.")\n    if explicit_group is None:\n        raise ValueError("Keine Gruppe ausgewählt. Bitte Vergleichsspieler 1 wählen.")\n\n    group_df_full = get_group_df_by_name(explicit_group).copy()\n    group_players = set(group_df_full[PLAYER_COL].unique())\n\n    missing = [p for p in [reference_player] + comparison_players if p not in group_players]\n    if missing:\n        raise ValueError(f"Diese Spieler sind nicht in der Gruppe \'{explicit_group}\': {missing}")\n\n    player_order = [reference_player] + comparison_players\n    group_df = group_df_full[group_df_full[PLAYER_COL].isin(player_order)].copy()\n\n    relative_values, raw_values, ref_values = prepare_relative_values(\n        group_df=group_df,\n        reference_player=reference_player,\n    )\n\n    relative_values = relative_values.reindex(player_order)\n    metrics = relative_values.columns.tolist()\n\n    if len(metrics) < 3:\n        raise ValueError(\n            f"Für den Plot gibt es weniger als 3 nutzbare Metriken in der Gruppe \'{explicit_group}\'."\n        )\n\n    non_fcn_players = [player for player in player_order if not is_fcn_player(player)]\n    legend_items = []\n    capped_annotations = []\n\n    # Feste Figure-Koordinaten: Plot und Infobereich teilen dieselbe volle Höhe.\n    fig = plt.figure(figsize=(10.35, 6.55), constrained_layout=False)\n\n    plot_left = 0.035\n    plot_bottom = 0.135\n    plot_width = 0.61\n    plot_height = 0.825\n\n    info_left = 0.705\n    info_bottom = plot_bottom\n    info_width = 0.255\n    info_height = plot_height\n\n    ax = fig.add_axes([plot_left, plot_bottom, plot_width, plot_height], polar=True)\n    info_ax = fig.add_axes([info_left, info_bottom, info_width, info_height])\n\n    # Logo oben links in den freien Whitespace des Spiderplots setzen.\n    try:\n        if "LOGO_PATH" in globals() and LOGO_PATH and Path(LOGO_PATH).exists():\n            logo_ax = fig.add_axes([0.005, 0.895, 0.098, 0.098], zorder=30)\n            logo_ax.axis("off")\n            logo_img = plt.imread(str(LOGO_PATH))\n            logo_ax.imshow(logo_img)\n    except Exception:\n        pass\n\n    fig.patch.set_facecolor("white")\n    ax.set_facecolor(FCN_BG)\n\n    n_metrics = len(metrics)\n    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()\n    angles += angles[:1]\n\n    fcn_counter = 0\n    non_fcn_counter = 0\n\n    for idx, player in enumerate(player_order):\n        true_vals = relative_values.loc[player]\n        clipped_vals = clip_for_plot(true_vals, cap=CAP_PERCENT)\n        vals_closed = clipped_vals.tolist() + [clipped_vals[0]]\n\n        if idx == 0:\n            linewidth = 2.8\n            linestyle = "-"\n            alpha_fill = 0.08\n        elif idx == 1:\n            linewidth = 2.3\n            linestyle = "--"\n            alpha_fill = 0.05\n        else:\n            linewidth = 2.2\n            linestyle = ":"\n            alpha_fill = 0.04\n\n        color, fcn_counter, non_fcn_counter = style_for_player(player, fcn_counter, non_fcn_counter)\n        ax.plot(angles, vals_closed, linewidth=linewidth, linestyle=linestyle, color=color)\n\n        legend_items.append({\n            "player": player,\n            "label": build_legend_label(player),\n            "color": color,\n            "linestyle": linestyle,\n        })\n\n        if not np.isnan(clipped_vals).any():\n            ax.fill(angles, vals_closed, alpha=alpha_fill, color=color)\n\n        for metric_idx, angle in enumerate(angles[:-1]):\n            true_value = true_vals.iloc[metric_idx]\n            if pd.notna(true_value) and true_value > CAP_PERCENT:\n                capped_annotations.append({\n                    "metric_idx": metric_idx,\n                    "metric": metrics[metric_idx],\n                    "angle": angle,\n                    "true_value": true_value,\n                    "player": player,\n                    "color": color,\n                })\n\n    axis_limit = determine_dynamic_radial_limit(relative_values=relative_values, cap=CAP_PERCENT, step=25)\n\n    if capped_annotations:\n        counts_by_metric = defaultdict(int)\n        for item in capped_annotations:\n            counts_by_metric[item["metric_idx"]] += 1\n        max_stack = max(counts_by_metric.values())\n        ylim_top = max(axis_limit, CAP_PERCENT) + CAPPED_LABEL_BASE_OFFSET + (max_stack - 1) * CAPPED_LABEL_LEVEL_GAP + 35\n    else:\n        ylim_top = axis_limit\n\n    ax.set_ylim(0, ylim_top)\n    ax.grid(alpha=0.45)\n\n    yticks = build_radial_ticks(axis_limit, step=25)\n    ax.set_yticks(yticks)\n    ax.set_yticklabels([f"{y}%" for y in yticks], fontsize=8.8, color=FCN_BLACK)\n    ax.set_rlabel_position(142)\n\n    place_reference_value_annotations(\n        ax=ax,\n        angles=angles,\n        metrics=metrics,\n        ref_values=ref_values,\n        base_radius=100,\n        axis_limit=axis_limit,\n    )\n\n    place_capped_annotations(\n        ax=ax,\n        capped_annotations=capped_annotations,\n        cap=CAP_PERCENT,\n        axis_limit=axis_limit,\n    )\n\n    wrapped_metrics = [wrap_metric_label(metric, width=16) for metric in metrics]\n    ax.set_xticks(angles[:-1])\n    ax.set_xticklabels(wrapped_metrics, fontsize=8.9, color=FCN_BLACK)\n    ax.tick_params(axis="x", pad=8)\n\n    comparison_text = " vs. ".join(player_order)\n    bottom_title = (\n        f"{comparison_text}\\n"\n        "Werte pro90, relativ zum Referenzspieler (= 100%); Labels an der FCN-Linie = absolute Referenzwerte"\n    )\n    ax.text(\n        0.5, -0.105,\n        bottom_title,\n        transform=ax.transAxes,\n        ha="center", va="top",\n        fontsize=8.7,\n        color=FCN_BLACK,\n        linespacing=1.18,\n        clip_on=False,\n    )\n\n    draw_info_panel(info_ax, legend_items=legend_items, non_fcn_players=non_fcn_players)\n\n    if save_plot:\n        filename = sanitize_filename(f"gui_tm_fcn_{explicit_group}_{\'_vs_\'.join(player_order)}")\n        png_path = OUTPUT_DIR / f"{filename}.png"\n        pdf_path = OUTPUT_DIR / f"{filename}.pdf"\n        fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())\n        fig.savefig(pdf_path, bbox_inches="tight", facecolor=fig.get_facecolor())\n        print(f"Gespeichert: {png_path}")\n        print(f"Gespeichert: {pdf_path}")\n\n    return {\n        "group": explicit_group,\n        "figure": fig,\n        "non_fcn_players": non_fcn_players,\n    }\n'

exec(SPIDER_FINAL_DESIGN_PATCH, SPIDER_NS)


In [ ]:
# =========================
# ClubProfile 2.0 – Datenzustand und Caching
# =========================

APP_STATES = {}
REFERENCE_MODE_OPTIONS = [
    ("2. Bundesliga-Referenz", "best_2_bundesliga"),
    ("Weltklasse-Referenz (experimentell)", "world_class"),
]


def get_or_compute_state(reference_mode="best_2_bundesliga"):
    """Berechnet die Rollenfit-Pipeline einmal pro Referenzmodus und cached das Ergebnis."""
    if reference_mode not in APP_STATES:
        APP_STATES[reference_mode] = run_pipeline(
            reference_mode=reference_mode,
            original_excel=ORIGINAL_EXCEL,
            world_reference_excel=WORLD_CLASS_REFERENCE_EXCEL,
            second_reference_excel=SECOND_LEAGUE_REFERENCE_EXCEL,
            export_excel=False,
        )
    return APP_STATES[reference_mode]


def build_player_records(state):
    top_df = state["top_matches"].copy()
    if top_df.empty or "Spieler" not in top_df.columns:
        return []

    top_df = top_df.dropna(subset=["Spieler"]).copy()
    top_df["Spieler"] = top_df["Spieler"].astype(str)
    if "Position" not in top_df.columns:
        top_df["Position"] = ""
    top_df = top_df.sort_values("Spieler").drop_duplicates("Spieler")

    records = []
    for _, rec in top_df.iterrows():
        name = str(rec.get("Spieler", "")).strip()
        position = str(rec.get("Position", "") if pd.notna(rec.get("Position", np.nan)) else "").strip()
        team = str(rec.get("Team", "") if pd.notna(rec.get("Team", np.nan)) else "").strip()
        role = str(rec.get("Top Rolle", "") if pd.notna(rec.get("Top Rolle", np.nan)) else "").strip()
        label = f"{name} · {position} · {team}" if team else (f"{name} · {position}" if position else name)
        search_blob = _normalize_search_text(" ".join([
            name,
            position,
            _position_search_aliases(position),
            team,
            role,
        ]))
        name_team_search = _normalize_search_text(" ".join([name, team]))
        records.append({
            "name": name,
            "position": position,
            "team": team,
            "role": role,
            "label": label,
            "search": search_blob,
            "name_team_search": name_team_search,
        })
    return records


In [ ]:
# =========================
# ClubProfile 2.0 – Teambuilder-Modul
# =========================

TEAMBUILDER_PLAYER_SCOPES = [
    ("FCN Spieler", "fcn"),
    ("Alle Spieler", "all"),
]
TEAMBUILDER_OPTIMIZATION_MODES = [
    ("Score-Optimierung", "score"),
    ("Fit-Optimierung", "fit"),
]

def cp_teambuilder_mode_from_controls(scope="fcn", optimization="score"):
    scope = scope if scope in {"fcn", "all"} else "fcn"
    optimization = optimization if optimization in {"score", "fit"} else "score"
    return f"{optimization}_{scope}"

def cp_teambuilder_mode_label(scope="fcn", optimization="score"):
    scope_label = dict(TEAMBUILDER_PLAYER_SCOPES).get(scope, "FCN Spieler")
    opt_label = dict(TEAMBUILDER_OPTIMIZATION_MODES).get(optimization, "Score-Optimierung")
    return f"{scope_label} · {opt_label}"

def _cp_tb_truthy(value, default=True):
    """Robustes Parsen des Aktiv-Felds aus der Excel-Vorlage."""
    if pd.isna(value):
        return default
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"false", "falsch", "0", "nein", "no", "n", "inaktiv"}:
        return False
    if text in {"true", "wahr", "1", "ja", "yes", "y", "aktiv"}:
        return True
    return default


def _cp_tb_required_columns(df, required, sheet_name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Im Sheet '{sheet_name}' fehlen folgende Pflichtspalten: {', '.join(missing)}"
        )


def _cp_tb_clean_text(value, default=""):
    if pd.isna(value):
        return default
    return str(value).strip()


def _cp_tb_percent(value, default=50.0):
    try:
        number = float(value)
    except Exception:
        number = default
    if not np.isfinite(number):
        number = default
    return max(0.0, min(100.0, number))


def cp_load_teambuilder_templates(excel_path=None):
    """Liest die Teambuilder-Formationen aus einer externen Excel-Datei.

    Erwartete Sheets:
    - Vorlagen: Vorlage, Formation, Kurzlabel, Beschreibung, Sortierung, Aktiv
    - Slots: Vorlage, Slot-Reihenfolge, Slot-ID, Positionslabel, Rolle, X, Y

    X/Y sind Prozentkoordinaten für die Pitch-Darstellung:
    X 0=links, 100=rechts; Y 0=vorne, 100=hinten.
    """
    path = Path(excel_path or TEAMBUILDER_TEMPLATES_EXCEL)
    if not path.exists():
        raise FileNotFoundError(
            f"Teambuilder-Vorlagen nicht gefunden: {path}. "
            "Lege die Datei 'teambuilder_formationen.xlsx' neben das Notebook bzw. in den Space."
        )

    templates_df = pd.read_excel(path, sheet_name="Vorlagen")
    slots_df = pd.read_excel(path, sheet_name="Slots")
    templates_df.columns = [str(c).strip() for c in templates_df.columns]
    slots_df.columns = [str(c).strip() for c in slots_df.columns]

    _cp_tb_required_columns(
        templates_df,
        ["Vorlage", "Formation", "Kurzlabel", "Beschreibung"],
        "Vorlagen",
    )
    _cp_tb_required_columns(
        slots_df,
        ["Vorlage", "Slot-Reihenfolge", "Slot-ID", "Positionslabel", "Rolle", "X", "Y"],
        "Slots",
    )

    if "Sortierung" not in templates_df.columns:
        templates_df["Sortierung"] = np.arange(1, len(templates_df) + 1)
    if "Aktiv" not in templates_df.columns:
        templates_df["Aktiv"] = True

    templates_df = templates_df.copy()
    templates_df["__active"] = templates_df["Aktiv"].apply(_cp_tb_truthy)
    templates_df = templates_df[templates_df["__active"]].copy()
    templates_df["__order"] = pd.to_numeric(templates_df["Sortierung"], errors="coerce")
    templates_df["__row_order"] = np.arange(len(templates_df))
    templates_df = templates_df.sort_values(["__order", "__row_order"], na_position="last")

    slots_df = slots_df.copy()
    slots_df["__slot_order"] = pd.to_numeric(slots_df["Slot-Reihenfolge"], errors="coerce")
    slots_df["__row_order"] = np.arange(len(slots_df))
    slots_df = slots_df.sort_values(["Vorlage", "__slot_order", "__row_order"], na_position="last")

    result = {}
    for _, row in templates_df.iterrows():
        template_name = _cp_tb_clean_text(row.get("Vorlage"))
        if not template_name:
            continue

        slot_subset = slots_df[
            slots_df["Vorlage"].astype(str).str.strip() == template_name
        ].copy()

        slots = []
        for _, slot in slot_subset.iterrows():
            slot_id = _cp_tb_clean_text(slot.get("Slot-ID"))
            if not slot_id:
                slot_id = f"slot_{len(slots) + 1:02d}"
            slots.append({
                "id": slot_id,
                "label": _cp_tb_clean_text(slot.get("Positionslabel"), slot_id),
                "role": _cp_tb_clean_text(slot.get("Rolle")),
                "x": _cp_tb_percent(slot.get("X"), 50.0),
                "y": _cp_tb_percent(slot.get("Y"), 50.0),
            })

        if not slots:
            continue

        result[template_name] = {
            "formation": _cp_tb_clean_text(row.get("Formation")),
            "short_label": _cp_tb_clean_text(row.get("Kurzlabel")),
            "description": _cp_tb_clean_text(row.get("Beschreibung")),
            "slots": slots,
        }

    if not result:
        raise ValueError(
            "Die Datei teambuilder_formationen.xlsx enthält keine aktiven Vorlagen mit Slots."
        )

    return result


try:
    TEAMBUILDER_TEMPLATES = cp_load_teambuilder_templates()
    TEAMBUILDER_TEMPLATE_LOAD_ERROR = None
except Exception as exc:
    TEAMBUILDER_TEMPLATES = {}
    TEAMBUILDER_TEMPLATE_LOAD_ERROR = exc


def cp_teambuilder_placeholder_html():
    return """
    <div class='clubprofile-spider-placeholder'>
      <div style='font-size:22px;font-weight:900;color:#8B0000;margin-bottom:8px;'>Teambuilder</div>
      <div style='font-size:14px;line-height:1.55;color:#374151;max-width:780px;'>
        Lade zuerst die Daten. Danach kannst du eine Formation wählen, die Rollen anpassen und die Elf automatisch
        über <b>Fit</b> oder <b>Score</b> füllen lassen – wahlweise aus allen Spielern oder nur aus FCN-Spielern.
      </div>
    </div>
    """


def _teambuilder_norm_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    replacements = {
        "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss",
        "á": "a", "à": "a", "â": "a", "é": "e", "è": "e", "ê": "e",
        "í": "i", "ì": "i", "î": "i", "ó": "o", "ò": "o", "ô": "o",
        "ú": "u", "ù": "u", "û": "u",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def cp_teambuilder_is_fcn_row(row):
    club_text = " ".join(str(row.get(col, "")) for col in ["Team", "TM aktueller Verein"] if col in row.index)
    norm = _teambuilder_norm_text(club_text)
    return any(token in norm for token in ["nuernberg", "nurnberg", "1 fc nuernberg", "1 fcn"])


def cp_teambuilder_role_options(state=None):
    configured = list(SIMPLIFIED_ROLE_CONFIG.keys()) if "SIMPLIFIED_ROLE_CONFIG" in globals() else []
    if state is not None and isinstance(state, dict) and "all_roles" in state:
        present = state["all_roles"].get("Rolle", pd.Series(dtype="object")).dropna().astype(str).unique().tolist()
        # Erst die definierte Rollenreihenfolge, danach eventuell unbekannte Rollen aus der Datenbasis.
        ordered = [r for r in configured if r in set(present)] + [r for r in present if r not in set(configured)]
        return ordered or configured
    return configured



def cp_teambuilder_mode_meta(mode):
    meta = {
        "fit_all": {"rank_col": "Role Fit", "secondary_col": "Final Role Score", "fcn_only": False},
        "score_all": {"rank_col": "Final Role Score", "secondary_col": "Role Fit", "fcn_only": False},
        "fit_fcn": {"rank_col": "Role Fit", "secondary_col": "Final Role Score", "fcn_only": True},
        "score_fcn": {"rank_col": "Final Role Score", "secondary_col": "Role Fit", "fcn_only": True},
    }
    return meta.get(mode, meta["score_fcn"])


def _cp_tb_float(value, default=np.nan):
    try:
        number = float(value)
    except Exception:
        return default
    return number if np.isfinite(number) else default


def _cp_tb_candidate_base_df(state, mode):
    all_roles = state.get("all_roles", pd.DataFrame()).copy() if isinstance(state, dict) else pd.DataFrame()
    if all_roles.empty or "Rolle" not in all_roles.columns or "Spieler" not in all_roles.columns:
        return pd.DataFrame(), cp_teambuilder_mode_meta(mode)

    meta = cp_teambuilder_mode_meta(mode)
    rank_col = meta["rank_col"]
    secondary_col = meta["secondary_col"]
    if rank_col not in all_roles.columns:
        return pd.DataFrame(), meta

    df = all_roles.copy()
    if meta["fcn_only"]:
        df = df.loc[df.apply(cp_teambuilder_is_fcn_row, axis=1)].copy()
    if df.empty:
        return pd.DataFrame(), meta

    df["_tb_player"] = df["Spieler"].astype(str).str.strip()
    df["_tb_role"] = df["Rolle"].astype(str).str.strip()
    df["_tb_primary"] = pd.to_numeric(df[rank_col], errors="coerce")
    df["_tb_secondary"] = pd.to_numeric(df[secondary_col], errors="coerce") if secondary_col in df.columns else np.nan
    df = df.dropna(subset=["_tb_primary"])
    df = df[(df["_tb_player"] != "") & (df["_tb_role"] != "")].copy()
    return df, meta


def cp_teambuilder_player_role_ranks(state, mode):
    """Berechnet je Spieler die Top-3-Rollen im aktiven Bewertungsmodus."""
    df, meta = _cp_tb_candidate_base_df(state, mode)
    if df.empty:
        return {}
    ranks = {}
    df = df.sort_values(["_tb_player", "_tb_primary", "_tb_secondary", "_tb_role"], ascending=[True, False, False, True], na_position="last")
    for player, group in df.groupby("_tb_player", sort=False):
        seen_roles = []
        for _, row in group.iterrows():
            role = row.get("_tb_role", "")
            if not role or role in seen_roles:
                continue
            seen_roles.append(role)
            ranks[(player, role)] = len(seen_roles)
            if len(seen_roles) >= 3:
                break
    return ranks


def cp_teambuilder_ranked_candidates(state, role, mode, used_players=None, limit=8, role_ranks=None):
    """Sortierte Kandidaten für eine Rolle; Top-3-Rollen des Spielers werden bevorzugt."""
    used_players = set(used_players or set())
    df, meta = _cp_tb_candidate_base_df(state, mode)
    if df.empty:
        return []

    role = str(role).strip()
    df = df.loc[df["_tb_role"].eq(role)].copy()
    if df.empty:
        return []

    if role_ranks is None:
        role_ranks = cp_teambuilder_player_role_ranks(state, mode)
    df["_tb_top_rank"] = df.apply(lambda r: role_ranks.get((r.get("_tb_player"), role), 99), axis=1)
    df["_tb_top3"] = df["_tb_top_rank"].le(3)
    df["_tb_rank_bonus"] = df["_tb_top_rank"].map({1: 3, 2: 2, 3: 1}).fillna(0)
    df = df.sort_values(
        ["_tb_top3", "_tb_primary", "_tb_secondary", "_tb_rank_bonus", "_tb_player"],
        ascending=[False, False, False, False, True],
        na_position="last",
    )

    candidates = []
    seen = set()
    for _, row in df.iterrows():
        name = str(row.get("Spieler", "")).strip()
        if not name or name in used_players or name in seen:
            continue
        seen.add(name)
        data = row.drop(labels=[c for c in row.index if str(c).startswith("_tb_")], errors="ignore").to_dict()
        data["_tb_top_rank"] = int(row.get("_tb_top_rank", 99)) if pd.notna(row.get("_tb_top_rank", np.nan)) else 99
        candidates.append(data)
        if len(candidates) >= limit:
            break
    return candidates


def _cp_tb_assignment_weight(row, role, role_ranks, meta=None):
    """Lexikografisch gewichteter Wert für Slot-Spieler-Kombinationen.

    Priorität: Rollenabdeckung mit Top-3-Profilen > gewählter Score-/Fit-Wert >
    Rollenrang als kleiner Tie-Breaker. Dadurch optimiert der Score-Modus wirklich
    auf den Final Role Score, ohne die gewünschte Top-3-Abdeckung aufzugeben.
    """
    player = str(row.get("_tb_player", row.get("Spieler", ""))).strip()
    top_rank = role_ranks.get((player, str(role).strip()), 99)
    primary = _cp_tb_float(row.get("_tb_primary", row.get("Final Role Score", np.nan)), 0.0)
    secondary = _cp_tb_float(row.get("_tb_secondary", row.get("Role Fit", np.nan)), 0.0)

    # Sehr großer Gleichbonus für Top-3-Kandidaten: Die Optimierung maximiert zuerst,
    # wie viele Rollen mit echten Top-3-Profilen gefüllt werden. Innerhalb dieser Gruppe
    # entscheidet aber der aktive Modus: Score-Modus = Score, Fit-Modus = Fit.
    if top_rank <= 3:
        coverage_bonus = 1_000_000.0
        rank_tiebreak = {1: 6.0, 2: 4.0, 3: 2.0}.get(top_rank, 0.0)
    else:
        # Nicht-Top-3-Spieler bleiben möglich, wenn ein Slot sonst leer wäre, können aber
        # keine Top-3-Abdeckung an anderer Stelle verdrängen.
        coverage_bonus = 20_000.0
        rank_tiebreak = 0.0

    return coverage_bonus + (primary * 10.0) + (secondary * 0.02) + rank_tiebreak


def _cp_tb_add_flow_edge(graph, u, v, cap, cost, payload=None):
    graph[u].append([v, len(graph[v]), cap, cost, payload])
    graph[v].append([u, len(graph[u]) - 1, 0, -cost, None])


def _cp_tb_min_cost_flow(graph, source, sink, max_flow):
    """Kleiner SPFA-basierter Min-Cost-Flow für die 11-Slot-Zuordnung."""
    n = len(graph)
    flow = 0
    total_cost = 0
    while flow < max_flow:
        dist = [float("inf")] * n
        prev_node = [-1] * n
        prev_edge = [-1] * n
        in_queue = [False] * n
        dist[source] = 0
        queue = [source]
        in_queue[source] = True
        qi = 0
        while qi < len(queue):
            u = queue[qi]
            qi += 1
            in_queue[u] = False
            for ei, edge in enumerate(graph[u]):
                v, rev, cap, cost, payload = edge
                if cap > 0 and dist[u] + cost < dist[v]:
                    dist[v] = dist[u] + cost
                    prev_node[v] = u
                    prev_edge[v] = ei
                    if not in_queue[v]:
                        queue.append(v)
                        in_queue[v] = True
        if dist[sink] == float("inf"):
            break
        add = max_flow - flow
        v = sink
        while v != source:
            u = prev_node[v]
            ei = prev_edge[v]
            add = min(add, graph[u][ei][2])
            v = u
        v = sink
        while v != source:
            u = prev_node[v]
            ei = prev_edge[v]
            edge = graph[u][ei]
            edge[2] -= add
            graph[edge[0]][edge[1]][2] += add
            v = u
        flow += add
        total_cost += add * dist[sink]
    return flow, total_cost


def cp_teambuilder_build_assignments(state, slots, mode):
    """Teamoptimale Zuordnung hinter den bestehenden vier Buttons.

    Jeder Spieler wird höchstens einmal vergeben. Die Optimierung maximiert zuerst die
    Rollenabdeckung über Top-3-Rollenprofile und danach den gewählten Score-/Fit-Wert.
    """
    slots = [dict(slot) for slot in slots]
    df, meta = _cp_tb_candidate_base_df(state, mode)
    if df.empty or not slots:
        return [{"slot": dict(slot), "player": None, "alternatives": []} for slot in slots]

    role_ranks = cp_teambuilder_player_role_ranks(state, mode)
    max_candidates_per_slot = 42
    candidate_rows_by_slot = []
    player_names = set()

    for slot in slots:
        role = str(slot.get("role", "")).strip()
        role_df = df.loc[df["_tb_role"].eq(role)].copy()
        if role_df.empty:
            candidate_rows_by_slot.append([])
            continue
        role_df["_tb_top_rank"] = role_df.apply(lambda r: role_ranks.get((r.get("_tb_player"), role), 99), axis=1)
        role_df["_tb_weight"] = role_df.apply(lambda r: _cp_tb_assignment_weight(r, role, role_ranks, meta), axis=1)
        top3_df = role_df.loc[role_df["_tb_top_rank"].le(3)].copy()
        best_df = role_df.sort_values(["_tb_weight", "_tb_primary", "_tb_secondary", "_tb_player"], ascending=[False, False, False, True], na_position="last")
        pool = pd.concat([top3_df, best_df.head(max_candidates_per_slot)], ignore_index=False)
        pool = pool.drop_duplicates(subset=["_tb_player"], keep="first")
        pool = pool.sort_values(["_tb_weight", "_tb_primary", "_tb_secondary", "_tb_player"], ascending=[False, False, False, True], na_position="last")
        rows = []
        for _, row in pool.iterrows():
            player = str(row.get("_tb_player", "")).strip()
            if not player:
                continue
            player_names.add(player)
            rows.append(row)
        candidate_rows_by_slot.append(rows)

    slot_count = len(slots)
    player_list = sorted(player_names)
    player_node = {name: 1 + slot_count + i for i, name in enumerate(player_list)}
    dummy_start = 1 + slot_count + len(player_list)
    source = 0
    sink = dummy_start + slot_count
    graph = [[] for _ in range(sink + 1)]

    for idx in range(slot_count):
        _cp_tb_add_flow_edge(graph, source, 1 + idx, 1, 0)
        for row in candidate_rows_by_slot[idx]:
            role = str(slots[idx].get("role", "")).strip()
            player = str(row.get("_tb_player", "")).strip()
            weight = _cp_tb_assignment_weight(row, role, role_ranks, meta)
            data = row.drop(labels=[c for c in row.index if str(c).startswith("_tb_")], errors="ignore").to_dict()
            data["_tb_top_rank"] = int(role_ranks.get((player, role), 99))
            payload = {"slot_idx": idx, "player": data}
            _cp_tb_add_flow_edge(graph, 1 + idx, player_node[player], 1, -int(round(weight * 10)), payload)
        # Dummy pro Slot: erlaubt leere Karten, aber nur wenn kein sinnvoller Match verfügbar ist.
        _cp_tb_add_flow_edge(graph, 1 + idx, dummy_start + idx, 1, 0, {"slot_idx": idx, "player": None})

    for name in player_list:
        _cp_tb_add_flow_edge(graph, player_node[name], sink, 1, 0)
    for idx in range(slot_count):
        _cp_tb_add_flow_edge(graph, dummy_start + idx, sink, 1, 0)

    _cp_tb_min_cost_flow(graph, source, sink, slot_count)

    selected_by_slot = {idx: None for idx in range(slot_count)}
    for idx in range(slot_count):
        slot_node = 1 + idx
        for edge in graph[slot_node]:
            payload = edge[4]
            if payload and payload.get("slot_idx") == idx and edge[2] == 0:
                selected_by_slot[idx] = payload.get("player")
                break

    selected_players = {
        str(player.get("Spieler", "")).strip()
        for player in selected_by_slot.values()
        if isinstance(player, dict) and str(player.get("Spieler", "")).strip()
    }

    assignments = []
    for idx, slot in enumerate(slots):
        selected = selected_by_slot.get(idx)
        selected_name = str(selected.get("Spieler", "")).strip() if isinstance(selected, dict) else ""
        other_used = selected_players - ({selected_name} if selected_name else set())
        candidates = cp_teambuilder_ranked_candidates(
            state,
            slot.get("role"),
            mode,
            used_players=other_used,
            limit=7,
            role_ranks=role_ranks,
        )
        alternatives = []
        seen_names = {selected_name} if selected_name else set()
        for cand in candidates:
            cand_name = str(cand.get("Spieler", "")).strip()
            if not cand_name or cand_name in seen_names:
                continue
            seen_names.add(cand_name)
            alternatives.append(cand)
            if len(alternatives) >= 5:
                break
        assignments.append({
            "slot": dict(slot),
            "player": selected,
            "alternatives": alternatives,
        })
    return assignments

def cp_teambuilder_display_name(template_name, template):
    formation = str(template.get("formation", "") or "").strip()
    name = str(template_name or "").strip()
    if name == "FCN 25/26" and formation:
        return f"FCN 25/26 ({formation})"
    if formation and name and name != formation:
        return f"{name} ({formation})"
    return name or formation or "Formation"


def cp_teambuilder_template_header_html(template_name, template):
    display_name = cp_teambuilder_display_name(template_name, template)
    return f"""
    <div style='font-family:Arial,sans-serif;border:1px solid #ead7d7;border-left:7px solid #8B0000;background:#FAF7F7;border-radius:18px;padding:14px 16px;margin:0 0 12px 0;'>
      <div style='font-size:18px;font-weight:900;color:#8B0000;margin-bottom:4px;'>{html.escape(display_name)}</div>
      <div style='font-size:13px;font-weight:800;color:#171717;margin-bottom:7px;'>{html.escape(template.get('short_label', ''))}</div>
      <div style='font-size:13px;line-height:1.5;color:#374151;'>{html.escape(template.get('description', ''))}</div>
    </div>
    """


def _cp_tb_num(value, digits=1):
    try:
        if pd.isna(value):
            return "–"
        return f"{float(value):.{digits}f}"
    except Exception:
        return "–"


def _cp_tb_score_css_class(score):
    value = _cp_tb_float(score, np.nan)
    if pd.isna(value):
        return "cp-tb-score-na"
    if value >= 120:
        return "cp-tb-score-darkgreen"
    if value >= 90:
        return "cp-tb-score-lightgreen"
    if value >= 75:
        return "cp-tb-score-yellow"
    if value >= 60:
        return "cp-tb-score-orange"
    return "cp-tb-score-red"



def _cp_tb_score_text_short(score):
    value = _cp_tb_float(score, np.nan)
    if pd.isna(value):
        return "–"
    return f"{value:.0f}"


def _cp_tb_player_accent(player):
    if not player:
        return "#8B0000"
    try:
        row = pd.Series(player)
        return "#8B0000" if cp_teambuilder_is_fcn_row(row) else "#10223b"
    except Exception:
        return "#8B0000"


def _cp_tb_player_bg(player):
    if not player:
        return "#fffafa"
    try:
        row = pd.Series(player)
        return "#fff3f3" if cp_teambuilder_is_fcn_row(row) else "#f1f5fb"
    except Exception:
        return "#fff3f3"


def _cp_tb_first_nonempty_from_dict(data, keys, default="–"):
    data = data or {}
    for key in keys:
        value = data.get(key, None)
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return default


def cp_teambuilder_candidate_line(player):
    if not player:
        return "–"
    name = _cp_tb_first_nonempty_from_dict(player, ["Spieler"], default="–")
    team = _cp_tb_first_nonempty_from_dict(player, ["Team", "TM aktueller Verein"], default="–")
    fit = _cp_tb_num(player.get("Role Fit"), 1)
    score = _cp_tb_num(player.get("Final Role Score"), 1)
    return f"{name} · Fit {fit} · Score {score} · {team}"



def _cp_tb_player_display_fields(player):
    if not player:
        return {"name": "Kein Treffer", "team_line": "–", "fit": "–", "score": "–"}
    name = _cp_tb_first_nonempty_from_dict(player, ["Spieler"], default="–")
    team = _cp_tb_first_nonempty_from_dict(player, ["Team", "TM aktueller Verein"], default="–")
    liga = _cp_tb_first_nonempty_from_dict(player, ["Liga"], default="")
    team_line = team if not liga or liga == "–" else f"{team} · {liga}"
    return {
        "name": name,
        "team_line": team_line,
        "fit": _cp_tb_num(player.get("Role Fit"), 1),
        "score": _cp_tb_num(player.get("Final Role Score"), 1),
        "score_class": _cp_tb_score_css_class(player.get("Final Role Score")),
    }


def _cp_tb_player_accent(player):
    if not player:
        return "#8B0000"
    try:
        row = pd.Series(player)
        return "#8B0000" if cp_teambuilder_is_fcn_row(row) else "#10223b"
    except Exception:
        return "#8B0000"


def _cp_tb_candidate_payload(player):
    fields = _cp_tb_player_display_fields(player)
    score_short = _cp_tb_score_text_short(player.get("Final Role Score") if player else np.nan)
    return {
        "name": fields["name"],
        "team": fields["team_line"],
        "fit": fields["fit"],
        "score": fields["score"],
        "scoreShort": score_short,
        "scoreClass": _cp_tb_score_css_class(player.get("Final Role Score") if player else np.nan),
        "accent": _cp_tb_player_accent(player),
        "bg": _cp_tb_player_bg(player),
        "label": f"{fields['name']} ({score_short})",
    }


def _cp_tb_option_html(player, selected=False):
    payload = _cp_tb_candidate_payload(player)
    attrs = {
        "data-name": payload["name"],
        "data-team": payload["team"],
        "data-fit": payload["fit"],
        "data-score": payload["score"],
        "data-score-short": payload["scoreShort"],
        "data-score-class": payload["scoreClass"],
        "data-accent": payload["accent"],
        "data-bg": payload["bg"],
    }
    attr_text = " ".join(f'{k}="{html.escape(str(v), quote=True)}"' for k, v in attrs.items())
    selected_attr = " selected" if selected else ""
    return f'<option value="{html.escape(payload["name"], quote=True)}" {attr_text}{selected_attr}>{html.escape(payload["label"])}</option>'


def _cp_tb_candidates_json(candidates):
    payloads = []
    seen = set()
    for cand in candidates or []:
        if not cand:
            continue
        payload = _cp_tb_candidate_payload(cand)
        name = str(payload.get("name", "")).strip()
        if not name or name in seen or name == "–":
            continue
        seen.add(name)
        payloads.append(payload)
    return html.escape(json.dumps(payloads, ensure_ascii=False), quote=True)


def _cp_tb_slot_side(slot):
    slot_id = str(slot.get("id", "")).strip().upper()
    label = str(slot.get("label", "")).strip().lower()
    if slot_id.startswith("L") or " links" in label or label in {"lv", "liv", "lwb", "la", "lm"}:
        return "left"
    if slot_id.startswith("R") or " rechts" in label or label in {"rv", "riv", "rwb", "ra", "rm"}:
        return "right"
    return "center"


def _cp_tb_slot_kind(slot):
    slot_id = str(slot.get("id", "")).strip().upper()
    label = str(slot.get("label", "")).strip().lower()
    if slot_id == "GK" or label == "tw":
        return "gk"
    if slot_id in {"LCB", "RCB", "CB"} or label in {"liv", "riv", "ziv", "iv"}:
        return "cb"
    if slot_id in {"LB", "RB", "LWB", "RWB"} or label in {"lv", "rv", "lwb", "rwb"}:
        return "fb"
    if slot_id in {"DM", "LDM", "RDM"} or label == "6" or "6 " in label or label.startswith("6"):
        return "dm"
    if slot_id in {"LCM", "RCM"} or "8" in label:
        return "cm8"
    if slot_id in {"CAM", "LAM", "RAM"} or label == "10" or label.startswith("halbraum"):
        return "am"
    if slot_id in {"LW", "RW", "LM", "RM"} or label in {"la", "ra", "lm", "rm", "la/lm", "ra/rm"}:
        return "wing"
    if slot_id in {"ST", "LST", "RST"} or label == "st" or label.startswith("st "):
        return "st"
    return "other"


def cp_teambuilder_adjust_slot_positions(template_name, template, slots):
    """Heuristische Layout-Optimierung für weniger Overlap und realistischere Staffelung."""
    adjusted = []
    base_slots = [dict(slot) for slot in slots]
    kinds = [_cp_tb_slot_kind(slot) for slot in base_slots]
    cb_count = sum(kind == "cb" for kind in kinds)
    has_ten = any(kind == "am" for kind in kinds)
    has_true_wings = any(kind == "wing" for kind in kinds)
    formation_name = str(template.get("formation", template_name) or template_name)

    for slot in base_slots:
        s = dict(slot)
        x = _cp_tb_percent(s.get("x"), 50.0)
        y = _cp_tb_percent(s.get("y"), 50.0)
        side = _cp_tb_slot_side(s)
        kind = _cp_tb_slot_kind(s)

        if kind == "fb":
            y -= 4.0
        elif kind == "cb":
            spread = 4.5 if cb_count <= 2 else 5.8
            if side == "left":
                x -= spread
            elif side == "right":
                x += spread
            else:
                y += 1.0
        elif kind == "dm":
            # Im 4-2-3-1 lagen die Sechser optisch zu hoch. Dort tiefer lassen,
            # in anderen Grundordnungen bleibt die leichte Vorwärtsstaffelung erhalten.
            if formation_name == "4-2-3-1":
                y += 1.5
            else:
                y -= 4.0
        elif kind == "wing":
            y -= 4.0
            if side == "left":
                x -= 1.5
            elif side == "right":
                x += 1.5
        elif kind == "cm8":
            if not has_ten:
                y -= 4.5
            if not has_true_wings:
                if side == "left":
                    x -= 5.5
                elif side == "right":
                    x += 5.5
                y -= 1.5
            elif formation_name == "4-4-2 Raute":
                if side == "left":
                    x -= 3.0
                elif side == "right":
                    x += 3.0
                y -= 1.0
        elif kind == "st" and formation_name in {"3-5-2", "4-4-2 Raute"}:
            if side == "left":
                x -= 2.5
            elif side == "right":
                x += 2.5

        s["x"] = max(10.0, min(90.0, x))
        s["y"] = max(14.0, min(90.0, y))
        adjusted.append(s)

    # Kollisionsvermeidung per leichter Relaxation auf Basis der Kartenmitten.
    min_dx = 16.0   # geschätzte halbe Kartenbreite in Prozent
    min_dy = 13.2   # geschätzte halbe Kartenhöhe in Prozent
    for _ in range(30):
        moved = False
        for i in range(len(adjusted)):
            for j in range(i + 1, len(adjusted)):
                a = adjusted[i]
                b = adjusted[j]
                dx = float(b["x"]) - float(a["x"])
                dy = float(b["y"]) - float(a["y"])
                overlap_x = min_dx - abs(dx)
                overlap_y = min_dy - abs(dy)
                if overlap_x <= 0 or overlap_y <= 0:
                    continue
                kind_a = _cp_tb_slot_kind(a)
                kind_b = _cp_tb_slot_kind(b)
                prefer_vertical = (
                    abs(dx) < 8.0
                    or "gk" in {kind_a, kind_b}
                    or "st" in {kind_a, kind_b}
                )
                if prefer_vertical or overlap_y < overlap_x:
                    push = min(2.8, overlap_y / 2.0 + 0.35)
                    if dy >= 0:
                        a["y"] -= push
                        b["y"] += push
                    else:
                        a["y"] += push
                        b["y"] -= push
                else:
                    push = min(3.2, overlap_x / 2.0 + 0.35)
                    if dx >= 0:
                        a["x"] -= push
                        b["x"] += push
                    else:
                        a["x"] += push
                        b["x"] -= push
                a["x"] = max(9.0, min(91.0, float(a["x"])))
                b["x"] = max(9.0, min(91.0, float(b["x"])))
                a["y"] = max(13.0, min(91.0, float(a["y"])))
                b["y"] = max(13.0, min(91.0, float(b["y"])))
                moved = True
        if not moved:
            break
    return adjusted


def cp_teambuilder_render_formation_html(template_name, template, slots, assignments=None, mode_label="Rollen-Vorschau"):
    assignments_by_id = {}
    if assignments:
        assignments_by_id = {a.get("slot", {}).get("id"): a for a in assignments}

    positioned_slots = cp_teambuilder_adjust_slot_positions(template_name, template, slots)

    css = """
    <style>
      .cp-tb-wrap { font-family:Arial, Helvetica, sans-serif; width:100%; max-width:1120px; box-sizing:border-box; overflow:hidden; }
      .cp-tb-meta { display:flex; align-items:flex-start; justify-content:space-between; gap:12px; flex-wrap:wrap; margin:0 0 8px 0; }
      .cp-tb-title { font-weight:900; color:#8B0000; font-size:19px; }
      .cp-tb-sub { color:#5F6368; font-size:12px; line-height:1.35; }
      .cp-tb-pitch { position:relative; width:100%; height:705px; border-radius:24px; overflow:hidden; border:2px solid #d7b0b0; background:linear-gradient(180deg,#fdfafa 0%,#f7eded 100%); box-shadow:0 8px 22px rgba(0,0,0,.08); }
      .cp-tb-pitch:before { content:""; position:absolute; inset:24px; border:2px solid rgba(139,0,0,.18); border-radius:18px; }
      .cp-tb-halfline { position:absolute; left:24px; right:24px; top:50%; border-top:2px solid rgba(139,0,0,.18); }
      .cp-tb-circle { position:absolute; left:50%; top:50%; width:140px; height:140px; margin-left:-70px; margin-top:-70px; border:2px solid rgba(139,0,0,.16); border-radius:999px; }
      .cp-tb-box-top { position:absolute; left:31%; right:31%; top:24px; height:82px; border:2px solid rgba(139,0,0,.14); border-top:0; border-radius:0 0 16px 16px; }
      .cp-tb-box-bottom { position:absolute; left:31%; right:31%; bottom:24px; height:82px; border:2px solid rgba(139,0,0,.14); border-bottom:0; border-radius:16px 16px 0 0; }
      .cp-tb-card { --cp-tb-accent:#8B0000; --cp-tb-bg:#fff3f3; position:absolute; width:172px; min-height:78px; transform:translate(-50%,-50%); border-radius:14px; padding:6px 8px; background:var(--cp-tb-bg); border:2px solid var(--cp-tb-accent); box-shadow:0 4px 13px rgba(0,0,0,.12); text-align:center; box-sizing:border-box; }
      .cp-tb-card.empty { background:#fffafa; border-style:dashed; box-shadow:none; }
      .cp-tb-topline { display:flex; align-items:center; justify-content:center; gap:5px; margin-bottom:3px; min-width:0; }
      .cp-tb-pos { display:inline-block; padding:2px 6px; border-radius:999px; background:#8B0000; color:white; font-size:10px; line-height:1.1; font-weight:900; flex:0 0 auto; }
      .cp-tb-role { font-size:10.5px; line-height:1.12; font-weight:900; color:#171717; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; min-width:0; }
      .cp-tb-player { font-size:12.5px; line-height:1.1; font-weight:900; color:var(--cp-tb-accent); margin-top:2px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
      .cp-tb-team { font-size:10.5px; line-height:1.12; color:#5F6368; margin-top:2px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
      .cp-tb-score { margin-top:2px; font-size:10.5px; color:#374151; font-weight:800; white-space:nowrap; }
      .cp-tb-score-badge { display:inline-block; padding:1px 5px; border-radius:6px; font-weight:900; line-height:1.15; }
      .cp-tb-score-darkgreen { background:#14532d; color:#ffffff; }
      .cp-tb-score-lightgreen { background:#bbf7d0; color:#14532d; }
      .cp-tb-score-yellow { background:#fef08a; color:#713f12; }
      .cp-tb-score-orange { background:#fed7aa; color:#7c2d12; }
      .cp-tb-score-red { background:#fecaca; color:#7f1d1d; }
      .cp-tb-score-na { background:#e5e7eb; color:#374151; }
      .cp-tb-select { width:100%; max-width:100%; margin-top:3px; height:22px; border:1px solid #ead7d7; border-radius:8px; background:#FAF7F7; color:#171717; font-size:10.5px; font-weight:800; padding:0 4px; outline:none; }
      .cp-tb-select:focus { border-color:#8B0000; box-shadow:0 0 0 2px rgba(139,0,0,.12); }
      .cp-tb-select:disabled { opacity:.75; cursor:default; }
      @media (max-width: 760px) { .cp-tb-pitch { height:760px; } .cp-tb-card { width:148px; padding:6px; } .cp-tb-role { font-size:10px; } .cp-tb-player { font-size:11.5px; } .cp-tb-select { font-size:10px; } }
    </style>
    """

    on_change = (
        "var s=this;"
        "var o=s.options[s.selectedIndex];"
        "var c=s.closest('.cp-tb-card');"
        "if(!o||!c||!o.value){return;}"
        "var candidates=[];"
        "try{candidates=JSON.parse(s.dataset.candidates||'[]');}catch(e){candidates=[];}"
        "var chosen=null;"
        "for(var i=0;i<candidates.length;i++){if(candidates[i].name===o.value){chosen=candidates[i];break;}}"
        "if(!chosen){return;}"
        "c.querySelector('.cp-tb-player').textContent=chosen.name||'–';"
        "c.querySelector('.cp-tb-team').textContent=chosen.team||'–';"
        "var sc=c.querySelector('.cp-tb-score');"
        "if(sc){sc.innerHTML='Fit '+(chosen.fit||'–')+' · <span class=\"cp-tb-score-badge '+(chosen.scoreClass||'cp-tb-score-na')+'\">Score '+(chosen.scoreShort||chosen.score||'–')+'</span>'; }"
        "c.style.setProperty('--cp-tb-accent', chosen.accent||'#8B0000');"
        "c.style.setProperty('--cp-tb-bg', chosen.bg||'#fff3f3');"
        "s.dataset.activeName=chosen.name||'';"
        "while(s.firstChild){s.removeChild(s.firstChild);}"
        "var opts=[];"
        "for(var j=0;j<candidates.length;j++){if(candidates[j].name!==chosen.name){opts.push(candidates[j]);}}"
        "if(!opts.length){var none=document.createElement('option'); none.textContent='--'; none.value=''; s.appendChild(none); s.disabled=true; return;}"
        "s.disabled=false;"
        "var ph=document.createElement('option'); ph.textContent=opts[0].label||opts[0].name||'Alternative'; ph.value=''; ph.disabled=true; ph.selected=true; ph.hidden=true; s.appendChild(ph);"
        "for(var k=0;k<opts.length;k++){var op=document.createElement('option'); op.value=opts[k].name||''; op.textContent=opts[k].label||opts[k].name||''; s.appendChild(op);}"
    )

    cards = []
    for slot in positioned_slots:
        assignment = assignments_by_id.get(slot.get("id"), {})
        player = assignment.get("player") if assignment else None
        alternatives = assignment.get("alternatives") or []
        has_player = bool(player)
        card_class = "cp-tb-card" if has_player else "cp-tb-card empty"
        fields = _cp_tb_player_display_fields(player) if assignments is not None else {"name": "Noch nicht gefüllt", "team_line": "–", "fit": "–", "score": "–"}
        accent = _cp_tb_player_accent(player)
        bg = _cp_tb_player_bg(player)
        score_class = _cp_tb_score_css_class(player.get("Final Role Score") if player else np.nan)
        score_short = _cp_tb_score_text_short(player.get("Final Role Score") if player else np.nan)
        candidate_objects = []
        if has_player:
            candidate_objects.append(player)
            seen = {str(player.get("Spieler", "")).strip()}
            for alt in alternatives:
                alt_name = str(alt.get("Spieler", "")).strip()
                if not alt_name or alt_name in seen:
                    continue
                seen.add(alt_name)
                candidate_objects.append(alt)
        select_html = ""
        if has_player:
            payload_json = _cp_tb_candidates_json(candidate_objects)
            alt_payloads = [_cp_tb_candidate_payload(alt) for alt in candidate_objects[1:]]
            if alt_payloads:
                placeholder = html.escape(alt_payloads[0].get("label", "Alternative"))
                options_html = [f'<option value="" disabled selected hidden>{placeholder}</option>']
                options_html.extend(_cp_tb_option_html(alt, selected=False) for alt in candidate_objects[1:])
                select_html = f'<select class="cp-tb-select" data-active-name="{html.escape(fields["name"], quote=True)}" data-candidates="{payload_json}" onchange="{html.escape(on_change, quote=True)}">{"".join(options_html)}</select>'
            else:
                select_html = "<select class='cp-tb-select' disabled><option>--</option></select>"

        cards.append(f"""
          <div class='{card_class}' style='left:{float(slot.get('x', 50)):.1f}%; top:{float(slot.get('y', 50)):.1f}%; --cp-tb-accent:{accent}; --cp-tb-bg:{bg};'>
            <div class='cp-tb-topline'>
              <span class='cp-tb-pos'>{html.escape(str(slot.get('label', '')))}</span>
              <span class='cp-tb-role' title='{html.escape(str(slot.get('role', '')), quote=True)}'>{html.escape(str(slot.get('role', '')))}</span>
            </div>
            <div class='cp-tb-player'>{html.escape(fields['name'])}</div>
            <div class='cp-tb-team'>{html.escape(fields['team_line'])}</div>
            <div class='cp-tb-score'>Fit {html.escape(fields['fit'])} · <span class='cp-tb-score-badge {score_class}'>Score {html.escape(score_short)}</span></div>
            {select_html}
          </div>
        """)

    return css + f"""
    <div class='cp-tb-wrap'>
      <div class='cp-tb-meta'>
        <div>
          <div class='cp-tb-title'>{html.escape(cp_teambuilder_display_name(template_name, template))}</div>
          <div class='cp-tb-sub'>{html.escape(template.get('short_label', ''))}</div>
        </div>
        <div class='cp-tb-sub'><b>Modus:</b> {html.escape(mode_label)}</div>
      </div>
      <div class='cp-tb-pitch'>
        <div class='cp-tb-halfline'></div>
        <div class='cp-tb-circle'></div>
        <div class='cp-tb-box-top'></div>
        <div class='cp-tb-box-bottom'></div>
        {''.join(cards)}
      </div>
    </div>
    """


def cp_build_teambuilder_controls(state):
    if not TEAMBUILDER_TEMPLATES:
        error_msg = html.escape(str(TEAMBUILDER_TEMPLATE_LOAD_ERROR) if TEAMBUILDER_TEMPLATE_LOAD_ERROR else "Unbekannter Fehler")
        return widgets.HTML(
            "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #fecaca;border-radius:12px;background:#fff1f2;color:#7f1d1d;'>"
            "<b>Teambuilder-Vorlagen konnten nicht geladen werden.</b><br>"
            f"{error_msg}<br><br>"
            "Erwartet wird die Datei <code>teambuilder_formationen.xlsx</code> mit den Sheets <code>Vorlagen</code> und <code>Slots</code>."
            "</div>"
        )

    all_roles = state.get("all_roles", pd.DataFrame()) if isinstance(state, dict) else pd.DataFrame()
    if all_roles.empty:
        return widgets.HTML(
            "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #fecaca;border-radius:12px;background:#fff1f2;color:#7f1d1d;'>"
            "Der Teambuilder konnte keine Rollenfit-Daten finden. Bitte prüfe, ob die Pipeline <code>all_roles</code> erzeugt."
            "</div>"
        )

    role_options = cp_teambuilder_role_options(state)
    scope_options = TEAMBUILDER_PLAYER_SCOPES
    optimization_options = TEAMBUILDER_OPTIMIZATION_MODES

    template_dropdown = widgets.Dropdown(
        options=list(TEAMBUILDER_TEMPLATES.keys()),
        value="FCN 25/26" if "FCN 25/26" in TEAMBUILDER_TEMPLATES else list(TEAMBUILDER_TEMPLATES.keys())[0],
        description="Taktik:",
        layout=widgets.Layout(width="330px"),
        style={"description_width": "70px"},
    )
    player_scope = widgets.ToggleButtons(
        options=scope_options,
        value="fcn",
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px", "button_width": "auto"},
    )
    player_scope.add_class("cp-tb-fill-modes")
    optimization_mode = widgets.ToggleButtons(
        options=optimization_options,
        value="score",
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px", "button_width": "auto"},
    )
    optimization_mode.add_class("cp-tb-fill-modes")

    generate_button = widgets.Button(
        description="Elf generieren",
        button_style="",
        icon="users",
        layout=widgets.Layout(width="220px"),
    )
    generate_button.add_class("clubprofile-red-button")
    reset_button = widgets.Button(
        description="Vorlage zurücksetzen",
        button_style="",
        icon="undo",
        layout=widgets.Layout(width="210px"),
    )
    description_html = widgets.HTML()
    role_grid = widgets.GridBox(layout=widgets.Layout(
        width="calc(100% - 8px)",
        grid_template_columns="repeat(auto-fit, minmax(230px, 1fr))",
        grid_gap="8px",
        margin="8px 8px 14px 0",
    ))
    pitch_html = widgets.HTML(layout=widgets.Layout(width="100%"))
    role_widgets = {}

    def current_template():
        return TEAMBUILDER_TEMPLATES[template_dropdown.value]

    def current_slots():
        slots = []
        for slot in current_template().get("slots", []):
            s = dict(slot)
            widget = role_widgets.get(slot.get("id"))
            if widget is not None:
                s["role"] = widget.value
            slots.append(s)
        return slots

    def render_preview(*args):
        template = current_template()
        template_name = template_dropdown.value
        mode_label = cp_teambuilder_mode_label(player_scope.value, optimization_mode.value)
        description_html.value = cp_teambuilder_template_header_html(template_name, template)
        pitch_html.value = cp_teambuilder_render_formation_html(template_name, template, current_slots(), assignments=None, mode_label=mode_label)

    def rebuild_role_grid(*args):
        role_widgets.clear()
        children = []
        for slot in current_template().get("slots", []):
            role_value = slot.get("role") if slot.get("role") in role_options else (role_options[0] if role_options else "")
            dropdown = widgets.Dropdown(
                options=role_options,
                value=role_value,
                description=str(slot.get("label", "")),
                layout=widgets.Layout(width="100%"),
                style={"description_width": "95px"},
            )
            dropdown.observe(render_preview, names="value")
            role_widgets[slot.get("id")] = dropdown
            children.append(dropdown)
        role_grid.children = children
        render_preview()

    def generate_assignments(_=None):
        template = current_template()
        template_name = template_dropdown.value
        mode = cp_teambuilder_mode_from_controls(player_scope.value, optimization_mode.value)
        mode_label = cp_teambuilder_mode_label(player_scope.value, optimization_mode.value)
        slots = current_slots()
        assignments = cp_teambuilder_build_assignments(state, slots, mode)
        pitch_html.value = cp_teambuilder_render_formation_html(template_name, template, slots, assignments=assignments, mode_label=mode_label)

    template_dropdown.observe(rebuild_role_grid, names="value")
    player_scope.observe(render_preview, names="value")
    optimization_mode.observe(render_preview, names="value")
    generate_button.on_click(generate_assignments)
    reset_button.on_click(rebuild_role_grid)
    rebuild_role_grid()

    mode_controls = widgets.HBox([
        widgets.VBox([
            widgets.HTML("<div style='font-family:Arial,sans-serif;font-weight:900;color:#171717;margin:0 0 6px 0;'>Spielerpool</div>"),
            player_scope,
        ], layout=widgets.Layout(width="calc(50% - 6px)", min_width="300px")),
        widgets.VBox([
            widgets.HTML("<div style='font-family:Arial,sans-serif;font-weight:900;color:#171717;margin:0 0 6px 0;'>Optimierung</div>"),
            optimization_mode,
        ], layout=widgets.Layout(width="calc(50% - 6px)", min_width="300px")),
    ], layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap", align_items="flex-start", margin="12px 0 4px 0"))

    controls = widgets.VBox([
        widgets.HBox([template_dropdown], layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap")),
        description_html,
        mode_controls,
        widgets.HTML("<div style='font-family:Arial,sans-serif;font-weight:900;color:#171717;margin:12px 0 6px 0;'>Rollen je Slot</div>"),
        role_grid,
        widgets.HBox([generate_button, reset_button], layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap", justify_content="center", margin="0 0 14px 0")),
        pitch_html,
    ], layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden"))
    return controls

In [ ]:
# =========================
# ClubProfile 2.0 – Vereinheitlichte GUI mit Lazy Start
# =========================

SEPARATOR = "|||"


def cp_encode_selection(group_name, player_name):
    return f"{group_name}{SEPARATOR}{player_name}"


def cp_decode_selection(value):
    if value in (None, ""):
        return None, None
    group_name, player_name = value.split(SEPARATOR, 1)
    return group_name, player_name


def cp_spider_player_exists(player_name):
    if not SPIDER_READY:
        return False
    return player_name in set(SPIDER_NS["df"][SPIDER_NS["PLAYER_COL"]].dropna().astype(str).unique())


def cp_spider_candidate_groups(player_name):
    if not SPIDER_READY or not cp_spider_player_exists(player_name):
        return []
    return SPIDER_NS["get_candidate_groups_for_player"](player_name)


def cp_spider_placeholder_html():
    return """
    <div class='clubprofile-spider-placeholder'>
      <div style='font-size:22px;font-weight:900;color:#8B0000;margin-bottom:8px;'>Metrikvergleich / Spiderplot</div>
      <div style='font-size:14px;line-height:1.55;color:#374151;max-width:760px;'>
        Wähle die Vergleichsspieler aus und klicke auf <b>Spiderplot generieren</b>.
        Der Platzhalter hält den Ergebnisbereich stabil, damit die Seite beim Wechsel zwischen den Tabs nicht springt.
      </div>
    </div>
    """


def cp_build_print_download_html(player_name, state):
    profile_html = player_profile_html(player_name, state)
    doc_html = _standalone_profile_html(profile_html, player_name=player_name)
    payload = base64.b64encode(doc_html.encode("utf-8")).decode("ascii")
    filename = f"clubprofile_{_slugify_filename(player_name)}.html"
    return f"""
    <div class='clubprofile-print-box'>
      <div style='font-weight:900;color:#8B0000;margin-bottom:6px;'>Druckversion vorbereitet</div>
      <div style='font-size:13px;color:#374151;margin-bottom:10px;'>
        Öffne die Druckversion in einem neuen Tab oder lade sie herunter. Dort kannst du sie über den Browser als PDF speichern.
      </div>
      <a class='clubprofile-print-link' href='data:text/html;base64,{payload}' target='_blank' download='{html.escape(filename, quote=True)}'>
        Druckversion herunterladen
      </a>
    </div>
    """


def cp_build_spider_controls(selected_player):
    if not SPIDER_READY:
        return widgets.VBox([
            widgets.HTML("<b>Spiderplot-Modul konnte nicht geladen werden.</b>"),
            widgets.HTML(f"<pre>{html.escape(str(SPIDER_LOAD_ERROR))}</pre>"),
        ])

    if not cp_spider_player_exists(selected_player):
        return widgets.HTML(
            "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #e5e7eb;border-radius:12px;background:#f9fafb;'>"
            f"Für <b>{html.escape(str(selected_player))}</b> liegen keine Spiderplot-Metriken im Sheet <code>Werte_lang</code> vor."
            "</div>"
        )

    is_fcn = SPIDER_NS["is_fcn_player"](selected_player)
    groups = cp_spider_candidate_groups(selected_player)
    if not groups:
        return widgets.HTML(
            "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #e5e7eb;border-radius:12px;background:#f9fafb;'>"
            f"Für <b>{html.escape(str(selected_player))}</b> wurde keine passende Spiderplot-Gruppe gefunden."
            "</div>"
        )

    plot_output = widgets.Output()
    links_output = widgets.Output()
    generate_button = widgets.Button(description="Spiderplot generieren", button_style="", icon="line-chart", layout=widgets.Layout(width="260px"))
    generate_button.add_class("clubprofile-red-button")
    status = widgets.Output()
    with plot_output:
        display(IPyHTML(cp_spider_placeholder_html()))

    if is_fcn:
        options = [("— Vergleichsspieler wählen —", None)]
        seen = set()
        for group_name in groups:
            group_df = SPIDER_NS["get_group_df_by_name"](group_name)
            players = sorted(p for p in group_df[SPIDER_NS["PLAYER_COL"]].dropna().astype(str).unique() if p != selected_player)
            for player in players:
                value = cp_encode_selection(group_name, player)
                if value in seen:
                    continue
                seen.add(value)
                options.append((f"{player} [{group_name}]", value))

        comparison1_dropdown = widgets.Dropdown(
            options=options,
            description="Vergleich 1",
            layout=widgets.Layout(width="390px"),
            style={"description_width": "100px"},
        )
        comparison2_dropdown = widgets.Dropdown(
            options=[("— kein zweiter Vergleichsspieler —", None)],
            description="Vergleich 2",
            layout=widgets.Layout(width="390px"),
            style={"description_width": "100px"},
            disabled=True,
        )

        def refresh_comparison2(*args):
            group_name, comp1 = cp_decode_selection(comparison1_dropdown.value)
            options2 = [("— kein zweiter Vergleichsspieler —", None)]
            if group_name is not None:
                group_df = SPIDER_NS["get_group_df_by_name"](group_name)
                players = sorted(
                    p for p in group_df[SPIDER_NS["PLAYER_COL"]].dropna().astype(str).unique()
                    if p not in {selected_player, comp1}
                )
                options2.extend((p, cp_encode_selection(group_name, p)) for p in players)
            comparison2_dropdown.options = options2
            comparison2_dropdown.value = None
            comparison2_dropdown.disabled = group_name is None
            generate_button.disabled = group_name is None
            # Statusausgaben für Endnutzer bewusst ausgeblendet.

        def generate_plot(_):
            with plot_output:
                clear_output(wait=True)
            with links_output:
                clear_output(wait=True)
            try:
                group_name, comp1 = cp_decode_selection(comparison1_dropdown.value)
                group2, comp2 = cp_decode_selection(comparison2_dropdown.value)
                if group_name is None or comp1 is None:
                    raise ValueError("Bitte Vergleichsspieler 1 auswählen.")
                comparison_players = [comp1]
                if comp2:
                    if group2 != group_name:
                        raise ValueError("Vergleich 2 muss aus derselben Gruppe stammen.")
                    comparison_players.append(comp2)
                result = SPIDER_NS["make_spider_plot_comparison"](
                    reference_player=selected_player,
                    comparison_players=comparison_players,
                    explicit_group=group_name,
                    save_plot=False,
                )
                with plot_output:
                    display(result["figure"])
                    plt.close(result["figure"])
                with links_output:
                    display(IPyHTML(SPIDER_NS["build_clickable_links_html"](result["non_fcn_players"])))
            except Exception as exc:
                with plot_output:
                    print(f"Fehler: {exc}")

        comparison1_dropdown.observe(refresh_comparison2, names="value")
        generate_button.on_click(generate_plot)
        generate_button.disabled = True
        refresh_comparison2()
        intro = widgets.HTML(
            f"<div style='font-family:Arial,sans-serif;margin-bottom:8px;'>"
            f"<b>{html.escape(selected_player)}</b> ist ein FCN-Spieler und wird automatisch als Referenzspieler verwendet."
            "</div>"
        )
        controls = widgets.VBox([
            intro,
            widgets.HBox([comparison1_dropdown, comparison2_dropdown], layout=widgets.Layout(flex_flow="row wrap", gap="10px")),
            widgets.HBox([generate_button], layout=widgets.Layout(width="100%", justify_content="center", margin="8px 0 2px 0")),
        ])
    else:
        ref_options = [("— FCN-Referenz wählen —", None)]
        seen = set()
        for group_name in groups:
            group_df = SPIDER_NS["get_group_df_by_name"](group_name)
            players = sorted(p for p in group_df[SPIDER_NS["PLAYER_COL"]].dropna().astype(str).unique() if p != selected_player)
            for player in players:
                if not SPIDER_NS["is_fcn_player"](player):
                    continue
                value = cp_encode_selection(group_name, player)
                if value in seen:
                    continue
                seen.add(value)
                ref_options.append((f"{player} [{group_name}]", value))

        if len(ref_options) == 1:
            return widgets.HTML(
                "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #e5e7eb;border-radius:12px;background:#f9fafb;'>"
                f"Für <b>{html.escape(str(selected_player))}</b> wurde kein passender FCN-Referenzspieler in derselben Spiderplot-Gruppe gefunden."
                "</div>"
            )

        reference_dropdown = widgets.Dropdown(
            options=ref_options,
            description="FCN-Referenz",
            layout=widgets.Layout(width="390px"),
            style={"description_width": "115px"},
        )
        comparison2_dropdown = widgets.Dropdown(
            options=[("— kein zweiter Vergleichsspieler —", None)],
            description="Vergleich 2",
            layout=widgets.Layout(width="390px"),
            style={"description_width": "100px"},
            disabled=True,
        )

        def refresh_comparison2(*args):
            group_name, reference_player = cp_decode_selection(reference_dropdown.value)
            options2 = [("— kein zweiter Vergleichsspieler —", None)]
            if group_name is not None:
                group_df = SPIDER_NS["get_group_df_by_name"](group_name)
                players = sorted(
                    p for p in group_df[SPIDER_NS["PLAYER_COL"]].dropna().astype(str).unique()
                    if p not in {selected_player, reference_player}
                )
                options2.extend((p, cp_encode_selection(group_name, p)) for p in players)
            comparison2_dropdown.options = options2
            comparison2_dropdown.value = None
            comparison2_dropdown.disabled = group_name is None
            generate_button.disabled = group_name is None
            # Statusausgaben für Endnutzer bewusst ausgeblendet.

        def generate_plot(_):
            with plot_output:
                clear_output(wait=True)
            with links_output:
                clear_output(wait=True)
            try:
                group_name, reference_player = cp_decode_selection(reference_dropdown.value)
                group2, comp2 = cp_decode_selection(comparison2_dropdown.value)
                if group_name is None or reference_player is None:
                    raise ValueError("Bitte FCN-Referenzspieler auswählen.")
                comparison_players = [selected_player]
                if comp2:
                    if group2 != group_name:
                        raise ValueError("Vergleich 2 muss aus derselben Gruppe stammen.")
                    comparison_players.append(comp2)
                result = SPIDER_NS["make_spider_plot_comparison"](
                    reference_player=reference_player,
                    comparison_players=comparison_players,
                    explicit_group=group_name,
                    save_plot=False,
                )
                with plot_output:
                    display(result["figure"])
                    plt.close(result["figure"])
                with links_output:
                    display(IPyHTML(SPIDER_NS["build_clickable_links_html"](result["non_fcn_players"])))
            except Exception as exc:
                with plot_output:
                    print(f"Fehler: {exc}")

        reference_dropdown.observe(refresh_comparison2, names="value")
        generate_button.on_click(generate_plot)
        generate_button.disabled = True
        refresh_comparison2()
        intro = widgets.HTML(
            f"<div style='font-family:Arial,sans-serif;margin-bottom:8px;'>"
            f"<b>{html.escape(selected_player)}</b> ist kein FCN-Spieler und wird automatisch als Vergleichsspieler 1 verwendet."
            "</div>"
        )
        controls = widgets.VBox([
            intro,
            widgets.HBox([reference_dropdown, comparison2_dropdown], layout=widgets.Layout(flex_flow="row wrap", gap="10px")),
            widgets.HBox([generate_button], layout=widgets.Layout(width="100%", justify_content="center", margin="8px 0 2px 0")),
        ])

    return widgets.VBox([
        controls,
        plot_output,
        links_output,
    ], layout=widgets.Layout(width="100%", max_width="1180px", overflow_x="visible"))


POSITION_FILTER_GROUPS = [
    ("Alle Positionen", ""),
    ("TW", "GK"),
    ("IV", "IV"),
    ("LV", "LV"),
    ("RV", "RV"),
    ("AV", "AV"),
    ("DM / 6", "DM"),
    ("ZM / 8", "ZM"),
    ("OM / 10", "OM"),
    ("Flügel", "WING"),
    ("ST", "ST"),
]

POSITION_FILTER_MAP = {
    "GK": {"GK"},
    "IV": {"CB", "LCB", "RCB"},
    "LV": {"LB", "LWB"},
    "RV": {"RB", "RWB"},
    "AV": {"LB", "RB", "LWB", "RWB", "FB"},
    "DM": {"DM", "DMF", "LDMF", "RDMF"},
    "ZM": {"CM", "CMF", "LCMF", "RCMF", "LCMF3", "RCMF3"},
    "OM": {"AM", "AMF", "CAM"},
    "WING": {"LW", "RW", "LWF", "RWF", "LM", "RM"},
    "ST": {"ST", "CF", "FW"},
}


def cp_position_filter_options(records):
    present_positions = {str(r.get("position", "")).strip().upper() for r in records if str(r.get("position", "")).strip()}
    options = [(label, key) for label, key in POSITION_FILTER_GROUPS]
    grouped_positions = set().union(*POSITION_FILTER_MAP.values()) if POSITION_FILTER_MAP else set()
    for pos in sorted(present_positions - grouped_positions):
        options.append((pos, f"POS::{pos}"))
    return options


def cp_record_matches_position(record_position, filter_key):
    if not filter_key:
        return True
    pos = str(record_position or "").strip().upper()
    if filter_key.startswith("POS::"):
        return pos == filter_key.split("::", 1)[1]
    return pos in POSITION_FILTER_MAP.get(filter_key, {filter_key})


def start_clubprofile_app():
    """Startet die App schnell und lädt die schwere Rollenfit-Pipeline erst nach Button-Klick."""
    default_mode = "best_2_bundesliga"

    mode_dropdown = widgets.Dropdown(
        options=REFERENCE_MODE_OPTIONS,
        value=default_mode,
        description="Referenz:",
        layout=widgets.Layout(width="330px"),
        style={"description_width": "85px"},
    )
    load_button = widgets.Button(
        description="Daten laden",
        button_style="",
        icon="database",
        layout=widgets.Layout(width="160px"),
    )
    load_button.add_class("clubprofile-red-button")
    search_text = widgets.Text(
        value="",
        placeholder="Name oder Team suchen ...",
        description="Name/Team:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "90px"},
        disabled=True,
    )
    position_dropdown = widgets.Dropdown(
        options=[("Alle Positionen", "")],
        value="",
        description="Position:",
        layout=widgets.Layout(width="260px"),
        style={"description_width": "80px"},
        disabled=True,
    )
    player_dropdown = widgets.Dropdown(
        options=[("Bitte zuerst Daten laden", "")],
        value="",
        description="Treffer:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "70px"},
        disabled=True,
    )
    count_label = widgets.HTML("", layout=widgets.Layout(min_width="220px"))
    status_out = widgets.Output(layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden"))
    print_button = widgets.Button(
        description="Druckversion erzeugen",
        button_style="",
        icon="print",
        disabled=True,
        layout=widgets.Layout(width="280px"),
    )
    print_button.add_class("clubprofile-red-button")
    print_output = widgets.Output(layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden"))
    profile_out = widgets.Output(layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden"))
    profile_tab = widgets.VBox([
        widgets.HBox([print_button], layout=widgets.Layout(width="100%", max_width="1120px", justify_content="flex-end", margin="0 0 12px 0")),
        print_output,
        profile_out,
    ], layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden"))
    spider_container = widgets.VBox(
        [widgets.HTML(cp_spider_placeholder_html())],
        layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden", min_height="650px")
    )
    teambuilder_container = widgets.VBox(
        [widgets.HTML(cp_teambuilder_placeholder_html())],
        layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden", min_height="650px")
    )
    tabs = widgets.Tab(children=[profile_tab, spider_container, teambuilder_container], layout=widgets.Layout(width="100%", max_width="1120px", overflow_x="hidden"))
    tabs.set_title(0, "Rollenprofil")
    tabs.set_title(1, "Metrikvergleich / Spiderplot")
    tabs.set_title(2, "Teambuilder")

    app_state = {"state": None, "records": [], "loaded_mode": None, "selected_player": None, "is_loading": False}

    app_css = widgets.HTML("""
    <style>
      .clubprofile-shell { width:100%; max-width:1120px; box-sizing:border-box; overflow-x:visible; }
      .clubprofile-shell * { box-sizing:border-box; }
      .jupyter-widgets.widget-tab, .widget-tab { width:100% !important; max-width:1120px !important; overflow-x:hidden !important; }
      .widget-tab > .p-TabBar, .widget-tab > .lm-TabBar { border-bottom:0 !important; margin-bottom:18px !important; display:flex !important; flex-wrap:nowrap !important; gap:10px !important; width:100% !important; box-sizing:border-box !important; }
      .widget-tab .p-TabBar-tab, .widget-tab .lm-TabBar-tab {
        padding:13px 14px !important;
        min-height:50px !important;
        display:flex !important;
        align-items:center !important;
        justify-content:center !important;
        text-align:center !important;
        min-width:0 !important;
        max-width:none !important;
        flex:1 1 0 !important;
        border-radius:16px !important;
        margin-right:0 !important;
        font-weight:900 !important;
        font-size:16px !important;
        background:#f3f4f6 !important;
        color:#374151 !important;
        border:1px solid #e5e7eb !important;
      }
      .widget-tab .p-TabBar-tab.p-mod-current, .widget-tab .lm-TabBar-tab.lm-mod-current,
      .widget-tab .p-TabBar-tab.jp-mod-current, .widget-tab .lm-TabBar-tab.jp-mod-current {
        background:#8B0000 !important;
        color:white !important;
        border-color:#8B0000 !important;
      }
      .widget-tab .p-TabBar-tabLabel, .widget-tab .lm-TabBar-tabLabel { font-size:16px !important; white-space:nowrap !important; text-align:center !important; width:100% !important; display:block !important; }
      .p-TabBar-tabLabel, .lm-TabBar-tabLabel { font-size:16px !important; font-weight:900 !important; white-space:nowrap !important; text-align:center !important; width:100% !important; display:block !important; }

      .widget-tab .p-TabBar-tab:before, .widget-tab .lm-TabBar-tab:before,
      .widget-tab .p-TabBar-tab.jp-mod-current:before, .widget-tab .lm-TabBar-tab.jp-mod-current:before,
      .widget-tab .p-TabBar-tab.p-mod-current:before, .widget-tab .lm-TabBar-tab.lm-mod-current:before { display:none !important; content:none !important; }
      .widget-tab .p-TabBar-tab, .widget-tab .lm-TabBar-tab { box-shadow:none !important; outline:none !important; }
      .widget-tab .p-TabBar-tab:focus, .widget-tab .lm-TabBar-tab:focus { box-shadow:none !important; outline:none !important; }
      .clubprofile-red-button { background:#8B0000 !important; color:white !important; border-color:#8B0000 !important; font-weight:900 !important; border-radius:12px !important; }
      .clubprofile-red-button:hover { background:#6f0000 !important; border-color:#6f0000 !important; }
      .cp-tb-fill-modes { width:100% !important; }
      .cp-tb-fill-modes .widget-toggle-button,
      .cp-tb-fill-modes button,
      .jupyter-widgets.cp-tb-fill-modes .widget-toggle-button {
        min-height:40px !important;
        padding:8px 12px !important;
        border-radius:12px !important;
        border:1px solid #ead7d7 !important;
        background:#f3f4f6 !important;
        color:#374151 !important;
        font-weight:900 !important;
        box-shadow:none !important;
        margin-right:8px !important;
      }
      .cp-tb-fill-modes .widget-toggle-button.mod-active,
      .cp-tb-fill-modes button.mod-active,
      .jupyter-widgets.cp-tb-fill-modes .widget-toggle-button.mod-active {
        background:#8B0000 !important;
        color:white !important;
        border-color:#8B0000 !important;
      }
      .cp-tb-fill-modes .widget-toggle-button:hover,
      .cp-tb-fill-modes button:hover {
        border-color:#8B0000 !important;
      }
      .clubprofile-spider-placeholder { min-height:620px; box-sizing:border-box; display:flex; flex-direction:column; align-items:center; justify-content:center; text-align:center; padding:32px; border:1px dashed #d7b0b0; border-radius:18px; background:#FAF7F7; font-family:Arial,sans-serif; }
      .clubprofile-print-box { font-family:Arial,sans-serif; margin:0 0 12px 0; padding:12px 14px; border:1px solid #ead7d7; border-left:6px solid #8B0000; border-radius:14px; background:#FAF7F7; max-width:1120px; box-sizing:border-box; }
      .clubprofile-print-link { display:inline-block; padding:9px 12px; border-radius:10px; background:#8B0000; color:white !important; text-decoration:none !important; font-weight:900; }
      .output_scroll { height:auto !important; max-height:none !important; overflow:visible !important; box-shadow:none !important; }
      .jp-OutputArea-output, .jp-RenderedHTMLCommon, .widget-output, .output { overflow-x:hidden !important; max-width:1120px !important; }
      body, .jp-Notebook, .jp-Cell, .jp-OutputArea, .jp-Cell-outputWrapper { overflow-x:hidden !important; }

    </style>
    """)


    def current_matches():
        q = _normalize_search_text(search_text.value)
        pos_filter = position_dropdown.value
        records = app_state["records"]
        if pos_filter:
            records = [r for r in records if cp_record_matches_position(r.get("position", ""), pos_filter)]
        if q:
            records = [r for r in records if q in r.get("name_team_search", r.get("search", ""))]
        return records

    def update_selected_player(*args):
        if app_state["state"] is None:
            print_button.disabled = True
            app_state["selected_player"] = None
            with print_output:
                clear_output(wait=True)
            try:
                print_output.outputs = ()
            except Exception:
                pass
            with profile_out:
                clear_output(wait=True)
                display(IPyHTML(
                    "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #e5e7eb;border-radius:12px;background:#f9fafb;'>"
                    "Bitte zuerst auf <b>Daten laden</b> klicken."
                    "</div>"
                ))
            spider_container.children = [widgets.HTML(cp_spider_placeholder_html())]
            return

        selected = player_dropdown.value
        app_state["selected_player"] = selected if selected else None
        print_button.disabled = not bool(selected)
        with print_output:
            clear_output(wait=True)
        try:
            print_output.outputs = ()
        except Exception:
            pass
        with profile_out:
            clear_output(wait=True)
            if not selected:
                display(IPyHTML("<div>Bitte einen Spieler auswählen.</div>"))
            else:
                display(IPyHTML(player_profile_html(selected, app_state["state"])))
        if selected:
            spider_container.children = [cp_build_spider_controls(selected)]
        else:
            spider_container.children = [widgets.HTML("<div>Bitte einen Spieler auswählen.</div>")]

    def refresh_dropdown(*args):
        if app_state["state"] is None:
            return
        matches = current_matches()
        shown = matches[:100]
        if shown:
            old_value = player_dropdown.value
            player_dropdown.options = [(r["label"], r["name"]) for r in shown]
            shown_names = [r["name"] for r in shown]
            player_dropdown.value = old_value if old_value in shown_names else shown[0]["name"]
            count_label.value = f"<span style='font-family:Arial,sans-serif;color:#5f6368;'>{len(matches)} Treffer, zeige maximal 100.</span>"
        else:
            player_dropdown.options = [("Keine Treffer", "")]
            player_dropdown.value = ""
            count_label.value = "<span style='font-family:Arial,sans-serif;color:#9f1239;'>Keine Treffer.</span>"
        update_selected_player()

    def load_mode(_=None):
        if app_state["is_loading"]:
            return
        reference_mode = mode_dropdown.value
        app_state["is_loading"] = True
        load_button.disabled = True
        mode_dropdown.disabled = True
        search_text.disabled = True
        player_dropdown.disabled = True
        position_dropdown.disabled = True

        with status_out:
            clear_output(wait=True)
            label = dict(REFERENCE_MODE_OPTIONS).get(reference_mode, reference_mode)
            print(f"Lade Daten und berechne Rollenprofile für: {label}")
            print("Das kann beim ersten Laden auf Hugging Face etwas dauern ...")

        with profile_out:
            clear_output(wait=True)
            display(IPyHTML(
                "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #ead7d7;border-left:6px solid #8B0000;border-radius:12px;background:#FAF7F7;'>"
                "<b>Daten werden geladen.</b><br>Die Rollenprofile werden jetzt berechnet. Bitte einen Moment warten."
                "</div>"
            ))

        try:
            state = get_or_compute_state(reference_mode)
            records = build_player_records(state)
            app_state["state"] = state
            app_state["records"] = records
            app_state["loaded_mode"] = reference_mode
            teambuilder_container.children = [cp_build_teambuilder_controls(state)]

            search_text.disabled = False
            position_dropdown.options = cp_position_filter_options(records)
            position_dropdown.disabled = False
            player_dropdown.disabled = False
            mode_dropdown.disabled = False
            load_button.disabled = False
            app_state["is_loading"] = False

            with status_out:
                clear_output(wait=True)
                label = dict(REFERENCE_MODE_OPTIONS).get(reference_mode, reference_mode)
                print(f"Referenzmodus: {label}")
                print(f"Spieler im Tool: {len(records)}")
                print("Tipp: Suche nach Name oder Team und nutze den Positionsfilter.")

            refresh_dropdown()

        except Exception as exc:
            app_state["is_loading"] = False
            load_button.disabled = False
            mode_dropdown.disabled = False
            search_text.disabled = True
            player_dropdown.disabled = True
            position_dropdown.disabled = True
            teambuilder_container.children = [widgets.HTML(cp_teambuilder_placeholder_html())]
            with status_out:
                clear_output(wait=True)
                print(f"Fehler beim Laden der Daten: {exc}")
            with profile_out:
                clear_output(wait=True)
                display(IPyHTML(
                    "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #fecaca;border-radius:12px;background:#fff1f2;color:#7f1d1d;'>"
                    f"<b>Fehler beim Laden:</b><br>{html.escape(str(exc))}"
                    "</div>"
                ))

    def on_mode_changed(*args):
        # Bei Moduswechsel nicht automatisch rechnen, sondern bewusst per Button laden.
        app_state["state"] = None
        app_state["records"] = []
        app_state["loaded_mode"] = None
        search_text.value = ""
        search_text.disabled = True
        position_dropdown.options = [("Alle Positionen", "")]
        position_dropdown.value = ""
        position_dropdown.disabled = True
        player_dropdown.options = [("Bitte Daten laden", "")]
        player_dropdown.value = ""
        player_dropdown.disabled = True
        print_button.disabled = True
        app_state["selected_player"] = None
        with print_output:
            clear_output(wait=True)
        try:
            print_output.outputs = ()
        except Exception:
            pass
        count_label.value = ""
        spider_container.children = [widgets.HTML(cp_spider_placeholder_html())]
        teambuilder_container.children = [widgets.HTML(cp_teambuilder_placeholder_html())]
        with profile_out:
            clear_output(wait=True)
            display(IPyHTML(
                "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #e5e7eb;border-radius:12px;background:#f9fafb;'>"
                "Referenzmodus geändert. Bitte erneut auf <b>Daten laden</b> klicken."
                "</div>"
            ))
        with status_out:
            clear_output(wait=True)
            print("Referenzmodus geändert. Bitte auf 'Daten laden' klicken.")

    def on_print_clicked(_):
        selected = app_state.get("selected_player")
        state = app_state.get("state")
        with print_output:
            clear_output(wait=True)
            if not selected or state is None:
                display(IPyHTML(
                    "<div class='clubprofile-print-box'>Bitte zuerst einen Spieler auswählen.</div>"
                ))
                return
            display(IPyHTML(cp_build_print_download_html(selected, state)))

    print_button.on_click(on_print_clicked)

    search_text.observe(refresh_dropdown, names="value")
    position_dropdown.observe(refresh_dropdown, names="value")
    player_dropdown.observe(update_selected_player, names="value")
    mode_dropdown.observe(on_mode_changed, names="value")
    load_button.on_click(load_mode)

    controls = widgets.VBox([
        widgets.HTML("""
        <div style='font-family:Arial, Helvetica, sans-serif; width:100%; max-width:1120px; margin:0 0 10px 0; padding:14px 16px; border:1px solid #ead7d7; border-left:7px solid #8B0000; border-radius:18px; background:#FAF7F7; box-sizing:border-box;'>
          <b>Daten laden & Modul wählen</b><br>
          Wähle den Referenzmodus und lade die Daten. Danach kannst du einen Spieler für Rollenprofil oder Spiderplot suchen — oder direkt im Tab <b>Teambuilder</b> eine Formation zusammenstellen.
        </div>
        """, layout=widgets.Layout(width="100%")),
        widgets.HBox([mode_dropdown, load_button], layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap")),
        widgets.HBox([search_text, position_dropdown], layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap")),
        widgets.HBox([player_dropdown, count_label], layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap")),
        status_out,
    ], layout=widgets.Layout(width="100%", max_width="1180px", margin="0 0 24px 0"))

    with status_out:
        print("Bereit. Bitte Referenzmodus wählen und auf 'Daten laden' klicken.")
    with profile_out:
        display(IPyHTML(
            "<div style='font-family:Arial,sans-serif;padding:14px;border:1px solid #e5e7eb;border-radius:12px;background:#f9fafb;'>"
            "Willkommen bei <b>ClubProfile</b>. Bitte zuerst Daten laden. Danach stehen Rollenprofil, Spiderplot und Teambuilder zur Verfügung."
            "</div>"
        ))

    display(app_css)
    display(controls)
    display(tabs)


start_clubprofile_app()
